# **Generación del Pipeline de Actualizaciones**
___
El Objetivo del Código es Crear el Pipeline de Actualizaciones para entender como se va actualmente con las actualizaciones y tener un punto de comparación con el Mejor Mes.

Para esto se va a manejar una estructura similar a la que se tiene en **Timeline** de **Bucket Cartera**, solo que quitando bastantes datos y agregando otros nuevos (como Estados de Alianzas y del Pipeline de Crédito, así como actualizaciones Delta).

Dado que se quiere comparar bajo otros meses, la comparación sera **EXCLUSIVA DE LA CARTERA ASIGNADA**, dado que no se conoce información previa de deudas que no estuvieran dentro de la asignación de la cartera.

La ejecución se realizará en 2 Tiempos:
- **Obtención de Mejor Mes**
- **Actualización de Datos MEC**

## **Categorización de Mejor Mes**
___
Para categorizar el mejor mes, se va a tener en cuenta que exista una buena relación entre **Cartera Actualizada** y **Cartera Actualizada con Descuento**, por lo que para esto, la formula más simple de categorización es el que tenga el máximo valor de esta fórmula:
$$Act\_Cartera = \%Cartera\_Act * \%Cartera\_Act\_Descuento$$

# **Inputs**
___
Estos Inputs son para controlar el efecto de ejecución del código:
- **MESES_ESTRUCTURACION**: La Cantidad de meses a considerar para realizar los calculos de estructuracion
- **COMISION_INICIAL**: El % de la Comisión de Éxito a liquidar en la primera cuota del estructurado.
- **CORREO_ALIANZAS**: El Correo de las Actualizaciones Masivas
- **LIMITE_SOBREACTUALIZACION_NEGOCIADOR**: El periodo en Días para dejar la última actualización del Negociador en Dado caso que sea de Alianzas
- **DESC_CONSIDERAR_EFECTIVA**: El Descuento Mínimo por el cuál se deba considerar una actualización como efectiva
- **CATEGORIAS_FASE_3**: Las Categorías a tener en cuenta para considerar la actualización de una deuda como Fase 3
- **USAR_NOMBRES_ANTIGUOS**: Un Booleano para confirmar si se utilizan los nombres antiguos o los nombres actualizados
- **BANCOS_PLAZOS**: Los Bancos específicos para los que se va a tener en cuenta los plazos designados a Estructurar
- **BANCOS_PORTAFOLIO**: Los Bancos a Considerar pago como
- **DESC_INVIABLE_TRADICIONAL**: El Descuento desde el cúal considerar Inviable y por ende dar a entender que no hay PaB_Ideal
- **DESC_MINIMO_TRADICIONAL**: El Descuento Mínimo a tener en cuenta para el PaB_Ideal
- **MANTENER_MORAS_ORIGINALES**: Un booleano para tener en cuenta si las moras se van a mantener con el mismo nombre o no
- **CORREOS_QUITAR**: Un string de todos los correos que se deben quitar debido a tests.
- **REESCRITURA_TOTAL**: Un Booleano para controlar si se reescribe todo
___

## **Inputs de Bucket**
___
Estos son los Inputs que controlan la generación y Distribución de los Buckets:
- **BUCKET_PESO_INICIAL**: El Peso a Otorgar al Cálculo inicial del Bucket (Por Contactabilidad y Potenciales)
- **BUCKET_PESO_BANCO**: El Peso a Otorgar al Bucket de Banco para la Score del Cliente
- **BUCKET_DISTRIBUCION_BANCO**: El Peso a Otorgar al Mejor Banco del Cliente (El peor Banco sera 1 - Peso)
- **BUCKET_PESO_RATIO**: El Peso a Otorgar al Ratio Ahorro Deuda del Cliente
- **DISTRIBUCION_BUCKETS**: Las Distribuciones Finales de los Buckets en Orden
- **BUCKET_CATEGORIAS_NV**: Las Categorías de Actualización para Considerar a las Deudas como Inviables
___
**FUNNEL_UPDATE_SELECTION** = La selección de la Actualización que queda en el Funnel:
- **Antigua**: Lógica Inicial de Última Deuda
- **Nueva**: Lógica Nueva según Prioridad de Actualización
- **Hibrida**: Se deja la Prioridad solo para Acuerdo, Deposito y Liquidacion, el resto se maneja la última
___
**MODO_DEBUGGING**: Booleano para decidir si se está en modo Debugging, Caso en el que se sube todo a una base Auxiliar usando las Configuraciones Cheveres

In [ ]:
MESES_ESTRUCTURACION = 9 # @param {"type":"slider","min":2,"max":12,"step":1}
COMISION_INICIAL = 0.3 # @param {"type":"slider","min":0,"max":1,"step":0.05}
CORREO_ALIANZAS = 'massive@resuelvetudeuda.com' # @
LIMITE_SOBREACTUALIZACION_NEGOCIADOR = 15 # @param {"type":"slider","min":1,"max":30,"step":1}
DESC_CONSIDERAR_EFECTIVA = 0.03 # @param {"type":"slider","min":0,"max":1,"step":0.01}
CATEGORIAS_FASE_3 = ["ACUERDO", "DESCUENTO", "CONTRAPROPUESTA"] # @param
USAR_NOMBRES_ANTIGUOS = False # @param {"type":"boolean"}
DESC_INVIABLE_TRADICIONAL = 0.95 # @param {"type":"slider","min":0,"max":1,"step":0.05}
DESC_MINIMO_TRADICIONAL = 0.1 # @param {"type":"slider","min":0,"max":1,"step":0.05}
MANTENER_MORAS_ORIGINALES = True # @param {"type":"boolean"}
BANCOS_PLAZOS = {
    'Bancolombia': 2,
    'Exito': 2, # En los Bancos se cambia a Tuya, pero dejarlo aquí por si acaso
    'Tuya': 2,
}
BANCOS_PORTAFOLIO = ['Banco de Occidente', 'Banco Popular', 'Itaú', 'BBVA Colombia']
CORREOS_QUITAR = "humbertoch@gobravo.io,nmcaro@gobravo.io,julio.delgado@gobravo.com.co" # @param {"type":"string","placeholder":"Correos con Ban"}
REESCRITURA_TOTAL = False # @param {"type":"boolean"}

# --- params de Bucket ---
BUCKET_PESO_BASE = 0.75 # @param {"type":"slider","min":0,"max":1,"step":0.05}
BUCKET_PESO_BANCO = 0.25 # @param {"type":"slider","min":0,"max":1,"step":0.05}
BUCKET_DISTRIBUCION_BANCO = 0.75 # @param {"type":"slider","min":0,"max":1,"step":0.05}
BUCKET_PESO_RATIO = 0.2 # @param {"type":"slider","min":0,"max":1,"step":0.05}
DISTRIBUCION_BUCKETS = [0.25,0.2,0.25,0.1,0.1,0.1] # @param
BUCKET_CATEGORIAS_NV = ["NO_VIABLE", "DEMANDA", "ILOCALIZADO", "PAGO_POR_FUERA","Liquidado"] # @param
PIPACT_SHEET_ID = "1NTQnNaAoIptWVz4hRi4-AqavmB5kvp2Pf7QJ90Nzc2A" # @param {"type":"string","placeholder":"Id del Sheets de Bucket Cartera"}

FUNNEL_UPDATE_SELECTION = "Hibrida - Ambas" # @param ["Antigua - Ultima","Nueva - Prioridad","Hibrida - Ambas"]

MODO_DEBUGGING = False # @param {"type":"boolean"}

# Corregimos correo alianzas quitando todo despues del @
CORREO_ALIANZAS = CORREO_ALIANZAS.split('@')[0]

# **Imports Generales**
___
Los Imports y funciones auxiliares necesarios para que el código se pueda ejecutar.

In [ ]:
!pip install python-calamine pandera

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.1/953.1 kB 12.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.6/604.6 kB 22.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandera]


In [ ]:
import pandas as pd
import gspread
from datetime import datetime
from zoneinfo import ZoneInfo
import json
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.oauth2.service_account import Credentials
from googleapiclient.http import MediaIoBaseDownload
from io import BytesIO
from gspread.exceptions import APIError, WorksheetNotFound
import numpy as np
from time import sleep, time
import os
import requests
from requests.exceptions import ChunkedEncodingError, ConnectionError, Timeout
from io import StringIO
# Importamos el diccionario con valores predefinidos
from googleapiclient.discovery import build
from collections import defaultdict
import re
import io
from typing import List, Optional, Literal, NotRequired
from typing_extensions import TypedDict
from itertools import groupby
from operator import itemgetter
from functools import lru_cache
import pandera.pandas as pa
from pandera.typing import Series, DataFrame
import holidays

mesesDict = {
    1: 'Enero',2: 'Febrero',3: 'Marzo',4: 'Abril',5: 'Mayo',6: 'Junio',
    7: 'Julio',8: 'Agosto',9: 'Septiembre',10: 'Octubre',11: 'Noviembre',12: 'Diciembre'
}

# Agregamos el offset para que tenga en cuenta solo business days
co_holidays = holidays.country_holidays('CO', years=pd.Timestamp.now().year)
co_bday = pd.offsets.CustomBusinessDay(holidays=co_holidays) # type: ignore
co_holidays_new = np.array(list(co_holidays.keys()), dtype='datetime64[D]')

# Funcion Auxiliar para obtener variables de entorno/ secretos de forma robusta
def getEnvVar(var: str):
  try: # Si el entorno es Colab las credenciales se obtendrán de esa forma
    from google.colab import userdata
    return userdata.get(var)
  except ImportError: # Si el entorno es GitHub las credenciales se obtienen es mediante variables de entorno
    var = os.getenv(var)
    if not var:
      raise ValueError("No se encontraron las credenciales en las variables de entorno.")
    return var

# Función Auxiliar para reintentar cualquier llamda al API de sheets
def _retry(fn, label="", tries=10, base_sleep=1.5, jitter=0.6, max_sleep=45):
  RETRIABLE_CODES = ["[500]", "[502]", "[503]", "[504]", "[429]","Invalid data[0]: Unable to parse range:"]
  last_err = None
  for i in range(tries):
    try:
      return fn()
    except APIError as e:
      last_err = e
      msg = str(e)
      # Si el error fue del servidor
      if any(c in msg for c in RETRIABLE_CODES):
        sleep_s = min((base_sleep) * (2 ** i) + np.random.uniform(0, jitter), max_sleep)
        print(f"[RETRY {i+1}/{tries}] {label} -> {msg[:120]}... sleep {sleep_s:.1f}s")
        # Esperamos para no saturar al API
        sleep(sleep_s)
        continue
      raise e
  raise last_err

def isWSEmpty(ws: gspread.Worksheet) -> bool:
    # Fetches all data in 1 API call.
    # If the sheet is empty, it returns an empty list [] or [['']]
    data = _retry(lambda: ws.row_values(1))

    if (not data) or not (data[0]):
        return True
    return False

# Función Auxiliar para Listar los Archivos presentes en una carpeta de Drive
def listFiles(driveService, folderId):
  results = driveService.files().list(
        q=f"'{folderId}' in parents and trashed = false",
        fields="files(id, name, mimeType)"
    ).execute()
  files = results.get('files', [])
  return files

# Función Auxiliar para imputar NaNs
def imputeNans(df: pd.DataFrame, col: str, value):
  # Se define la mascara de valores nulos
  mask = df[col].isna()
  # A los valores nulos se aplica el valor
  df.loc[mask, col] = value

# Función Auxiliar para limpiar valores
def cleanText(txt):
  if pd.isna(txt) or isinstance(txt,float):
    return 'nan'
  return txt.lower().replace("ó", "o").replace("á", "a").replace("í", "i").replace("é", "e").replace("ú", "u").upper().strip()

def print_grid(strings, n=5, sep= ' / '):
    """
    Prints a list of strings with 'n' items per row,
    separated by 'sep'.
    """
    for i, word in enumerate(strings):
        # Print the word. Use end='' to manage the separator manually.
        print(word, end="")

        # Check if we need a separator or a newline
        if (i + 1) % n == 0:
            print()  # Newline after every n-th element
        elif i < len(strings) - 1:
            print(sep, end="") # Only print separator if not at the end of a row

    # Add a final newline if the last row wasn't "full"
    if len(strings) % n != 0:
        print()

# Función Auxiliar para Obtener un Diccionario con las Variables de Entorno
def getEnvVarsFromSheet(ws, cellRange: str = "A:B") -> dict:
  # Get values using your retry logic
  values = _retry(lambda: ws.get(cellRange, pad_values=True), label=f"get {cellRange} for {ws.title}")

  if not values:
    raise ValueError('No hay Datos Suficientes')

  result = {}
  for row in values:
    key = row[0] if len(row) > 0 else None
    # If the row is shorter than 2 elements, the second value is 'missing'
    # We assign None (Python's version of NaN) or ""
    val = row[1] if len(row) > 1 else None
    if key:
        result[key] = val
  return result

# Función Auxiliar para obtener un DF a partir de la hoja y el rango
def gettingAsDF(ws, cellRange: str) -> pd.DataFrame:
  # Obtenemos los valores del rango propuesto
  values = _retry(lambda: ws.get(cellRange, pad_values=True))
  # Definimos los headers y las filas
  headers = values[0]
  rows = values[1:]
  # Creamos el DF
  df = pd.DataFrame(rows, columns=headers)
  return df

# Función Auxiliar para Limpiar Números
# This automatically finds currency/number strings and converts them to floats
def cleanNumber(value):
  if not isinstance(value, str):
    return value
  # Reemplazamos X por ''
  value = value.replace('X','')

  # Check if the string contains numbers and currency symbols/commas
  # This regex matches things like: "$ 1.234,56", "50,000", or "1.200.000,00"
  clean_val = value.replace('$', '').replace(' ', '')

  try:
    # Common in Latin America: 1,000.00 -> 1000.00
    if (',' in clean_val) and ('.' in clean_val) and (clean_val.index(',') > clean_val.index('.')):
        clean_val = clean_val.replace('.', '').replace(',', '.')
    elif (',' in clean_val) and ('.' in clean_val) and (clean_val.index(',') < clean_val.index('.')):
        # Reemplzamos , con nada
        clean_val = clean_val.replace(',', '')
    elif '.' in clean_val and clean_val.count('.') > 1:
        # Handle "666.666.666" as 666666666
        clean_val = clean_val.replace('.','')
    elif ',' in clean_val and clean_val.count(',') > 1:
        # Handle "666.666.666" as 666666666
        clean_val = clean_val.replace(',','')
    elif ',' in clean_val and clean_val.count(',') == 1:
        parts = clean_val.split(',')
        # The first part can be 1-3 digits; all following parts MUST be exactly 3 digits
        if 1 <= len(parts[0]) <= 3 and all(len(s) == 3 for s in parts[1:]):
            clean_val = clean_val.replace(',', '')
        else:
            # If it's a decimal separator used multiple times (e.g., European format typo or specific notation)
            # Note: A valid float can only have ONE decimal point.
            # If there are multiple commas acting as decimals, replacing them all with '.' will still error out.
            clean_val = clean_val.replace(',', '.')
    elif '.' in clean_val and clean_val.count('.') == 1:
        # 2 Possible Cases: Decimal Separator or Thousand Separator
        # Detecting case thousand by: splitting and Confirming len of all == 3
        if all([len(s) == 3 for s in clean_val.split('.')]):
            clean_val = clean_val.replace('.', '')
        # Else it's decimal so its ignored
    return float(clean_val)
  except ValueError:
      return value # Not a number? Return original text

# Función Auxiliar para Permitir diferentes Nombres de Columnas
def cleanCols(df: pd.DataFrame, realCol: str, colPossibleVals: list = []) -> pd.DataFrame:
  dfCols = df.columns.tolist()
  # Por cada una de las Columnas Posibles
  for col in colPossibleVals:
    # Si existe se actualiza y se termina la Ejecución
    if col in dfCols:
      if col != realCol and realCol in dfCols:
        df = df.drop(columns=[realCol])
      df = df.rename(columns={col: realCol})
      break
  else:
    print('✖️No se realizó el cambió para {} con {} posibles valores.'.format(realCol, ', '.join(colPossibleVals)))
    df[realCol] = np.nan
  return df

def getBDDaysDiffFloat_vectorized(start_series: pd.Series, end_series: pd.Series) -> pd.Series:
    global co_holidays_new
    # 1. FORZAR PARSEO: Convierte strings/objetos a datetime64[ns] real.
    # Los valores corruptos o no parseables se transformarán en NaT de forma segura.
    s_dt = pd.to_datetime(start_series, errors='coerce')
    e_dt = pd.to_datetime(end_series, errors='coerce')

    # 2. Identificar NaNs/NaTs (Tanto los originales como los que fallaron al parsear)
    nan_mask = pd.isna(s_dt) | pd.isna(e_dt)

    # 3. Asegurar el orden (start debe ser menor o igual que end)
    s = s_dt.copy()
    e = e_dt.copy()

    # Si hay casos donde start > end, los invertimos vectorialmente
    mask_inverse = s > e
    if mask_inverse.any():
        s[mask_inverse], e[mask_inverse] = end_series[mask_inverse], start_series[mask_inverse]

    # --- SOLUCIÓN REAL AL ERROR ---
    # Rellenamos los NaT manteniendo el tipo nativo datetime64 de Pandas
    s_clean = s.fillna(pd.Timestamp('1970-01-01'))
    e_clean = e.fillna(pd.Timestamp('1970-01-01'))

    # Convertimos al formato nativo de NumPy (.values) y de ahí
    # transformamos a precisión de día ('datetime64[D]') directamente.
    # Ahora que garantizamos un tipo datetime64[ns], pasamos a NumPy 'datetime64[D]' con total seguridad
    s_date = s_clean.to_numpy(dtype='datetime64[D]')
    e_date = e_clean.to_numpy(dtype='datetime64[D]')
    # --------------------------------------

    # 2. Identificar NaNs originales para el final
    nan_mask = pd.isna(start_series) | pd.isna(end_series)

    if co_holidays_new is not None:
        if isinstance(co_holidays_new, holidays.HolidayBase):
            co_holidays_new = np.array(list(co_holidays_new.keys()), dtype='datetime64[D]')
        else:
            co_holidays_new = np.array(co_holidays_new, dtype='datetime64[D]')
    else:
        co_holidays_new = np.array([], dtype='datetime64[D]')

    # 3. Calcular días hábiles completos
    full_bus_days = np.busday_count(s_date, e_date,holidays=co_holidays_new)

    # 4. Calcular fracciones de tiempo usando componentes .dt (Vectorizado)
    start_fraction = (s.dt.hour + s.dt.minute / 60 + s.dt.second / 3600) / 24
    end_fraction = (e.dt.hour + e.dt.minute / 60 + e.dt.second / 3600) / 24

    # 5. Operación matemática vectorial
    float_diff = full_bus_days - start_fraction + end_fraction

    # Redondear y reasignar NaNs originales
    result = np.round(float_diff, 4)
    result = np.where(nan_mask, np.nan, result)

    return pd.Series(result, index=start_series.index)

# Función Auxiliar para obtener credenciales
def getCreds():
  try: # Si el entorno es Colab las credenciales se obtendrán de esa forma
    from google.colab import userdata
    return json.loads(userdata.get('MI_JSON'))
  except ImportError: # Si el entorno es GitHub las credenciales se obtienen es mediante variables de entorno
    creds = os.getenv('MI_JSON')
    if not creds:
      raise ValueError("No se encontraron las credenciales en las variables de entorno.")
    return json.loads(creds.strip())

# 1. Retrieve the secret from Colab
service_account_info = getCreds()

# 2. Define the scope and authenticate
scope = ['https://www.googleapis.com/auth/spreadsheets','https://www.googleapis.com/auth/drive']

creds = Credentials.from_service_account_info(service_account_info, scopes=scope)
client = gspread.authorize(creds)

# Creamos el Servicio de Drive
driveService = build('drive', 'v3', credentials=creds)

today = pd.Timestamp.now()

# **Datos de Cambios de Referencia**
Los Datos que se obtienen son: **refChangesDict**

In [ ]:
# Abrimos la SpreadSheet
refChangesShId = '1FN1u5b6N5mzcrzYzLKpBjwSkhddgYPOrgyZgbKChc68'
refChangesSH = _retry(lambda: client.open_by_key(refChangesShId))
# Aca se cargan los cambios de referencia
refChangesWS = _retry(lambda: refChangesSH.worksheet('Cambios de Referencia'))
records = _retry(lambda: refChangesWS.get_all_values())

## La llave sera la referencia vieja y el valor la referencia nueva
if len(records)>0 and len(records[0])>1:
  refChangesDict = {str(row[0]).replace('.0','').strip():str(row[1]).replace('.0','').strip() for row in records}
else:
  refChangesDict = {}

print('✅Cambios de Referencia Cargados con Éxito')
print('ℹ️Cargas Totales: {}'.format(
    len(refChangesDict)
))

✅Cambios de Referencia Cargados con Éxito
ℹ️Cargas Totales: 3716


# **Clases Auxiliares**
___
Se crea la Clase **ColumnaBucket** para manejar de forma más eficiente los cambios que se puedan producir de los nombres de las Columnas de modo que el código no deba ser ajustado constantemente.

In [ ]:
clasesPosibles = Literal['numeric','semistring','string','date']

class ColumnaBucket:
  def __init__(self, nombre: str, tipo: clasesPosibles, oldName: str = ''):
    self.nombre = nombre
    self.tipo = tipo
    self.oldName = oldName

  def isNumeric(self):
    return self.tipo == 'numeric'

  def isSemiString(self):
    return self.tipo == 'semistring'

  def isString(self):
    return self.tipo == 'string'

  def isDate(self):
    return self.tipo == 'date'

  def getActualName(self):
    return self.nombre

  def getOldName(self):
    return self.oldName if self.oldName else self.nombre

  def getRequiredName(self, is_old: bool):
    return self.getOldName() if is_old else self.getActualName()

# **Configuraciones**
___
Parámetros rara vez cambiados pero que si se requieren para los cálculos

In [ ]:
MAPPEADOR_ETIQUETAS_ETAPA = {
  "INTENTO DE CONTACTO": 0,
  "ACTUALIZADO_ALIANZAS": 0,
  "LLAMADA": 0,
  "ILOCALIZADO (INUBICABLE)": 1,
  "PAGO EXTERNO (FUERA DEL PROGRAMA)": 1,
  "SIN OFERTA": 1,
  "SIN LIQUIDEZ": 1,
  "SOLICITUD DE BAJA": 1,
  "DEMANDA JUDICIAL": 1,
  "VERIFICACIÓN DE SALDOS": 1,
  "VERIFICACION DE SALDOS": 1,
  "SALDOS": 1,
  "SIN LIQUIDEZ": 1,
  "CAMBIO PAB": 1,
  "CAMBIO DE PAB": 1,
  "CONTRAOFERTA": 2,
  "DEPÓSITO": 2,
  "DEPOSITO": 2,
  "ACUERDO": 3,
  "BANK_LETTER": 4,
  "LIQUIDADO": 4,
  "Liquidado": 4,
  "Liquidado en Portafolio": 4,
}

# **Obtención de Mejor Mes**
___
En este apartado se consolida cuál es el mejor mes con base en los datos de la cartera asignada. Para evitar cargar todas las carteras siempre que se requiera de una actualización, solo se cálcula el mejor mes bajo estas 2 condiciones:
- **Cambio de MEC**
- **Falta Input en Hoja**

## **Obtención de Datos de Control**
___
Los datos que se obtienen son: **controlDict: dict[str,Any]**

In [ ]:
# Abrimos la Spreadsheet del Pipeline de Actualizaciones
controlSHId = '1NTQnNaAoIptWVz4hRi4-AqavmB5kvp2Pf7QJ90Nzc2A'
controlSH = _retry(lambda: client.open_by_key(controlSHId))
# Abrimos la Hoja llamada 'Control'
controlWS = _retry(lambda: controlSH.worksheet('Control'))
# Obtenemos el Diccionario
controlDict = getEnvVarsFromSheet(controlWS)

# Ahora Ajustamos 'Mejor-Mes' y 'Mes-Mec' para que queden en formato %Y-%m
controlDict['Mejor-Mes'] = pd.to_datetime(controlDict['Mejor-Mes'])
controlDict['Mes-Mec'] = pd.to_datetime(controlDict['Mes-Mec'])
controlDict['Mejor-Mes'] = controlDict['Mejor-Mes'].strftime('%Y-%m') if pd.notna(controlDict['Mejor-Mes']) else None
controlDict['Mes-Mec'] = controlDict['Mes-Mec'].strftime('%Y-%m') if pd.notna(controlDict['Mes-Mec']) else None

# Abrimos la Hoja 'Mejor_Mes'
mejorMesWS = _retry(lambda: controlSH.worksheet('Mejor_Mes'))
# Actualizamos la Llave mejor mes si la hoja esta vacía
if isWSEmpty(mejorMesWS):
  controlDict['Mejor-Mes'] = None

print('✅Control Cargado con Éxito')
print('ℹ️Datos Cargados: {}'.format(
    len(controlDict)
))

✅Control Cargado con Éxito
ℹ️Datos Cargados: 3


## **Determinación de Necesidad de Cálculos**
___
Se define si se requiere actualización. Los datos que se obtienen son. **CALCULAR_MEJOR_MES: bool y fecha_cartera_ultima: pd.Timestamp**

In [ ]:
## --- Funciones de Ayuda ---
def obtener_prioridad_archivo(file_name: str):
    if not file_name.endswith('.xlsx'):
        return 0

    raw_month_year = file_name.split('-')[-1]
    month = raw_month_year[:3].lower()
    year = raw_month_year[3:5]

    month_num = next(
        (m_num for m_num in mesesDict
         if month in mesesDict[m_num].lower()),
        1
    )
    year_num = int(year) + 2000

    return year_num * 13 + month_num


def obtener_prioridad_hoja(sheet_name: str) -> tuple[int, int, int]:
    try:
        month_name, year = sheet_name.split()

        month_num = next(
            (m_num for m_num in mesesDict
             if month_name.lower() in mesesDict[m_num].lower()),
            1
        )
        year_num = int(year)

        return year_num * 13 + month_num, year_num, month_num

    except (ValueError, TypeError):
        print(sheet_name, '❌Error')
        return 0, 0, 0


@lru_cache(maxsize=None)
def _obtener_fecha_cartera_cached(file_id: str, file_name: str):
    request = driveService.files().get_media(fileId=file_id)

    print(f'😁Descargando archivo "{file_name}"')
    time_init = time()

    file_bytes = io.BytesIO(request.execute())

    print(
        f'ℹ️Archivo descargado con éxito '
        f'({time() - time_init:,.0f} segundos)'
    )

    # Obtener únicamente los nombres de las hojas y su orden de prioridad.
    excel_file = pd.ExcelFile(file_bytes, engine='calamine')

    sheet_priorities = sorted(
        (
            (sheet_name, *obtener_prioridad_hoja(sheet_name)[1:])
            for sheet_name in excel_file.sheet_names
        ),
        key=lambda x: x[1] * 13 + x[2],
        reverse=True
    )

    for sheet_name, year, month in sheet_priorities:
        if year == 0 or not (1 <= month <= 12):
            continue

        # Leer una hoja a la vez para evitar cargar todos los DataFrames.
        df_sheet = pd.read_excel(
            excel_file,
            sheet_name=sheet_name
        )

        if df_sheet.empty or len(df_sheet.columns) <= 2:
            continue

        fecha_cartera = pd.Timestamp(
            year=year,
            month=month,
            day=1
        )

        print(
            f'✅Fecha_Cartera encontrada en "{file_name}", '
            f'hoja "{sheet_name}": {fecha_cartera:%Y-%m-%d}'
        )

        return fecha_cartera

    print(f'⚠️No se encontraron hojas válidas en "{file_name}"')
    return None


def obtener_fecha_cartera_archivo(file_info: dict):
    return _obtener_fecha_cartera_cached(
        file_info['id'],
        file_info['name']
    )


# Paso 1: Listar los archivos
folderAsignaciones = '1cf2p3R7iM0xowAt4muEruDwxZoZqD_jB'
archivosList = listFiles(driveService, folderAsignaciones)

if not archivosList:
    raise FileNotFoundError(
        '✖️No se encontraron archivos en la carpeta de Asignaciones'
    )


# Paso 2: Ordenar archivos por prioridad (más recientes primero)
archivosList.sort(
    key=lambda x: obtener_prioridad_archivo(x['name']),
    reverse=True
)


# Paso 3: Buscar la Fecha_Cartera del primer archivo/hoja válido
fecha_cartera_ultima = None

for file in archivosList:
    if not file['name'].endswith('.xlsx'):
        continue

    fecha_cartera_ultima = obtener_fecha_cartera_archivo(file)

    if fecha_cartera_ultima is not None:
        break


if fecha_cartera_ultima is None:
    raise ValueError(
        '✖️No se encontró ninguna hoja válida en los archivos de Asignaciones'
    )

print(f'📅Fecha Cartera válida: {fecha_cartera_ultima:%Y-%m}\n')

# Ahora Procederemos a Verificar si se requiere de actualizar el mejor mes teniendo en cuenta el cambio Mec
if (fecha_cartera_ultima.strftime('%Y-%m') == controlDict['Mes-Mec']) and pd.notna(controlDict['Mejor-Mes']):
  CALCULAR_MEJOR_MES = False
else:
  CALCULAR_MEJOR_MES = True

if CALCULAR_MEJOR_MES:
  print('✅🛠️ℹ️ Se definiò que se requiere RECALCULAR EL MEJOR MES, esto dado el motivo de: {}'.format(
      "CAMBIO MEC" if fecha_cartera_ultima.strftime('%Y-%m') != controlDict['Mes-Mec'] else "FALTA CALCULAR"
  ))
  # Definimos startDate como mayo del 2026
  startDate = pd.Timestamp(2026,5,1)
else:
  print('😊ℹ️ NO es necesario realizar el càlculo del mejor mes')
  # Definimos startDate como fecha_cartera_ultima
  startDate = fecha_cartera_ultima

endDate = today + pd.Timedelta(days=1)

print('\n🛠️📅Fecha de Inicio: {}'.format(
    startDate.strftime('%Y-%m-%d')
))
print('📅Fecha de Fin: {}'.format(
    endDate.strftime('%Y-%m-%d')
))

😁Descargando archivo "Asignaciones de Cartera Sep26-Dic26.xlsx"
ℹ️Archivo descargado con éxito (32 segundos)
✅Fecha_Cartera encontrada en "Asignaciones de Cartera Sep26-Dic26.xlsx", hoja "Noviembre 2026": 2026-11-01
📅Fecha Cartera válida: 2026-11

✅🛠️ℹ️ Se definiò que se requiere RECALCULAR EL MEJOR MES, esto dado el motivo de: FALTA CALCULAR

🛠️📅Fecha de Inicio: 2026-05-01
📅Fecha de Fin: 2026-11-18


## **Obtención de Datos Iniciales**
___
Se obtienen los datos de las Carteras Asignadas, Liquidaciones (Para saber el mes operativo) y Actualizaciones

### **Datos de la Asignación de Cartera**
Los Datos que se obtienen son: **clientesDF**
- Referencia
- Id_Deuda
- Cedula
- Nombre_Cliente
- Meses_en_Programa
- Comision_Mensual
- Ahorro_Neto
- Ahorro_Medio
- Apartado_Neto
- Meses_Sin_Contacto
- Pricing
- Prioridad
- Mora_Estructurado
- Mora_Credito
- Banco
- Dum_Bravo
- PaB_PL
- Mora_Deuda
- Descuento_Esperado
- PaB_Esperado
- Ingreso_Esperado
- Potencial_Esperado
- Bucket_Inicial
- Sub_Estado
- Estado
- Descuento_PL
- Estructurable
- Vehiculo_Ahorro
- Estado_Cliente

In [ ]:
# Definimos las Columnas a Cambiar
colsToChange = {
    'Referencia': ['Referencia'],
    'Id_Deuda': ['Id deuda'],
    'Cedula': ['Cédula','Cedula'],
    'Nombre_Cliente': ['Nombre Cliente','Nombre del cliente'],
    'Negociador_Asignado': ['Negociador'],
    'Meses_en_Programa': ['Meses en el Programa','Meses en el Programa'],
    'Comision_Mensual': ['Comisión Mensual', 'Comision Mensual'],
    'Apartado_Mensual': ['Apartado Mensual','Apartado Mensual'],
    'Ahorro_Total': ['Ahorro Total','Ahorro total'],
    'Ahorro_Medio': ['Ahorro Medio','Ahorro medio'],
    'Por_Cobrar': ['Por Cobrar','Por cobrar','Por cobrar'],
    'Meses_Sin_Contacto': ['ultimo contacto'],
    'Pricing': ['CE','Pricing'],
    'Prioridad': ['Prioridad','Priority_level'],
    'Mora_Estructurado': ['Mora Estructurado','Mora estructurado','Mora_estructurado'],
    'Mora_Credito': ['Mora Crédito','Mora_credito','Mora Crédito','MORA_CREDITO'],
    'Banco': ['BANCOS_ESTANDAR','Banco'],
    'Dum_Bravo': ['Deuda Bravo','Deuda Resuelve','Deuda Resuelve'],
    'PaB_PL': ['PaB PL','PaB PL','PB_PL'],
    'Mora_Deuda': ['Meses de atraso'],
    'Descuento_Esperado': ['Descuento','Descuento Esperado'],
    'Potencial_Esperado': ['Potencial'],
    'Bucket_Inicial': ['Bucket'],
    'Sub_Estado': ['sub_estado_deuda'],
    'Estado': ['estado_deuda','Estado Deuda'],
    'Estructurable': ['Estructurable'],
    'Vehiculo_Ahorro': ['vehiculo','vehiculo_ahorro'],
    'Estado_Cliente': ['sub_estado_reparadora'],
}

sourceFolderId = '1cf2p3R7iM0xowAt4muEruDwxZoZqD_jB'
# Se realiza una Query donde se obtienen todos los archivos no eliminados
results = driveService.files().list(
        q=f"'{sourceFolderId}' in parents and trashed = false",
        fields="files(id, name, mimeType)"
    ).execute()
files = results.get('files', [])

# Definimos Mes Mínimo y Año Mínimo
minYear = startDate.year
minMonth = startDate.month
maxYear = endDate.year       # <-- NUEVO: Año límite superior
maxMonth = endDate.month     # <-- NUEVO: Mes límite superior
# Nota: Los Rangos son Inclusivos

# # Segunda Modificación: Cuando endDate es de año >= 2026 y mes >=2 se reduce '1 mes'
# if endDate.year >= 2026 and endDate.month >= 2:
#   maxMonth -= 1
#   if maxMonth == 0:
#     maxMonth = 12
#     maxYear -= 1

# Creamos una lista para guardar los df
clientesDFList = []

# Iteramos por cada uno de los archivos
for file in files:
  fileID = file['id']
  mime_type = file['mimeType']
  name = file['name']

  print('\n⚙️Analizando',name)
  # Cambiamos el nombre para que solo sea la ultima parte del nombre
  name = name.split(' ')[-1]

  # Ahora toca obtener el nombre de las hojas a escanear
  # Toca definir el año y mes de inicio
  # El mes son los primeros 3 caracteres del nombre
  # El año son el caracter 4-5
  initMonth = name[:3]
  initYear = int(name[3:5]) + 2000 # Se añade el +2000 porque solo estan los ultimos 2 años

  # Se obtiene el número del mes
  initMonthNum = [k for k,v in mesesDict.items() if initMonth.lower() in v.lower()][0]

  # 2. Normalize both to the start of their respective 4-month periods (Quadrimesters)
  # The file's period start:
  filePeriodStart = (initMonthNum - 1) // 4 * 4 + 1
  # The minimum threshold's period start:
  limitPeriodStart = (minMonth - 1) // 4 * 4 + 1
  maxLimitPeriodStart = (maxMonth - 1) // 4 * 4 + 1 # <-- NUEVO: Periodo de inicio máximo permitido

  # 3. Corrected Comparison
  # Skip if: Year is lower OR (Year is same AND file month is lower than limit month)
  if (initYear < minYear) or (initYear == minYear and filePeriodStart < limitPeriodStart):
    print('✖️ El Archivo {} se omitió ya que es menor a {} {}'.format(name, mesesDict[minMonth], minYear))
    continue

  # FILTRO 2: Corrección Límite Superior (endDate) <-- NUEVO
  # Omitir si el año del archivo es mayor, o si es el mismo año pero el inicio del periodo supera al del endDate
  if (initYear > maxYear) or (initYear == maxYear and filePeriodStart > maxLimitPeriodStart):
    print('✖️ El Archivo {} se omitió ya que supera a {} {}'.format(name, mesesDict[maxMonth], maxYear))
    continue

  # FILTRO 3: Tiene que ser un archivo de Excel
  if not name.endswith('.xlsx'):
    print('❌El Archivo {} se omitió ya que no es un excel'.format(
        name
    ))
    continue

  # Se definen el nombre de las hojas filtrando por los dos extremos (startDate y endDate)
  sheetNames = []
  for m in range(initMonthNum, initMonthNum + 4):
    # Evaluamos si el mes 'm' y el año 'initYear' están dentro del rango permitido
    is_after_start = (initYear > minYear) or (initYear == minYear and m >= minMonth)
    is_before_end = (initYear < maxYear) or (initYear == maxYear and m <= maxMonth)

    if is_after_start and is_before_end:
        sheetNames.append((f'{mesesDict[m]} {initYear}', m))

  # Descargamos el Archivo
  request = driveService.files().get_media(fileId=fileID)
  fh = io.BytesIO()
  downloader = MediaIoBaseDownload(fh, request)
  done = False
  while done is False:
    status, done = downloader.next_chunk()
    print("ℹ️Descargando %d%%." % int(status.progress() * 100))

  # Por cada nombre de hoja se leen los datos
  for sheet,m in sheetNames:
    # Leemos los datos de la hoja
    df = pd.read_excel(fh, sheet_name=sheet, engine='calamine')
    # Si el archivo esta vacio continuamos
    if df.empty:
      continue
    # --- Limpieza de Columnas ---
    for col, possibleVals in colsToChange.items():
      df = cleanCols(df, col, possibleVals)

    df['Fecha_Cartera'] = pd.Timestamp(year=initYear, month=m, day=1)
    # Dejamos Solo las Columnas Requeridas
    df = df[['Fecha_Cartera'] + list(colsToChange.keys())]

    # Se guarda la inormación en la lista designada
    clientesDFList.append(df)
    # Imprimimos para Logs
    print('🆔Datos para {} fueron cargados con éxito con {} filas.'.format(sheet, len(df)))

# Unimos todos los DF en uno solo
clientesDF = pd.concat(clientesDFList, ignore_index=True)

# --- Limpieza de Datos ---
# Limpieza de Ids: Volver string, quitar .0 y espacios
clientesDF['Referencia'] = clientesDF['Referencia'].apply(lambda s: str(s).replace('.0','').strip())
clientesDF['Id_Deuda'] = clientesDF['Id_Deuda'].apply(lambda s: str(s).replace('.0','').strip())
clientesDF['Cedula'] = clientesDF['Cedula'].apply(lambda s: str(s).replace('.0','').strip())

# Aplicamos los cambios de referencia si es que existen
clientesDF['Referencia'] = clientesDF['Referencia'].apply(lambda x: refChangesDict.get(x, x))

# Volvemos las Columnas Numericas a tipo numerico
numCols = ['Meses_en_Programa', 'Comision_Mensual', 'Apartado_Mensual', 'Ahorro_Total', 'Ahorro_Medio', 'Por_Cobrar',
          'Meses_Sin_Contacto', 'Pricing','PaB_PL', 'Mora_Deuda', 'Dum_Bravo',
          'Descuento_Esperado','Bucket_Inicial']
for col in numCols:
  clientesDF[col] = pd.to_numeric(clientesDF[col], errors='coerce')

# Empezamos a Imputar NaNs de Columnas Númericas
imputeNans(clientesDF, 'Meses_en_Programa', 0)
imputeNans(clientesDF, 'Comision_Mensual', 0)
imputeNans(clientesDF, 'Apartado_Mensual', 0)
imputeNans(clientesDF, 'Ahorro_Total', 0)
imputeNans(clientesDF, 'Ahorro_Medio', 0)
imputeNans(clientesDF, 'Por_Cobrar', 0)
imputeNans(clientesDF, 'Meses_Sin_Contacto', 12) # Cuando no existe se deja un año
imputeNans(clientesDF, 'Pricing', 0.15) # Se imputa con el valor de Pricing general
imputeNans(clientesDF, 'Mora_Deuda', 0)
imputeNans(clientesDF, 'Dum_Bravo', 0)
imputeNans(clientesDF, 'Descuento_Esperado', 0)

# Creamos la Columna de Descuento_PL como: 1 - (PaB_PL / Dum_Bravo); Si el Dum_Bravo es 0, se imputa con 1
# Inicializamos Descuento_PL con 1
clientesDF['Descuento_PL'] = 1 - (clientesDF['PaB_PL'] / clientesDF['Dum_Bravo'])
maskZeroOrigen = (clientesDF['Dum_Bravo'] == 0)
clientesDF.loc[maskZeroOrigen, 'Descuento_PL'] = 1
maskNanPL = clientesDF['PaB_PL'].isna()
clientesDF.loc[maskNanPL, 'Descuento_PL'] = np.nan

# Ahora vamos a definir el Imputador de PaB_PL según el Banco
# Para esto agrupamos quitando sin PL por Banco y calculamos el Descuento Promedio
# Luego aplicamos ese descuento a los NaNs
# Paso 1: Aplicar la Agrupación quitando sin PL y con Origen = 0
plImputer = clientesDF[~(maskNanPL | maskZeroOrigen)].groupby('Banco')[['Dum_Bravo','PaB_PL']].sum().reset_index()
# Paso 2: Calcular el Descuento Promedio
plImputer['Desc'] = 1 - (plImputer['PaB_PL']/plImputer['Dum_Bravo'])
# Paso 3: Convertir el Resultado a un Diccionario
plImputer = dict(zip(plImputer['Banco'], plImputer['Desc']))
# Paso 4: Convertir a defaultdict con 0.3
plImputer = defaultdict(lambda: 0.3, plImputer)
# Paso 4: Aplicar la Imputación
clientesDF.loc[maskNanPL, 'PaB_PL'] = clientesDF.loc[maskNanPL, 'Dum_Bravo'] * (1 - clientesDF.loc[maskNanPL,'Banco'].map(plImputer))
clientesDF.loc[maskNanPL, 'Descuento_PL'] = clientesDF.loc[maskNanPL,'Banco'].map(plImputer)

# Al Nommbre del Cliente aplicamos .title() si no es NaN
clientesDF['Nombre_Cliente'] = clientesDF['Nombre_Cliente'].apply(lambda s: s.title() if pd.notna(s) else s)

# Volvemos Columnas que son String a Númericas creando diccionarios de mapeo
# 1. Prioridad: Funciona así: regex(Prioridad (\d+)) -> extraemos el número y ese es el valor, si no se encuentra se imputa con 3
def extractPriority(x):
  if pd.isna(x):
    return 3
  match = re.search(r'Prioridad\s*(\d+)', str(x))
  if match:
    return int(match.group(1))
  else:
    return 3
clientesDF['Prioridad'] = clientesDF['Prioridad'].apply(extractPriority)

# 2. Mora_Estructurado
mapEstr = {
  'No tiene estructurado': 0,
  'No': 1,
  'Sí': 2,
}
clientesDF['Mora_Estructurado'] = clientesDF['Mora_Estructurado'].apply(lambda x: mapEstr.get(x, 0))

# 3. Mora_Credito
mapMora = {
  'No': 0,
  'Sí': 1,
}
clientesDF['Mora_Credito'] = clientesDF['Mora_Credito'].apply(lambda x: mapMora.get(x, 0))

# --- Creación de Nuevas Columnas ---
# Columna Ahorro_Neto: Ahorro_Total - Por_Cobrar
clientesDF['Ahorro_Neto'] = clientesDF['Ahorro_Total'] - clientesDF['Por_Cobrar']
# Quitamos las Columnas de Ahorro_Total y Por_Cobrar para evitar confusiones
#clientesDF = clientesDF.drop(columns=['Ahorro_Total', 'Por_Cobrar'])

# Columna Apartado_Neto: Apartado_Mensual - Comision_Mensual
clientesDF['Apartado_Neto'] = clientesDF['Apartado_Mensual'] - clientesDF['Comision_Mensual']
# Quitamos la Columna de Apartado_Mensual para evitar confusiones, se necesita Comisón mensual a futuro
clientesDF = clientesDF.drop(columns=['Apartado_Mensual'])

# Columna PaB_Esperado: Dum_Bravo * (1 - Descuento_Esperado); No se elimina ninguna Columna
clientesDF['PaB_Esperado'] = clientesDF['Dum_Bravo'] * (1 - clientesDF['Descuento_Esperado'])
# Reescribimos Columna Ingreso_Esperado como: Dum_Bravo * Descuento_Esperado * Pricing # Sin IVA
clientesDF['Ingreso_Esperado'] = clientesDF['Dum_Bravo'] * clientesDF['Descuento_Esperado'] * clientesDF['Pricing']
# Aplicamos un clip con  lower=0 al Ingreso_Esperado
clientesDF['Ingreso_Esperado'] = clientesDF['Ingreso_Esperado'].clip(lower=0)

# Creamos un Diccionario que: dict[Id_Deuda ->  Dum_Bravo]
debtPaBMap = dict(zip(clientesDF['Id_Deuda'], clientesDF['Dum_Bravo']))
# Lo volvemos un defaultdict con valor por defecto de 100 mil M
debtPaBMap = defaultdict(lambda: 100_000_000_000, debtPaBMap)

# Imprimimos para logs
print('\n✅Datos de los Clientes Obtenidos con éxito, columnas: {}'.format(', '.join(clientesDF.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(clientesDF)))


⚙️Analizando Asignaciones de Cartera Sep26-Dic26.xlsx
ℹ️Descargando 100%.
🆔Datos para Septiembre 2026 fueron cargados con éxito con 39732 filas.
🆔Datos para Octubre 2026 fueron cargados con éxito con 41368 filas.
🆔Datos para Noviembre 2026 fueron cargados con éxito con 42596 filas.

⚙️Analizando Asignaciones de Cartera Sep26-Dic26
❌El Archivo Sep26-Dic26 se omitió ya que no es un excel

⚙️Analizando Asignaciones de Cartera May26-Ago26.xlsx
ℹ️Descargando 100%.
🆔Datos para Mayo 2026 fueron cargados con éxito con 40080 filas.
🆔Datos para Junio 2026 fueron cargados con éxito con 38528 filas.
🆔Datos para Julio 2026 fueron cargados con éxito con 38535 filas.
🆔Datos para Agosto 2026 fueron cargados con éxito con 40349 filas.

⚙️Analizando Asignaciones de Cartera May25-Ago25.xlsx
✖️ El Archivo May25-Ago25.xlsx se omitió ya que es menor a Mayo 2026

⚙️Analizando Asignaciones de Cartera Sep25-Dic25.xlsx
✖️ El Archivo Sep25-Dic25.xlsx se omitió ya que es menor a Mayo 2026

⚙️Analizando Asignacio

### **Datos de Liquidaciones**
Los datos que se obtienen son: **liqsDF**
- Id_Deuda
- Fecha_Liq
- Negociador_Liq
- Tipo_Liq
- PaB_Liq
- Ingreso_Liq
- Portafolio_Liq

In [ ]:
# Abrimos la SpreadSheet
liqsSHId = '1H3sYEtkeu47POnu8xZMaMtID1Vj53YIcWblWeZ8d0rc'
liqsSH = _retry(lambda: client.open_by_key(liqsSHId))

# Se define el rango de columnas a obtener
# PARAMETRO CAMBIABLE -----------------
cellRange = 'A1:U'

minYear = startDate.year

# Se definen las columnas a usar de las a modo de Diccionario
# PARAMETRO CAMBIABLE -----------------
colsliqs = {
    'PaB_Liq': ['Pago a banco'],
    'Id_Deuda': ['Deuda Berex'],
    'Ingreso_Liq': ['Comisión','Comisión RESUELVE'],
    'Negociador_Liq': ['Negociador'],
    'Tipo_Liq': ['Tipo de liquidacion'],
    'Fecha_Liq': ['Fecha de Liquidación','Fecha de Liquidación '],
    'Portafolio_Liq': ['Portafolio','Portafolio'],
    'Mes_Liq': ['Mes'],
    'Año_Liq': ['Año'],
}

# ------------ liqs Mensuales --------------
nombreHojaliqs = 'BD del mes'
liqs = _retry(lambda: liqsSH.worksheet(nombreHojaliqs))
liquidacioneMensualessDF = _retry(lambda: gettingAsDF(liqs, cellRange))

# --- Limpieza de Columnas ---
for col, possibleVals in colsliqs.items():
  liquidacioneMensualessDF = cleanCols(liquidacioneMensualessDF, col, possibleVals)

# Dejamos Solo las Columnas Necesarias
liquidacioneMensualessDF = liquidacioneMensualessDF[list(colsliqs.keys())]

print('🆔Liquidaciones del Mes Cargadas con Éxito, {} filas'.format(len(liquidacioneMensualessDF)))

# --------- liqs Anuales -------------------
nombresHojasliqs = ['BD ' + str(y) for y in range(minYear, today.year+1)]

liqDFList = []

for name in nombresHojasliqs:
  liqs = _retry(lambda: liqsSH.worksheet(name))
  liqsDF = gettingAsDF(liqs, cellRange)
  # --- Limpieza de Columnas ---
  for col, possibleVals in colsliqs.items():
    liqsDF = cleanCols(liqsDF, col, possibleVals)
  # Dejamos las Columnas Necesarias
  liqsDF = liqsDF[list(colsliqs.keys())]
  # Agregamos el DF a la Lista
  liqDFList.append(liqsDF)
  print('🆔Liquidaciones para {} cargadas con éxito: {} filas'.format(name, len(liqsDF)))

liqsAnualesDF = pd.concat(liqDFList, ignore_index=True)

# Combinar liqs de Año con liqs Mensuales
liqsDF = pd.concat([liqsAnualesDF, liquidacioneMensualessDF], ignore_index=True)

# Volvemos Id_Deuda y Ref_Liq a String
liqsDF['Id_Deuda'] = liqsDF['Id_Deuda'].apply(lambda s: str(s).replace('.0',''))

# Volvemos la Columna Fecha_Liq a Datetime
liqsDF['Fecha_Liq'] = pd.to_datetime(liqsDF['Fecha_Liq'], errors='coerce',dayfirst=True)

# Cambiamos la Fecha_Liq a 23:59:59
liqsDF['Fecha_Liq'] = liqsDF['Fecha_Liq'].apply(lambda dt: dt.replace(hour=23, minute=59, second=59))

# Obtenemos la Primera Fecha sin Mes_Liq (NaN)
firstDate = liqsDF[liqsDF['Mes_Liq'].isna()]['Fecha_Liq'].min()

# Imputamos Mes_Liq con firsDate.month y Año_Liq con firstDate.year
imputeNans(liqsDF, 'Mes_Liq', firstDate.month)
imputeNans(liqsDF, 'Año_Liq', firstDate.year)

# Imputamos Portafolio_Liq con NO
imputeNans(liqsDF, 'Portafolio_Liq', 'NO')

# Limpiamos PaB_Liq y Ingreso_Liq
liqsDF['PaB_Liq'] = liqsDF['PaB_Liq'].apply(cleanNumber)
liqsDF['Ingreso_Liq'] = liqsDF['Ingreso_Liq'].apply(cleanNumber)

# Volvemos las Columnas de PaB_Liq, PaB_Or_Liq y Ingreso_Liq a Numerico
liqsDF['PaB_Liq'] = pd.to_numeric(liqsDF['PaB_Liq'], errors='coerce')
liqsDF['Ingreso_Liq'] = pd.to_numeric(liqsDF['Ingreso_Liq'], errors='coerce')

# Dejamos Datos de liqs este Mes
maskEsteMes = (liqsDF['Fecha_Liq'] >= startDate) & (liqsDF['Fecha_Liq'] <= endDate)
liqsDF = liqsDF[maskEsteMes]

# Quitamos Datos donde Fecha_Liq sea NaN
liqsDF = liqsDF.dropna(subset=['Fecha_Liq'])

# Eliminamos Duplicados de Id_Deuda dejando el primer registro
liqsDF = liqsDF.drop_duplicates(subset=['Id_Deuda'], keep='first')

# Creamos la Columna 'Fecha_Cartera' como: pd.Timestamp(Año_Liq,Mes_Liq,1)
liqsDF['Fecha_Cartera'] = liqsDF.apply(
    lambda row: pd.Timestamp(int(row['Año_Liq'] if row['Año_Liq'] else '2025'),int(row['Mes_Liq'] if row['Mes_Liq'] else '12'),1), axis=1
)

# --- Creación de Buscador de Período ---
def crearBuscadorPeriodo(liqs: pd.DataFrame):
    # Paso 1: Obtener la Fecha Máxima de Liquidación por Mes Operativo
    limites = (
        liqs.groupby("Fecha_Cartera")["Fecha_Liq"]
        .max()
        .reset_index(name="Fecha_Max_Liq")
        .sort_values("Fecha_Max_Liq")
    )

    # Paso 2: Cambiamos a Numpy para mayor velocidad
    periodos = limites["Fecha_Cartera"].dt.tz_localize(None).to_numpy()
    fechasMax = limites["Fecha_Max_Liq"].dt.tz_localize(None).to_numpy()

    # Paso 3: Construcción del Buscador de Periodo
    def buscarPeriodo(fecha: pd.Timestamp):
        fecha = fecha.normalize()

        # Eliminamos timezone si existe
        if fecha.tzinfo is not None:
          fecha = fecha.tz_localize(None)

        fecha_np = np.datetime64(fecha, "ns")

        # Buscamos el primer límite superior >= a la fecha
        idx = np.searchsorted(fechasMax, fecha_np, side="left")

        # Si existe un periodo que contenga la fecha, lo devolvemos
        return pd.Timestamp(periodos[idx]) if idx < len(periodos) else (fecha_cartera_ultima if max(fechasMax) < fecha_np else None)

    # Devolvemos la función de búsqueda
    return buscarPeriodo

# Creamos el Buscador
buscadorFC = crearBuscadorPeriodo(liqsDF)

# Quitamos las Columnas Año_Liq, Fecha_Liq y Mes_Liq
liqsDF = liqsDF.drop(columns=['Año_Liq','Mes_Liq'])

# Imprimimos para logs
print('✅Liquidaciones Cargadas con éxito, columnas: {}'.format(', '.join(liqsDF.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(liqsDF)))

✖️No se realizó el cambió para Mes_Liq con Mes posibles valores.
✖️No se realizó el cambió para Año_Liq con Año posibles valores.
🆔Liquidaciones del Mes Cargadas con Éxito, 288 filas
🆔Liquidaciones para BD 2026 cargadas con éxito: 12654 filas
✅Liquidaciones Cargadas con éxito, columnas: PaB_Liq, Id_Deuda, Ingreso_Liq, Negociador_Liq, Tipo_Liq, Fecha_Liq, Portafolio_Liq, Fecha_Cartera
🚼Filas totales: 8234 filas.


### **Datos de Actualizaciones**
Los Datos que se obtienen son: **actsDF**
- Id_Deuda
- Fecha_Act
- Origen_Act
- Etiqueta_Act
- Comentario_Act
- PaB_Act
- Etapa_Act
- Fecha_Cartera

In [ ]:
# Abrimos la Spreadsheet de actualizaciones
actsShId = '1O8OHuVhgwhLw8XYEBf1uBzLYrxQ45rPiZecHOnAa1Go'
actsSH = _retry(lambda: client.open_by_key(actsShId))
# Abrimos la Worksheet f'act startDate.year'
actsWS = _retry(lambda: actsSH.worksheet(f'act {startDate.year}'))
# Cargamos los Datos
actsDF = _retry(lambda: get_as_dataframe(actsWS, evaluate_formulas=True))

# Dejamos solo las columnas necesarias
actsCols = ['inserted_at','Status_Act','debt_id','payment_to_bank','end','observations']
actsDF = actsDF[actsCols]

# Renombramos debt_id a Id_Deuda
actsDF = actsDF.rename(columns={'debt_id':'Id_Deuda',
                            'payment_to_bank':'PaB_Act',
                            'Status_Act':'Etiqueta_Act',
                            'inserted_at':'Fecha_Act',
                            'end': 'Origen_Act',
                            'observations': 'Comentario_Act',
}) # type: ignore

# Quitamos Actualizaciones de CORREOS_QUITAR.split(,)
actsDF = actsDF[~(actsDF['Origen_Act'].isin(CORREOS_QUITAR.split(',')))]

# Convertimos la columna debt_id a string
actsDF['Id_Deuda'] = actsDF['Id_Deuda'].apply(lambda s: str(s).replace('.0','').strip())

# Volvemos Fecha_Act a String y quitamos el Apostrofe
actsDF['Fecha_Act'] = actsDF['Fecha_Act'].apply(lambda s: str(s).replace("'",''))

# Volvemos Fecha_Act a Datetime
actsDF['Fecha_Act'] = pd.to_datetime(actsDF['Fecha_Act'], errors='coerce', dayfirst=False)
# Quitamos 5 horas
actsDF['Fecha_Act'] = actsDF['Fecha_Act'] - pd.Timedelta(hours=5)

# Dejamos las Actualizaciones en el Intervalo Correcto
actsDF = actsDF[(actsDF['Fecha_Act'] >= startDate) & (actsDF['Fecha_Act'] <= endDate)]

# Agregamos la Columna Fecha_Cartera usando el buscadorFC
actsDF['Fecha_Cartera'] = actsDF['Fecha_Act'].apply(buscadorFC)

# Quitamos Datos con Fecha_Cartera NaN
actsDF = actsDF.dropna(subset=['Fecha_Cartera'])

# Creamos Funcion para extraer el payment_to_bank ya que esta en formato: (###,COP)
def getPaymentToBank(s: str) -> float:
  if not s:
    return np.nan
  return float(s.split(',')[0][1:]) / 100
# Cambiamos el payment_to_bank a la nueva versión
actsDF['PaB_Act'] = actsDF['PaB_Act'].apply(getPaymentToBank)

# Volvemos Etiqueta_Act a String y limpiamos el texto
actsDF['Etiqueta_Act'] = actsDF['Etiqueta_Act'].apply(cleanText)

# Eliminamos la Etiqueta_Act que empiece con CREDIT
actsDF = actsDF[~actsDF['Etiqueta_Act'].str.startswith('CREDIT')]

# Ahora Limpiamos el Origen quitando todo lo que siga despues del @
actsDF['Origen_Act'] = actsDF['Origen_Act'].apply(lambda s: str(s).split('@')[0].strip().lower())

# Ahora reemplazamos saltos de linea en Comentario_Act por ' - '
actsDF['Comentario_Act'] = actsDF['Comentario_Act'].str.replace('\n', ' - ')

# Eliminamos Duplicados
actsDF = actsDF.drop_duplicates(keep='last',subset=['Id_Deuda','Fecha_Act'])

# Creamos Columna Etapa_Act usando el MAPPEADOR_ETIQUETAS_ETAPA
actsDF['Etapa_Act'] = actsDF['Etiqueta_Act'].apply(lambda x: MAPPEADOR_ETIQUETAS_ETAPA.get(x, -1))

# Añadimos Temporalmente Columna Valor_Real como el Mappeo por Id_Deuda usando debtPaBMap y redondeando a 0
actsDF['Valor_Real'] = actsDF['Id_Deuda'].map(debtPaBMap).round(0)
# Redondeamos ahora también PaB_Act
actsDF['PaB_Act'] = actsDF['PaB_Act'].round(0)

# --- Ajuste para CAMBIO DE PAB ---
maskChangePAB = actsDF['Etiqueta_Act'].str.contains("cambio",case=False,na=False) & actsDF['Etiqueta_Act'].str.contains("pab",case=False,na=False)
maskwithDisc = (actsDF['PaB_Act'] <= actsDF['Valor_Real'] * (1-DESC_CONSIDERAR_EFECTIVA))
# Ajustamos también Etapa_Act a 2 cuando PaB_Act <= Valor_Real * (1-DESC_CONSIDERAR_EFECTIVA)
actsDF.loc[maskChangePAB & maskwithDisc, 'Etapa_Act'] = 2

# Ahora: Donde hay Cambio de PaB y no hay Descuento, cambiamos la Etiqueta a "CAMBIO DE PAB - Sin Descuento"
actsDF.loc[maskChangePAB & ~maskwithDisc, 'Etiqueta_Act'] = 'CAMBIO DE PAB - Sin Descuento'

# Por Último, quitamos la Columna Valor_Real
actsDF = actsDF.drop(columns=['Valor_Real'])

# Imprimimos para logs
print('✅Actualizaciones Totales creados con éxito, columnas: {}'.format(', '.join(actsDF.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(actsDF)))

# Verificamos si existe alguna etiqueta con Etapa_Act -1
if (actsDF['Etapa_Act'] == -1).any():
  # Mostramos una alerta y luego un ValueError
  print('❌Error: Etiquetas sin Etapa')
  print(actsDF[actsDF['Etapa_Act'] == -1]['Etiqueta_Act'].value_counts())
  raise ValueError('❌Error: Etiquetas sin Etapa')

✅Actualizaciones Totales creados con éxito, columnas: Fecha_Act, Etiqueta_Act, Id_Deuda, PaB_Act, Origen_Act, Comentario_Act, Fecha_Cartera, Etapa_Act
🚼Filas totales: 184112 filas.


## **Definición de Mejor Mes**
___
En caso de ser necesario, se procede a calcular el mejor mes, de lo contrario, se continua con la ejecución normal. Los datos que se obtienen son: **FECHA_MEJOR_MES: Optional[pd.Timestamp]**

In [ ]:
if CALCULAR_MEJOR_MES:
  # --- Unión de la Información ---
  # Primero Unimos clientesDF con actsDF por Id_Deuda y Fecha_Cartera
  unionDF = pd.merge(clientesDF, actsDF, on=['Id_Deuda','Fecha_Cartera'], how='left')
  # Ahora creamos el Descuento_Act como: 1 - (PaB_Act/Dum_Bravo)
  unionDF['Descuento_Act'] = 1 - (unionDF['PaB_Act'] / unionDF['Dum_Bravo'])
  # Ajustamos donde Dum_Bravo sea 0 con 0
  unionDF.loc[unionDF['Dum_Bravo'] == 0, 'Descuento_Act'] = 0
  # Creamos Columna Desc_Viable como Descuento_Act.between(DESC_CONSIDERAR_EFECTIVA,0.98)
  unionDF['Desc_Viable'] = unionDF['Descuento_Act'].between(DESC_CONSIDERAR_EFECTIVA,0.98)
  # Ordenamos por Fecha_Cartera, Desc_Viable y Descuento_Act
  unionDF = unionDF.sort_values(by=['Fecha_Cartera','Desc_Viable','Descuento_Act'], ascending=False, na_position='last')
  # Dejamos el Primer Registro por Fecha_Cartera e Id_Deuda
  unionDF = unionDF.drop_duplicates(subset=['Fecha_Cartera','Id_Deuda'], keep='first')

  # --- Cálculo de Métricas ---
  # Creamos Columna Tiene_Act como Fecha_Act.notna
  unionDF['Tiene_Act'] = unionDF['Fecha_Act'].notna()
  # Creamos Columna Tiene_Act_Desc como Tiene_Act & Descuento_Act.between(DESC_CONSIDERAR_EFECTIVA,0.98)
  unionDF['Tiene_Act_Desc'] = unionDF['Tiene_Act'] & (unionDF['Descuento_Act'].between(DESC_CONSIDERAR_EFECTIVA,0.98))
  # Agrupamos por Fecha_Cartera y dejamos:
  # Countunique de Id_Deuda, Suma de Tiene_Act y Suma de Tiene_Act_Desc
  unionDF = unionDF.groupby('Fecha_Cartera').agg({'Id_Deuda': 'nunique', 'Tiene_Act': 'sum', 'Tiene_Act_Desc': 'sum'}).reset_index()
  # Calculamos %Act y %Act_Desc
  unionDF['%Act'] = unionDF['Tiene_Act'] / unionDF['Id_Deuda']
  unionDF['%Act_Desc'] = unionDF['Tiene_Act_Desc'] / unionDF['Id_Deuda']
  # Calculamos KPI como %Act * %Act_Desc
  unionDF['KPI'] = unionDF['%Act'] * unionDF['%Act_Desc']
  # Ordenamos los Datos por Fecha_Cartera
  unionDF = unionDF.sort_values(by=['Fecha_Cartera'])

  # Mostramos cada una de la fila los resultados
  for _, row in unionDF.iterrows():
    print("📅Fecha de Cartera: {}, 🆔Deudas: {:,.0f}, ℹ️Acts: {:,.0f} | ✅Acts con Desc: {:,.0f} --> 🛠️%Act: {:.2%} | 😁%Act con Desc: {:.2%}. ❤️KPI: {:.4f}".format(
      row['Fecha_Cartera'].strftime('%d/%m/%Y'),
      row['Id_Deuda'],
      row['Tiene_Act'],
      row['Tiene_Act_Desc'],
      row['%Act'],
      row['%Act_Desc'],
      row['KPI'],
    ))

  # Ahora Escogemos la Fecha_Cartera con Mayor valor de KPI
  # Primero ordenamos los Datos por KPI de mayor a menor
  unionDF = unionDF.sort_values(by=['KPI'], ascending=False)
  # Ahora escogemos el Primer Registro
  unionWinner = unionDF.head(1).iloc[0]
  # Mostramos la Fecha Ganadora
  print('\n📅✅😁 Fecha Ganadora: [{}]'.format(
    unionWinner['Fecha_Cartera'].strftime('%Y-%m | %B-%Y')
  ))

  # Por último definimos la variable FECHA_MEJOR_MES
  FECHA_MEJOR_MES = unionWinner['Fecha_Cartera']

  # Verificamos la FECHA_MEJOR_MES cambió, y en caso de que no, se deja como None ya que se cuenta con los datos relevantes
  if (FECHA_MEJOR_MES.strftime("%Y-%m") == controlDict['Mejor-Mes']):
    print('👍La Fecha de Mejor mes se mantiene, por lo que no es necesario recalcular todo')
    FECHA_MEJOR_MES = None
else:
  print('😁No es Necesario cambiar el Mejor Mes (ℹ️Actual: {})'.format(
    controlDict['Mejor-Mes']
  ))
  FECHA_MEJOR_MES = None

📅Fecha de Cartera: 01/05/2026, 🆔Deudas: 40,079, ℹ️Acts: 11,253 | ✅Acts con Desc: 5,579 --> 🛠️%Act: 28.08% | 😁%Act con Desc: 13.92%. ❤️KPI: 0.0391
📅Fecha de Cartera: 01/06/2026, 🆔Deudas: 38,528, ℹ️Acts: 12,336 | ✅Acts con Desc: 6,007 --> 🛠️%Act: 32.02% | 😁%Act con Desc: 15.59%. ❤️KPI: 0.0499
📅Fecha de Cartera: 01/07/2026, 🆔Deudas: 38,535, ℹ️Acts: 11,177 | ✅Acts con Desc: 6,409 --> 🛠️%Act: 29.00% | 😁%Act con Desc: 16.63%. ❤️KPI: 0.0482
📅Fecha de Cartera: 01/08/2026, 🆔Deudas: 40,349, ℹ️Acts: 11,316 | ✅Acts con Desc: 6,303 --> 🛠️%Act: 28.05% | 😁%Act con Desc: 15.62%. ❤️KPI: 0.0438
📅Fecha de Cartera: 01/09/2026, 🆔Deudas: 39,732, ℹ️Acts: 11,292 | ✅Acts con Desc: 6,694 --> 🛠️%Act: 28.42% | 😁%Act con Desc: 16.85%. ❤️KPI: 0.0479
📅Fecha de Cartera: 01/10/2026, 🆔Deudas: 41,368, ℹ️Acts: 11,513 | ✅Acts con Desc: 6,605 --> 🛠️%Act: 27.83% | 😁%Act con Desc: 15.97%. ❤️KPI: 0.0444
📅Fecha de Cartera: 01/11/2026, 🆔Deudas: 42,596, ℹ️Acts: 7,754 | ✅Acts con Desc: 4,330 --> 🛠️%Act: 18.20% | 😁%Act con Desc: 1

# **Obtención de Datos**
___
Los datos que se requieren para el seguimiento tradicional:
- **Datos de Depósitos**
- **Datos de Bajas**
- **Datos de Crédito**
- **Datos de Headcount**
- **Datos de Status de Crédito**
- **Datos de Solicitudes de Alianzas**

## **Datos de Depósitos**
Los Datos que se obtienen son: **depositosDF y depositosBMDF**
- Referencia
- Meses_Sin_Deposito

In [ ]:
# Abrimos la Spreadsheet
depositosShId = '1_W8NAWqTY9yJBzqWx1PoLy7eRnDE_Ln16LVz3u9kkGg'
depositosSH = _retry(lambda: client.open_by_key(depositosShId))
# Ahora abrimos la hoja llamada 'dep-{año}'
depositosWS = _retry(lambda: depositosSH.worksheet(f'dep-{startDate.year}'))
# Obtenemos los Datos como un DF
depositosDF = _retry(lambda: get_as_dataframe(depositosWS, evaluate_formulas=True))

# Dejamos las Columnas Requeridas
depositosCols = ['Referencia','Fecha_Transac']
depositosDF = depositosDF[depositosCols]

# Renombramos la Columna Fecha_Transac a Fecha_Deposito
depositosDF = depositosDF.rename(columns={'Fecha_Transac':'Fecha_Deposito'})

# Volvemos la Columna Fecha_Deposito a Datetime
depositosDF['Fecha_Deposito'] = pd.to_datetime(depositosDF['Fecha_Deposito'], errors='coerce', dayfirst=False)

# Volvemos la Columna Referencia a String
depositosDF['Referencia'] = depositosDF['Referencia'].apply(lambda s: str(s).replace('.0',''))

# Aplicamos los cambios de referencia si es que existen
depositosDF['Referencia'] = depositosDF['Referencia'].apply(lambda x: refChangesDict.get(x, x))

# Ahora: Si existe FECHA_MEJOR_MES, se debe de crear depositosBMDF, y aplicar el mismo preprocesamiento
if not (FECHA_MEJOR_MES is None):
  # Creamos la Copia
  depositosBMDF = depositosDF.copy()
  # Dejamos datos inferiores a final de mes de FECHA_MEJOR_MES
  depositosBMDF = depositosBMDF[depositosBMDF['Fecha_Deposito'] <= (FECHA_MEJOR_MES + pd.offsets.MonthEnd(0))]
  # Eliminamos Duplicados por Referencia dejando la máxima Fecha_Deposito
  depositosBMDF = depositosBMDF.sort_values(by=['Referencia','Fecha_Deposito'], ascending=False)
  depositosBMDF = depositosBMDF.drop_duplicates(subset=['Referencia'], keep='first')
  # Creamos la Columna Meses_Sin_Deposito a FECHA_MEJOR_MES
  depositosBMDF['Meses_Sin_Deposito'] = (FECHA_MEJOR_MES.year - depositosBMDF['Fecha_Deposito'].dt.year) * 12 + (FECHA_MEJOR_MES.month - depositosBMDF['Fecha_Deposito'].dt.month)
  # Quitamos la Columna Fecha_Deposito
  depositosBMDF = depositosBMDF.drop(columns=['Fecha_Deposito'])
else:
  depositosBMDF = pd.DataFrame()


# Dejamos el Último Deposito por Referencia
depositosDF = depositosDF.sort_values(by=['Referencia','Fecha_Deposito'], ascending=False)
depositosDF = depositosDF.drop_duplicates(subset=['Referencia'], keep='first')

# Ahora creamos la Columna de Meses_Sin_Deposito que es la diferencia en meses entre la Fecha_Deposito y endDate
depositosDF['Meses_Sin_Deposito'] = (endDate.year - depositosDF['Fecha_Deposito'].dt.year) * 12 + (endDate.month - depositosDF['Fecha_Deposito'].dt.month)

# Quitamos la Columna de Fecha_Deposito porque no es necesaria, se tiene la de Meses_Sin_Deposito que es la relevante para el análisis
depositosDF = depositosDF.drop(columns=['Fecha_Deposito'])

print('✅Depositos creadas con éxito, columnas: {}'.format(', '.join(depositosDF.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(depositosDF)))

✅Depositos creadas con éxito, columnas: Referencia, Meses_Sin_Deposito
🚼Filas totales: 19499 filas.


## **Datos de Bajas**
Los datos que se obtienen son: **bajasDF**
- Referencia
- Fecha_Baja

In [ ]:
# Abrimos la Spreadsheet de Bajas
bajasShId = '1FN1u5b6N5mzcrzYzLKpBjwSkhddgYPOrgyZgbKChc68'
bajasSH = _retry(lambda: client.open_by_key(bajasShId))
# Abrimos la Worksheet llamada 'HISTORICO DE BAJAS FINANZAS'
bajasWS = _retry(lambda: bajasSH.worksheet('HISTORICO DE BAJAS FINANZAS'))
# Cargamos los Datos saltando la primera fila
bajasDF = _retry(lambda: get_as_dataframe(bajasWS, evaluate_formulas=True))

# Dejamos las Columnas Requeridas
bajasCols = ['REFERENCIA','FECHA ENVIO']
bajasDF = bajasDF[bajasCols]

# Renombramos la Columna FECHA ENVIO a Fecha_Baja
bajasDF = bajasDF.rename(columns={'REFERENCIA':'Referencia','FECHA ENVIO':'Fecha_Baja'}) # type: ignore

# Cambiamos la Columna Referencia a String
bajasDF['Referencia'] = bajasDF['Referencia'].apply(lambda s: str(s).replace('.0',''))

# Aplicamos los cambios de referencia si es que existen
bajasDF['Referencia'] = bajasDF['Referencia'].apply(lambda x: refChangesDict.get(x, x))

# Volvemos Fecha_Baja a Datetime con dayfirst = True
bajasDF['Fecha_Baja'] = pd.to_datetime(bajasDF['Fecha_Baja'], errors='coerce', dayfirst=True)

print('✅Bajas creadas con éxito, columnas: {}'.format(', '.join(bajasDF.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(bajasDF)))

✅Bajas creadas con éxito, columnas: Referencia, Fecha_Baja
🚼Filas totales: 10386 filas.


## **Datos de Crédito**
Los Datos que se obtienen son:

**pabIdealDF y pabIdealBMDF**
- Id_Deuda
- PaB_Ideal_Credito

**potCredDF y potCredBMDF**
- Referencia
- Batch_Credito
- Potencial_Credito
- Bucket_Credito

In [ ]:
## Abrimos el Sheets de Batches
creditoId = '1R1hjdSXwbh_gYfQ8id3rKlh0Xbn_uc465gD0guzcie0'
creditoSH = _retry(lambda: client.open_by_key(creditoId))

# Abrimos el Sheets de PaB Ideal
pabIdealSHId = '1Obm0O5hfIIzCMy5RvdX5b1JBf3pmzIrYdYa1vPOB83M'
pabIdealSH = _retry(lambda: client.open_by_key(pabIdealSHId))

def load_datos_cred(sheet_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
  # Abrimos la Hoja
  creditoWS = _retry(lambda: creditoSH.worksheet(sheet_name))
  # Obtenemos los datos
  potCredDF = _retry(lambda: get_as_dataframe(creditoWS, evaluate_formulas=True))
  # Renombramos Columna Potencial a Potencial_Credito
  potCredDF = potCredDF.rename(columns={'Potencial':'Potencial_Credito','Origen':'Batch_Credito',"Bucket CE": "Bucket_Credito"}) # type: ignore

  # Volvemos la Columna Referencia a String
  potCredDF['Referencia'] = potCredDF['Referencia'].apply(lambda s: str(s).replace('.0',''))

  # Aplicamos los cambios de referencia si es que existen
  potCredDF['Referencia'] = potCredDF['Referencia'].apply(lambda x: refChangesDict.get(x, x))

  # Vamos a dejar el Último Registro por Referencia primero ordenando por Potencial_Credito con nans first
  potCredDF = potCredDF.sort_values(by=['Referencia','Potencial_Credito'], na_position='first')
  potCredDF = potCredDF.drop_duplicates(subset=['Referencia'], keep='last')

  # Dejamos solo las Columnas Necesarias
  potCredDF = potCredDF[['Referencia','Potencial_Credito','Batch_Credito',"Bucket_Credito"]]

  # Imprimimos para Logs
  print('✅Potenciales de Crédito Cargados con éxito, columnas: {}'.format(', '.join(potCredDF.columns.tolist())))
  print('🚼Filas totales: {} filas.'.format(len(potCredDF)))

  # Abrimos la Hoja del mismo nombre
  pabIdealWS = _retry(lambda: pabIdealSH.worksheet(sheet_name))
  # Obtenemos los datos
  pabIdealDF = _retry(lambda: get_as_dataframe(pabIdealWS, evaluate_formulas=True))
  # Renombramos Columna PB Ideal a PaB_Ideal_Credito
  pabIdealDF = pabIdealDF.rename(columns={'PB Ideal':'PaB_Ideal_Credito','Id deuda':'Id_Deuda'}) # type: ignore

  # Volvemos la Columna Id_Deuda a String
  pabIdealDF['Id_Deuda'] = pabIdealDF['Id_Deuda'].apply(lambda s: str(s).replace('.0','').strip())

  # Volvemos la Columna PaB_Ideal_Credito a Numerica
  pabIdealDF['PaB_Ideal_Credito'] = pd.to_numeric(pabIdealDF['PaB_Ideal_Credito'], errors='coerce')

  # Quitamos Datos sin PaB_Ideal_Credito porque no aportan información para el análisis
  pabIdealDF = pabIdealDF.dropna(subset=['PaB_Ideal_Credito'])

  # Quitamos Datos donde PaB_Ideal_Credito sea 0
  pabIdealDF = pabIdealDF[pabIdealDF['PaB_Ideal_Credito'] != 0.0]

  # Quitamos Duplicados de PaB_Ideal_Credito dejando el último registro
  pabIdealDF = pabIdealDF.drop_duplicates(subset=['Id_Deuda'], keep='last')

  # Dejamos solo las Columnas Necesarias
  pabIdealDF = pabIdealDF[['Id_Deuda','PaB_Ideal_Credito']]

  # Imprimimos para Logs
  print('\n✅PaB Ideal de Crédito Cargado con éxito, columnas: {}'.format(', '.join(pabIdealDF.columns.tolist())))
  print('🚼Filas totales: {} filas.\n'.format(len(pabIdealDF)))

  print('⚛️Datos de "{}" Leídos Correctamente\n___\n'.format(sheet_name))
  return potCredDF, pabIdealDF

# Primero Cargamos los del MEC según fecha_cartera_ultima
sheetUltCartera = f"{mesesDict[fecha_cartera_ultima.month]}-{fecha_cartera_ultima.year%100}"
potCredDF, pabIdealDF = load_datos_cred(sheetUltCartera)

# Ahora, si FECHA_MEJOR_MES existe, se cargan los datos para esa fecha
if not (FECHA_MEJOR_MES is None):
  sheetBestMonth = f"{mesesDict[FECHA_MEJOR_MES.month]}-{FECHA_MEJOR_MES.year%100}"
  potCredBMDF, pabIdealBMDF = load_datos_cred(sheetBestMonth)
else:
  potCredBMDF = pd.DataFrame()
  pabIdealBMDF = pd.DataFrame()

✅Potenciales de Crédito Cargados con éxito, columnas: Referencia, Potencial_Credito, Batch_Credito, Bucket_Credito
🚼Filas totales: 3870 filas.

✅PaB Ideal de Crédito Cargado con éxito, columnas: Id_Deuda, PaB_Ideal_Credito
🚼Filas totales: 7937 filas.

⚛️Datos de "Noviembre-26" Leídos Correctamente
___

✅Potenciales de Crédito Cargados con éxito, columnas: Referencia, Potencial_Credito, Batch_Credito, Bucket_Credito
🚼Filas totales: 3689 filas.

✅PaB Ideal de Crédito Cargado con éxito, columnas: Id_Deuda, PaB_Ideal_Credito
🚼Filas totales: 7731 filas.

⚛️Datos de "Junio-26" Leídos Correctamente
___



## **Datos de Headcount**
Los Datos que se obtienen son: **headCountDF**
- Negociador_HC
- Email

In [ ]:
# Obtenemos la Spreadsheet de HeadCount
headCountShId = '1KO4ImvhNZB_jtgpvs9DU-6_0FskFmxC9Xo4Rz5Yt6dM'
headCountSH = _retry(lambda: client.open_by_key(headCountShId))
# Obtenemos la Worksheet llamada 'HC'
headCountWS = _retry(lambda: headCountSH.worksheet('HC'))
# Obtenemos los Datos como un DataFrame
headCountDF = _retry(lambda: get_as_dataframe(headCountWS, evaluate_formulas=True))

# Dejamos solo los datos donde status == 'active'
# headCountDF = headCountDF[headCountDF['status'] == 'active']

# Renombramos a Columnas más Amigables
headCountDF = headCountDF.rename(columns={'email':'Email','name':'Negociador_HC','leader':'Lider'}) # type: ignore

# Limpiamos el Nombre del Negociador_HC
headCountDF['Negociador_HC'] = headCountDF['Negociador_HC'].apply(lambda s: (cleanText(s)).strip().title())

# Dejamos Solo Columnas de Email y Negociador_HC
headCountDF = headCountDF[['Email','Negociador_HC','Lider']]

# Ahora al Email le quitamos los espacios y dejamos todo antes del @
headCountDF['Email'] = headCountDF['Email'].apply(lambda s: str(s).split('@')[0].replace(' ','').strip().lower())

# Imprimimos para logs
print('✅Headcount creadas con éxito, columnas: {}'.format(', '.join(headCountDF.columns.tolist())))
print('🚼Filas totales: {} filas.'.format(len(headCountDF)))

✅Headcount creadas con éxito, columnas: Email, Negociador_HC, Lider
🚼Filas totales: 509 filas.


## **Datos de Prospectos de Liquidación**
___
Los datos que se obtienen son: **prospDF y prospBMDF**
- Id_Deuda
- Prosp_Liq
- Prob_Liq

In [ ]:
# Abrimos la hoja de Prospectos
prospSHId = '1lIfUXupnXC3CCCFDY3wbzTEhPmBiFmkvv37XSmVpee0'
prospSH = _retry(lambda: client.open_by_key(prospSHId))
# Abrimos la Worksheet 'Prosp_MEC_D_Liq'
prospWS = _retry(lambda: prospSH.worksheet('Prosp_MEC_D_Liq'))
# Cargamos los Datos
prospDF = _retry(lambda: get_as_dataframe(prospWS, evaluate_formulas=True))

# Renombramos Columnas
prospDF = prospDF.rename(columns={
    'Prospecto Liq': 'Prosp_Liq'
}) # type: ignore

# Dejamos solo las Columnas Necesarias
prospDF = prospDF[['Id_Deuda','Prosp_Liq']]

# Quitamos Datos con NaNs
prospDF = prospDF.dropna(subset=['Prosp_Liq','Id_Deuda'])

# Volvemos Id_Deuda a String
prospDF['Id_Deuda'] = prospDF['Id_Deuda'].apply(lambda s: str(s).replace('.0',''))

# Ahora, si existe FECHA_MEJOR_MES, buscamos los prospectos para ese mes
if not (FECHA_MEJOR_MES is None):
  # Abrimos la Worksheet 'Prosp_{mesesDict[FECHA_MEJOR_MES.month]}-{FECHA_MEJOR_MES.year%100}'
  try:
    prospWS = _retry(lambda: prospSH.worksheet(f'Prosp_{mesesDict[FECHA_MEJOR_MES.month]}-{FECHA_MEJOR_MES.year%100}'))
    # Cargamos los Datos
    prospBMDF = _retry(lambda: get_as_dataframe(prospWS, evaluate_formulas=True))
    # Renombramos Columnas
    prospBMDF = prospBMDF.rename(columns={
      'Prospecto Liq': 'Prosp_Liq'
    }) # type: ignore

    # Volvemos Id_Deuda a String
    prospBMDF['Id_Deuda'] = prospBMDF['Id_Deuda'].apply(lambda s: str(s).replace('.0',''))
  except WorksheetNotFound:
    prospBMDF = pd.DataFrame(columns=['Id_Deuda','Prosp_Liq'])
else:
  prospBMDF = pd.DataFrame()

# Imprimimos para logs
print('✅Prospectos de Liquidación creados con éxito, columnas: {}'.format(', '.join(prospDF.columns.tolist())))
print('ℹ️Filas totales: {} filas.'.format(len(prospDF)))

✅Prospectos de Liquidación creados con éxito, columnas: Id_Deuda, Prosp_Liq
ℹ️Filas totales: 1200 filas.


## **Datos de Status de Crédito**
___
Los datos que se obtienen son: **pipecredDF y pipecredBMDF**
- Referencia
- Fecha_Pipeline
- Status_Credito

In [ ]:
# Ahora abrimos los Datos del Pipeline de crédito
pipcredSHId = '1xHJ_yVmeQiE0PFOrmfPEXtKKHp8ZEoQQYiruzfXS-2Q'
pipcredSH = _retry(lambda: client.open_by_key(pipcredSHId))
# Abrimos la Hoja 'Pipeline Crédito'
pipcredWS = _retry(lambda: pipcredSH.worksheet('Pipeline Crédito'))
# Cargamos los Datos
pipecredDF = _retry(lambda: get_as_dataframe(pipcredWS, evaluate_formulas=True))

# Volvemos la Columna Referencia a String
pipecredDF['Referencia'] = pipecredDF['Referencia'].apply(lambda s: str(s).replace('.0','').strip())
# Aplicamos los Cambios de Referencia
pipecredDF['Referencia'] = pipecredDF['Referencia'].apply(lambda s: refChangesDict.get(s,s))
# Renombramos Cosecha a Fecha_Cartera, Status a Status_PP_Cred y Fecha de Subida a Fecha_Pipeline
pipecredDF = pipecredDF.rename(columns={
    'Status': 'Status_Credito',
    "Fecha de Subida": 'Fecha_Pipeline',
})
# Dejamos solo las Columnas Necesarias
pipecredDF = pipecredDF[['Referencia','Status_Credito','Fecha_Pipeline']]

# Volvemos Fecha_Pipeline a Datetime con dayfirst = False
pipecredDF['Fecha_Pipeline'] = pd.to_datetime(pipecredDF['Fecha_Pipeline'], errors='coerce', dayfirst=False)

# Quitamos Datos con NaNs
pipecredDF = pipecredDF.dropna(subset=['Fecha_Pipeline','Status_Credito'])

print('✅Datos del Pipeline del Crédito Cargados con Éxito')
print('ℹ️Filas Totales: {}'.format(
    len(pipecredDF)
))

# Ahora, si FECHA_MEJOR_MES existe cargamos el histórico
if not (FECHA_MEJOR_MES is None):
  # Abrimos la Hoja del Pipeline de Crédito
  pipecredBMSHId = "1QYBylPbwiLw-Yy5KxI3K1Dq0NmUYvqhaLUFnS5YusJ0"
  pipecredBMSH = _retry(lambda: client.open_by_key(pipecredBMSHId))
  pipecredBMWS = _retry(lambda: pipecredBMSH.worksheet('Total'))

  # Cargamos los Datos como un DF
  pipecredBMDF = _retry(lambda: get_as_dataframe(pipecredBMWS, evaluate_formulas=True, parse_dates=['Cosecha']))

  # Volvemos la Columna Referencia a String
  pipecredBMDF['Referencia'] = pipecredBMDF['Referencia'].apply(lambda s: str(s).replace('.0','').strip())
  # Aplicamos los Cambios de Referencia
  pipecredBMDF['Referencia'] = pipecredBMDF['Referencia'].apply(lambda s: refChangesDict.get(s,s))

  # Renombramos Cosecha a Fecha_Cartera, Status a Status_Credito y Substatus a Sub_Status_Credito
  pipecredBMDF = pipecredBMDF.rename(columns={
      'Status': 'Status_Credito',
      'Fecha de Subida': 'Fecha_Pipeline'
  })

  # Dejamos solo las Columnas Necesarias
  pipecredBMDF = pipecredBMDF[['Referencia','Fecha_Pipeline','Status_Credito']]

  # Volvemos Fecha_Pipeline a Datetime usando dayfirst=False
  pipecredBMDF['Fecha_Pipeline'] = pd.to_datetime(pipecredBMDF['Fecha_Pipeline'], errors='coerce', dayfirst=False)
  # Ahora Ordenamos los Datos de mayor a menor por Fecha_Pipeline
  pipecredBMDF = pipecredBMDF.sort_values(by='Fecha_Pipeline', ascending=False)

  print('\n✅Datos del Pipeline del Crédito Histórico Cargados con Éxito')
  print('ℹ️Filas Totales: {}'.format(
    len(pipecredBMDF)
  ))
else:
  print('😁No es necesario cargar el mejor mes de Crédito')

✅Datos del Pipeline del Crédito Cargados con Éxito
ℹ️Filas Totales: 464

✅Datos del Pipeline del Crédito Histórico Cargados con Éxito
ℹ️Filas Totales: 40309


## **Datos de Solicitudes de Alianzas**
___
Los datos que se obtienen son: **statusalyDF y statusalyBMDF**
- Id_Deuda
- Fecha_Sol
- Estado_Sol_Alianzas
- Tipo_Sol_Alianzas

___
Además de esto, se va a crear el DF **originallyDF**, que va a contener los escalamientos de 1:1 y solicitudes con:
- Id_Deuda
- Fecha_Alianzas
- Origen_Alianzas: 1:1

### **Datos de Formulario Antiguo de Alianzas**
___
Los datos que se obtienen son: **oldsolDF y oldsolBMDF**. En este caso solo serán tomadas en cuenta las solicitudes que ya tengan respuesta

In [ ]:
# Abrimos la hoja de solicitudes
solsSHId = '1tlHeLPJgIlRw3-_yv8lG4_w07n44o6KUxwxS1jmhjLk'
solsSH = _retry(lambda: client.open_by_key(solsSHId))
# Abrimos la Hoja llamada 'Solicitudes Alianzas'
solsWS = _retry(lambda: solsSH.worksheet('Solicitudes Alianzas'))
# Cargamos los Datos
oldsolDF = _retry(lambda: get_as_dataframe(solsWS, evaluate_formulas=True))

# Renombramos Columnas
oldsolDF = oldsolDF.rename(columns={
  "ID Deuda": "Id_Deuda",
  "Escriba los ID de TODAS las deudas del portafolio separadas por guion. eje.(145268-455846-...)": "Ids_Deuda",
  "Fecha de respuesta": "Fecha_Sol",
  "Estado": "Estado_Sol_Alianzas",
  "Tipo de solicitud": "Tipo_Sol_Alianzas",
})

# Dejamos solo las Columnas Necesarias
oldsolDF = oldsolDF[["Id_Deuda","Ids_Deuda","Fecha_Sol","Estado_Sol_Alianzas","Timestamp","Tipo_Sol_Alianzas"]]

# Volvemos Id_Deuda e Ids_Deuda a String
oldsolDF['Id_Deuda'] = oldsolDF['Id_Deuda'].apply(lambda s: str(s).replace('.0','').strip())
mkntIds = oldsolDF['Ids_Deuda'].notna()
oldsolDF.loc[mkntIds,'Ids_Deuda'] = oldsolDF.loc[mkntIds,'Ids_Deuda'].apply(lambda s: str(s).replace('.0','').strip())

# Ahora limpiamos los Estados de Solicitud
estadosNotFound = set()
def solstateCleaner(estado: str) -> str:
  if estado == "Solicitud no exitosa":
    return "No Exitosa"
  elif estado == "Solicitud exitosa":
    return "Exitosa"
  elif (estado == "Aliado no responde") or (estado == "No hay contacto"):
    return "No esta con Aliado"
  elif estado == "Solicitud erronea":
    return "Erronea"
  elif (estado == "En espera aliado") or (estado == "Solicitado"):
    return "Solicitado"
  if not (estado in estadosNotFound):
    print('❌Estado ({}) no Cambiado'.format(estado))
    estadosNotFound.add(estado)
  return estado

# Aplicamos la Función a los Estados
oldsolDF['Estado_Sol_Alianzas'] = oldsolDF['Estado_Sol_Alianzas'].apply(solstateCleaner)

# -- Explotar por Ids --
# Primero imputamos Ids_Deuda con Id_Deuda
oldsolDF.loc[~mkntIds,'Ids_Deuda'] = oldsolDF.loc[~mkntIds,'Id_Deuda']
# Ahora Quitamos la Columna Id_Deuda
oldsolDF = oldsolDF.drop(columns=['Id_Deuda'])
# Ahora aplicamos el split por '-' y explotamos el DF
oldsolDF['Ids_Deuda'] = oldsolDF['Ids_Deuda'].str.split()
oldsolDF = oldsolDF.explode(['Ids_Deuda'], ignore_index=True)
# Renombramos Ids_Deuda a Id_Deuda
oldsolDF.rename(columns={"Ids_Deuda":"Id_Deuda"},inplace=True)

# Volvemos Timestamo a Datetime con dayfirst=False
oldsolDF['Timestamp'] = pd.to_datetime(oldsolDF['Timestamp'], dayfirst=False, errors='coerce')

# Vamos a Crear una Copia del DF que servirá para guardar el inicio de la solicitud
iniciooldCopy = oldsolDF.copy(deep=True)
# Quitamos Columna Fecha_Sol y Renombramos Timestamp a Fecha_Sol
iniciooldCopy = iniciooldCopy.drop(columns=['Fecha_Sol'])
iniciooldCopy = iniciooldCopy.rename(columns={'Timestamp':'Fecha_Sol'})
# Cambiamos Estado_Sol_Alianzas a "Sin Tocar"
iniciooldCopy['Estado_Sol_Alianzas'] = "Sin Tocar"
# Quitamos la Columna Timestamp de oldsolDF
oldsolDF = oldsolDF.drop(columns=['Timestamp'])
# Por último, lo concatenamos con el DF Inicial
oldsolDF = pd.concat([iniciooldCopy,oldsolDF], ignore_index=True)

# Volvemos Fecha_Sol a Datetime con dayfirst=False
oldsolDF['Fecha_Sol'] = pd.to_datetime(oldsolDF['Fecha_Sol'], dayfirst=False, errors='coerce')

# Quitamos datos con NaNs en Fecha_Sol o en Estado_Sol_Alianzas
oldsolDF = oldsolDF.dropna(subset=['Fecha_Sol','Estado_Sol_Alianzas'])

# Agregamos la Columna Fecha_Cartera usando el buscadorFC
oldsolDF['Fecha_Cartera'] = oldsolDF['Fecha_Sol'].apply(buscadorFC)

# Eliminamos NaNs por Fecha_Cartera
oldsolDF = oldsolDF.dropna(subset=['Fecha_Cartera'])

# Por Último, separamos BMDF de DF
# BMDF será el que tenga Fecha_Cartera == FECHA_MEJOR_MES
oldsolBMDF = oldsolDF[oldsolDF['Fecha_Cartera'] == FECHA_MEJOR_MES].copy()
# El DF solo será Fecha_Cartera == fecha_cartera_ultima
oldsolDF = oldsolDF[oldsolDF['Fecha_Cartera'] == fecha_cartera_ultima].copy()

# Les quitamos las Columnas Fecha_Cartera a ambos DF
oldsolBMDF = oldsolBMDF.drop(columns=['Fecha_Cartera'])
oldsolDF = oldsolDF.drop(columns=['Fecha_Cartera'])

# Imprimimos para Logs
print('✅Datos de Solicitudes Antiguas Cargados con Éxito')
print('ℹ️Filas de Actual: {}'.format(
    len(oldsolDF)
))
print('🆔Filas Mejor Mes: {}'.format(
  len(oldsolBMDF)
))

❌Estado (nan) no Cambiado
✅Datos de Solicitudes Antiguas Cargados con Éxito
ℹ️Filas de Actual: 0
🆔Filas Mejor Mes: 3087


### **Datos de Formulario Nuevo de Alianzas**
___
Los datos que se obtienen son: **newsolDF y newsolBMDF**

#### **Funciones y Esquemas de los Datos**
___
Funciones para limpiar la información y mantener el esquema de los datos.

In [ ]:
class DeudasSolicitud(TypedDict):
    Id_Deuda: str
    Banco: str
    Numero_Credito: str
    Num_Cuotas: int
    Monto_Actual: NotRequired[float]
    Monto_Propuesto: NotRequired[float]

class MetadataModificacion(TypedDict):
    Comentario_Ejecutivo: NotRequired[str]
    Estado_Comite: NotRequired[int]
    Estado_Titular_Ilocalizable: NotRequired[int]
    Pago_Total_Obligatorio: NotRequired[bool]
    Max_Descuento_Otorgado: NotRequired[bool]
    Es_Directo_Base: NotRequired[bool]
    Addendums: NotRequired[List[DeudasSolicitud]]

class ModificacionSolicitud(TypedDict):
    Fecha_Respuesta: datetime
    Estado_Solicitud: str
    Ejecutivo: str
    JSON_Respuesta: List[DeudasSolicitud]
    Fecha_Limite_Pago: NotRequired[datetime]
    Metadata: MetadataModificacion

class MetadataSolicitud(TypedDict):
    Comentario_Negociador: str
    Nombre_Cliente: str
    Estado_Comite: NotRequired[int]
    Estado_Titular_Ilocalizable: NotRequired[int]
    Pago_Total_Obligatorio: NotRequired[bool]
    Metodo_Pago: NotRequired[Literal['Efectivo-Cheque','PSE','Transferencia']]
    Comentario_Ejecutivo: NotRequired[str]
    Fue_Llamada: NotRequired[bool]
    Es_Reasignado: NotRequired[bool]
    Id_Acuerdo_Pago: NotRequired[str]
    Origen_Solicitud: NotRequired[str]
    Id_Respuesta_Autom: NotRequired[str]
    Fecha_Solicitado: NotRequired[str]
    Max_Descuento_Otorgado: NotRequired[bool]
    Es_Directo_Base: NotRequired[bool]
    Fecha_Limite_Respuesta: NotRequired[datetime]
    Addendums: NotRequired[List[DeudasSolicitud]]
    Modificaciones: NotRequired[List[ModificacionSolicitud]]

class SolicitudesSchema(pa.DataFrameModel):
    """
    Esquema para validar la estructura de los datos de solicitudes.
    """
    ID_Solicitud: Series[str] = pa.Field(unique=True)  # Aseguramos que ID_Solicitud sea único
    Timestamp: Series[pa.dtypes.Timestamp]
    Correo: Series[str] = pa.Field(str_matches=r"^[\w\.-]+@[\w\.-]+\.\w+$")  # Validación de correo electrónico
    Referencia: Series[str]
    Cedula: Series[str] = pa.Field(str_matches=r"^[\d\.]{6,15}$")  # Validación de cédula
    Ids_Deuda: Series[str]  # Lista de Ids de Deuda como cadena separada por -
    Casa_Cobro: Series[str]
    Tipo_Solicitud: Series[str] = pa.Field(isin=['Validación','Acuerdo de Pago','Oferta de Acuerdo'])
    Datos_Solicitud: Series[list[DeudasSolicitud]]
    Fecha_Esperada_Pago: Series[pa.dtypes.Timestamp] = pa.Field(nullable=True)  # Puede ser nulo si no hay fecha esperada de pago
    Tipo_Pago: Series[str] = pa.Field(nullable=True)  # Puede ser nulo si no hay tipo de pago
    Ejecutivo: Series[str] = pa.Field(nullable=True)  # Puede ser nulo si no hay ejecutivo asignado
    Metadata_Solicitud: Series[MetadataSolicitud]
    Estado_Solicitud: Series[str] = pa.Field(nullable=True)  # Puede ser nulo si no hay estado definido
    Fecha_Respuesta: Series[pa.dtypes.Timestamp] = pa.Field(nullable=True)  # Puede ser nulo si no hay respuesta
    Fecha_Limite_Pago: Series[pa.dtypes.Timestamp] = pa.Field(nullable=True)  # Puede ser nulo si no hay fecha límite de pago
    JSON_Respuesta: Series[list[DeudasSolicitud]] = pa.Field(nullable=True)
    Es_Historico: Series[bool]

# Función Auxiliar para Normalizar una Deuda al tipo DeudasSolicitud
def normalizeDeuda(deuda: dict) -> dict:
    deuda = dict(deuda)
    numero_credito = deuda.get('Numero_Credito')
    if numero_credito is not None:
        if isinstance(numero_credito, float) and numero_credito.is_integer():
            numero_credito = int(numero_credito)
        deuda['Numero_Credito'] = str(numero_credito).strip()
    return deuda

# Función Auxiliar para Normalizar una Modificación al tipo ModificacionSolicitud
def normalizeModificacion(modificacion: dict) -> dict:
    modificacion = dict(modificacion)
    for col in ['Fecha_Respuesta', 'Fecha_Limite_Pago']:
        if col not in modificacion:
            continue
        fecha = pd.to_datetime(modificacion.get(col), errors='coerce')
        if pd.notna(fecha):
            modificacion[col] = fecha
        else:
            modificacion.pop(col, None)
    json_respuesta = modificacion.get('JSON_Respuesta')
    if isinstance(json_respuesta, list):
        modificacion['JSON_Respuesta'] = [normalizeDeuda(dd) for dd in json_respuesta if isinstance(dd, dict)]
    metadata = modificacion.get('Metadata')
    if isinstance(metadata, dict):
        metadata = dict(metadata)
        comentario = metadata.get('Comentario_Ejecutivo')
        if comentario is not None and not isinstance(comentario, str):
            metadata['Comentario_Ejecutivo'] = str(comentario)
        if metadata.get('Addendums'):
            metadata['Addendums'] = [normalizeDeuda(dd) for dd in metadata['Addendums']]
        modificacion['Metadata'] = metadata
    return modificacion

# Función Auxiliar para Normalizar la Metadata al tipo MetadataSolicitud
def normalizeMetadata(metadata: dict) -> dict:
    metadata = dict(metadata)
    comentario = metadata.get('Comentario_Negociador')
    if comentario is not None and not isinstance(comentario, str):
        metadata['Comentario_Negociador'] = str(comentario)
    if metadata.get('Metodo_Pago') == '':
        metadata.pop('Metodo_Pago', None)
    if 'Addendums' in metadata and metadata['Addendums']:
        metadata['Addendums'] = [normalizeDeuda(dd) for dd in metadata['Addendums']]
    if metadata.get('Modificaciones'):
        metadata['Modificaciones'] = [normalizeModificacion(md) for md in metadata['Modificaciones']]
    fecha_limite_respuesta = metadata.get('Fecha_Limite_Respuesta')
    if fecha_limite_respuesta is not None and not (isinstance(fecha_limite_respuesta, str) and fecha_limite_respuesta.strip() == ''):
        fecha_limite_respuesta = pd.to_datetime(fecha_limite_respuesta, errors='coerce')
        if pd.notna(fecha_limite_respuesta):
            metadata['Fecha_Limite_Respuesta'] = fecha_limite_respuesta.normalize()
        else:
            metadata.pop('Fecha_Limite_Respuesta', None)
    return metadata

# Función Auxiliar para Limpiar Solicitudes
def clean_solicitudes(solicitudes_df: pd.DataFrame, es_historico: bool) -> DataFrame[SolicitudesSchema]:
    # Volvemos las Columnas Necesarias a Timestamp
    for col in ['Timestamp', 'Fecha_Esperada_Pago', 'Fecha_Respuesta', 'Fecha_Limite_Pago']:
        solicitudes_df[col] = pd.to_datetime(solicitudes_df[col], errors='coerce', dayfirst=False)

    # Volvemos las Columnas Referencia, ID_Solicitud y Cedula a String
    for col in ['Referencia', 'ID_Solicitud', 'Cedula', 'Ids_Deuda']:
        solicitudes_df[col] = solicitudes_df[col].apply(lambda s: str(s).replace('.0','').strip() if pd.notna(s) else '')

    # Hacemos Parsing de la Columna Datos_Solicitud y Metadata_Solicitud a JSON
    for col in ['Datos_Solicitud', 'Metadata_Solicitud','JSON_Respuesta']:
        solicitudes_df[col] = solicitudes_df[col].apply(lambda s: json.loads(s) if pd.notna(s) else {})

    # Las Columnas Vácias las Volvemos NaNs
    for col in ['Tipo_Pago']:
        solicitudes_df[col] = solicitudes_df[col].mask(solicitudes_df[col] == '', np.nan).str.strip()

    # Imputamos Ejecutivo con 'Sin Asignar'
    imputeNans(solicitudes_df, 'Ejecutivo', 'Sin Asignar')

    # Agregamos la Columna Es_Historico como False
    solicitudes_df['Es_Historico'] = es_historico

    # Volvemos cada JSON a su respectivo Esquema
    solicitudes_df['Datos_Solicitud'] = solicitudes_df['Datos_Solicitud'].apply(
        lambda d: [DeudasSolicitud(**normalizeDeuda(dd)) for dd in d] if d else []
    )
    solicitudes_df['JSON_Respuesta'] = solicitudes_df['JSON_Respuesta'].apply(
        lambda d: [DeudasSolicitud(**normalizeDeuda(dd)) for dd in d] if d else []
    )
    solicitudes_df['Metadata_Solicitud'] = solicitudes_df['Metadata_Solicitud'].apply(
        lambda d: MetadataSolicitud(**normalizeMetadata(d))
    )

    # Quitamos Cualquier Columna Unnamed
    solicitudes_df = solicitudes_df.loc[:, ~solicitudes_df.columns.str.contains('^Unnamed', case=False)]

    # Validamos el DataFrame con el esquema (Si no esta vacio)
    if not solicitudes_df.empty:
        solicitudes_df = SolicitudesSchema.validate(solicitudes_df, lazy=True)
    else:
        solicitudes_df = SolicitudesSchema.empty()

    return solicitudes_df
import numpy as np
import pandas as pd
from collections.abc import Mapping
from typing import Any


def extraer_historial_estados_deuda(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Construye el historial de estados de cada deuda a partir de las solicitudes.

    Columnas de salida:
        - Id_Deuda
        - Fecha_Solicitud
        - Estado_Solicitud

    Reglas:
        1. Toda solicitud genera un registro "Sin Tocar", con Timestamp
           y los Id_Deuda de Datos_Solicitud.
        2. Si existe Metadata_Solicitud.Fecha_Solicitado, genera un
           registro "Solicitado" para los Id_Deuda de Datos_Solicitud.
        3. El estado actual utiliza Fecha_Respuesta si existe; de lo
           contrario, utiliza Timestamp.
        4. Las modificaciones utilizan su propia Fecha_Respuesta y
           agregan " - Cambiado" al Estado_Solicitud.
        5. Para "Exitosa", utiliza JSON_Respuesta con Monto_Propuesto > 0.
        6. Para "Bajo Comité", utiliza JSON_Respuesta con Monto_Propuesto
           > 0 si JSON_Respuesta no está vacío. Si está vacío, utiliza
           Datos_Solicitud.
        7. Finalmente, genera una fila por cada Id_Deuda de cada evento.
    """

    columnas_salida = [
        "Id_Deuda",
        "Fecha_Solicitud",
        "Estado_Solicitud",
        "Tipo_Sol_Alianzas",
    ]

    registros = []

    # ---------------------------------------------------------
    # Funciones auxiliares
    # ---------------------------------------------------------

    def esta_vacio(valor: Any) -> bool:
        """Identifica None, NaN, NaT, pd.NA y contenedores vacíos."""

        if valor is None:
            return True

        if isinstance(valor, str):
            return not valor.strip()

        if isinstance(valor, (list, tuple, dict)):
            return len(valor) == 0

        try:
            resultado = pd.isna(valor)

            if isinstance(resultado, (bool, np.bool_)):
                return bool(resultado)

        except (TypeError, ValueError):
            pass

        return False

    def obtener_lista(valor: Any) -> list:
        """Devuelve una lista válida o una lista vacía."""

        return valor if isinstance(valor, list) else []

    def convertir_fecha(valor: Any):
        """Convierte una fecha a Timestamp y conserva NaT si no es válida."""

        if esta_vacio(valor):
            return pd.NaT

        return pd.to_datetime(valor, errors="coerce")

    def monto_positivo(deuda: Mapping) -> bool:
        """Indica si Monto_Propuesto es estrictamente mayor que cero."""

        monto = deuda.get("Monto_Propuesto")

        if esta_vacio(monto):
            return False

        try:
            return float(monto) > 0
        except (TypeError, ValueError):
            return False

    def obtener_ids_de_deudas(
        deudas: Any,
        solo_monto_positivo: bool = False,
    ) -> list[str]:
        """
        Extrae Id_Deuda de una lista de diccionarios.

        Evita repetir un mismo Id_Deuda dentro del mismo evento.
        """

        ids = []

        for deuda in obtener_lista(deudas):
            if not isinstance(deuda, Mapping):
                continue

            id_deuda = deuda.get("Id_Deuda")

            if esta_vacio(id_deuda):
                continue

            if solo_monto_positivo and not monto_positivo(deuda):
                continue

            ids.append(id_deuda)

        # Conserva el orden y elimina duplicados.
        return list(dict.fromkeys(ids))

    def obtener_ids_para_estado(
        estado: str,
        datos_solicitud: Any,
        json_respuesta: Any,
    ) -> list[str]:
        """Selecciona los Id_Deuda aplicables al estado correspondiente."""

        estado_base = estado.strip()

        # Permite evaluar correctamente los estados de las modificaciones.
        sufijo = " - Cambiado"

        if estado_base.endswith(sufijo):
            estado_base = estado_base[:-len(sufijo)].strip()

        respuesta = obtener_lista(json_respuesta)

        # Estado Exitosa: siempre se consideran las deudas respondidas
        # cuyo Monto_Propuesto sea estrictamente mayor que cero.
        if estado_base == "Exitosa":
            return obtener_ids_de_deudas(
                respuesta,
                solo_monto_positivo=True,
            )

        # Estado Bajo Comité:
        # - Respuesta no vacía: deudas con Monto_Propuesto > 0.
        # - Respuesta vacía: deudas de Datos_Solicitud.
        if estado_base == "Bajo Comité" and respuesta:
            return obtener_ids_de_deudas(
                respuesta,
                solo_monto_positivo=True,
            )

        # Demás estados, y Bajo Comité sin respuesta:
        # se utilizan las deudas originales de la solicitud.
        return obtener_ids_de_deudas(datos_solicitud)

    def agregar_evento(
        ids_deuda: list[str],
        fecha: Any,
        estado: str,
    ) -> None:
        """Agrega una fila al historial por cada Id_Deuda del evento."""

        for id_deuda in ids_deuda:
            registros.append({
                "Id_Deuda": id_deuda,
                "Fecha_Solicitud": fecha,
                "Estado_Solicitud": estado,
            })

    # ---------------------------------------------------------
    # Recorrido de solicitudes
    # ---------------------------------------------------------

    for _, solicitud in df.iterrows():

        timestamp = convertir_fecha(solicitud.get("Timestamp"))
        datos_solicitud = solicitud.get("Datos_Solicitud")
        metadata = solicitud.get("Metadata_Solicitud")

        if not isinstance(metadata, Mapping):
            metadata = {}

        ids_originales = obtener_ids_de_deudas(datos_solicitud)

        # 1. Estado inicial obligatorio: Sin Tocar.
        agregar_evento(
            ids_deuda=ids_originales,
            fecha=timestamp,
            estado="Sin Tocar",
        )

        # 2. Registro adicional: Solicitado.
        # Fecha_Solicitado está definido como str en el esquema,
        # por lo que se convierte a una fecha para la salida.
        fecha_solicitado = convertir_fecha(
            metadata.get("Fecha_Solicitado")
        )

        if not pd.isna(fecha_solicitado):
            agregar_evento(
                ids_deuda=ids_originales,
                fecha=fecha_solicitado,
                estado="Solicitado",
            )

        # 3. Estado actual de la solicitud.
        estado_actual = solicitud.get("Estado_Solicitud")

        fecha_respuesta = convertir_fecha(
            solicitud.get("Fecha_Respuesta")
        )

        fecha_estado_actual = (
            fecha_respuesta
            if not pd.isna(fecha_respuesta)
            else timestamp
        )

        if not esta_vacio(estado_actual):

            estado_actual = str(estado_actual).strip()

            # Evita duplicar el registro inicial si el estado actual
            # también es "Sin Tocar" y tiene la misma fecha.
            es_sin_tocar_inicial = (
                estado_actual == "Sin Tocar"
                and (
                    fecha_estado_actual == timestamp
                    or (
                        pd.isna(fecha_estado_actual)
                        and pd.isna(timestamp)
                    )
                )
            )

            if not es_sin_tocar_inicial:
                ids_estado_actual = obtener_ids_para_estado(
                    estado=estado_actual,
                    datos_solicitud=datos_solicitud,
                    json_respuesta=solicitud.get("JSON_Respuesta"),
                )

                agregar_evento(
                    ids_deuda=ids_estado_actual,
                    fecha=fecha_estado_actual,
                    estado=estado_actual,
                )

        # 4. Historial de modificaciones.
        modificaciones = obtener_lista(
            metadata.get("Modificaciones")
        )

        for modificacion in modificaciones:

            if not isinstance(modificacion, Mapping):
                continue

            estado_modificacion = modificacion.get("Estado_Solicitud")

            if esta_vacio(estado_modificacion):
                continue

            estado_modificacion = str(
                estado_modificacion
            ).strip()

            # Añade el sufijo sin duplicarlo si ya está presente.
            sufijo = " - Cambiado"

            if not estado_modificacion.endswith(sufijo):
                estado_modificacion += sufijo

            # Cada modificación utiliza su propia fecha de respuesta.
            fecha_modificacion = convertir_fecha(
                modificacion.get("Fecha_Respuesta")
            )

            # Respaldo para datos incompletos.
            if pd.isna(fecha_modificacion):
                fecha_modificacion = timestamp

            ids_modificacion = obtener_ids_para_estado(
                estado=estado_modificacion,
                datos_solicitud=datos_solicitud,
                json_respuesta=modificacion.get("JSON_Respuesta"),
            )

            agregar_evento(
                ids_deuda=ids_modificacion,
                fecha=fecha_modificacion,
                estado=estado_modificacion,
            )

    # ---------------------------------------------------------
    # Construcción y ordenamiento del DataFrame final
    # ---------------------------------------------------------

    resultado = pd.DataFrame(
        registros,
        columns=columnas_salida,
    )

    if resultado.empty:
        resultado["Fecha_Solicitud"] = pd.to_datetime(
            resultado["Fecha_Solicitud"]
        )
        return resultado

    resultado["Fecha_Solicitud"] = pd.to_datetime(
        resultado["Fecha_Solicitud"],
        errors="coerce",
    )

    # Orden cronológico; conserva el orden original cuando coinciden
    # las fechas.
    resultado = (
        resultado
        .sort_values("Fecha_Solicitud", kind="stable", na_position="last")
        .reset_index(drop=True)
    )

    return resultado

#### **Lectura y Limpieza de la Información**
___
Se limpian los datos y se dejan en el formato correcto.

In [ ]:
# Abrimos las Hojas "Solicitudes_MEC" y "Solicitudes_Hist"
solsMECWS = _retry(lambda: solsSH.worksheet('Solicitudes_MEC'))
solsHISTWS = _retry(lambda: solsSH.worksheet('Solicitudes_Hist'))

# Cargamos los Datos como un DF
solsMECDF = _retry(lambda: get_as_dataframe(solsMECWS, evaluate_formulas=True))
solsHISTDF = _retry(lambda: get_as_dataframe(solsHISTWS, evaluate_formulas=True))

# Unimos ambos DFs en uno solo
solsDF = pd.concat([solsMECDF, solsHISTDF], ignore_index=True)

# Limpiamos los Datos
solsDF = clean_solicitudes(solsDF, es_historico=False)

# Extraemos la Información usando la función extraer_historial_estados_deuda
solsDF = extraer_historial_estados_deuda(solsDF)

# Renombramos Estado_Solicitud a Estado_Sol_Alianzas y Fecha_Solicitud a Fecha_Sol
solsDF = solsDF.rename(columns={'Estado_Solicitud':'Estado_Sol_Alianzas',"Fecha_Solicitud":"Fecha_Sol","Tipo_Solicitud":"Tipo_Sol_Alianzas"})

# Creamos la Columna Fecha_Cartera usando el buscadorFC
solsDF['Fecha_Cartera'] = solsDF['Fecha_Sol'].apply(buscadorFC)
# Quitamos Datos con NaNs
solsDF = solsDF.dropna(subset=['Fecha_Cartera'])

# Por Último, separamos BMDF de DF
# BMDF será el que tenga Fecha_Cartera == FECHA_MEJOR_MES
newsolBMDF = solsDF[solsDF['Fecha_Cartera'] == FECHA_MEJOR_MES].copy()
# El DF solo será Fecha_Cartera == fecha_cartera_ultima
newsolDF = solsDF[solsDF['Fecha_Cartera'] == fecha_cartera_ultima].copy()

# Les quitamos las Columnas Fecha_Cartera a ambos DF
newsolBMDF = newsolBMDF.drop(columns=['Fecha_Cartera'])
newsolDF = newsolDF.drop(columns=['Fecha_Cartera'])

# Imprimimos para Logs
print('✅Datos de Solicitudes Antiguas Cargados con Éxito')
print('ℹ️Filas de Actual: {}'.format(
    len(newsolDF)
))
print('🆔Filas Mejor Mes: {}'.format(
  len(newsolBMDF)
))

✅Datos de Solicitudes Antiguas Cargados con Éxito
ℹ️Filas de Actual: 3240
🆔Filas Mejor Mes: 1


### **Consolidación de ambos formularios en uno solo**
___
Se crean los DFs finales que van a servir para la unión

In [ ]:
# Concatenamos oldsolBMDF y newsolBMDF en un nuevo DF llamado statusalyBMDF
statusalyBMDF = pd.concat([oldsolBMDF, newsolBMDF], ignore_index=True)
# Realizamos lo mismo con oldsolDF y newsolDF para statusalyDF
statusalyDF = pd.concat([oldsolDF, newsolDF], ignore_index=True)

# Imprimimos para logs
print('😁Datos de Estados de Alianzas Obtenidos con Éxito')
print('ℹ️Filas de Actual: {}'.format(
    len(statusalyDF)
))
print('🆔Filas Mejor Mes: {}'.format(
  len(statusalyBMDF)
))

😁Datos de Estados de Alianzas Obtenidos con Éxito
ℹ️Filas de Actual: 3240
🆔Filas Mejor Mes: 3088


### **Consolidación del DF originallyDF**
___
Con base en los datos cargados se procede a crear **originallyDF y originallyBMDF**

In [ ]:
# Para crear estos DFs hay que simplemente filtrar los statusaly donde:
# Estado == "Sin Tocar"
originallyDF = statusalyDF[statusalyDF['Estado_Sol_Alianzas'] == 'Sin Tocar'].copy()
originallyBMDF = statusalyBMDF[statusalyBMDF['Estado_Sol_Alianzas'] == 'Sin Tocar'].copy()

# Ahora Creamos la Columna Origen_Alianzas como "1:1"
originallyDF['Origen_Alianzas'] = '1:1'
originallyBMDF['Origen_Alianzas'] = '1:1'

# Por Último, renombramos Fecha_Sol a Fecha_Alianzas
originallyDF = originallyDF.rename(columns={'Fecha_Sol':'Fecha_Alianzas'})
originallyBMDF = originallyBMDF.rename(columns={'Fecha_Sol':'Fecha_Alianzas'})

# Quitamos la Columna Tipo_Sol_Alianzas
originallyDF = originallyDF.drop(columns=['Tipo_Sol_Alianzas'])
originallyBMDF = originallyBMDF.drop(columns=['Tipo_Sol_Alianzas'])

# Imprimimos para logs
print('😁Datos de Estados de Alianzas Obtenidos con Éxito')

😁Datos de Estados de Alianzas Obtenidos con Éxito


# **Pre-Limpieza antes de Unión**
___
Aquí se van a realizar las siguientes limpiezas:
- **Añadir Liquidaciones como Actualizaciones**: Se añaden Liquidaciones como actualizaciones de modo que al momento de realizar la Unión esta quedé como última. Además, se quitan las actualizaciones anteriores (si existen) para evitar problemas de duplicidad.
- **Limpiar DFs de BM**: Se crean los dfs auxiliares BM para clientesDF, liqsDF y actsDF.
- **Agregar Masivas a originallyDF**: Se agregan las actualizaciones masivas a este DF para conocer el Origen de Alianzas
- **Verificar Existencia de Duplicados**: Se verifica que los datos no tenga duplicados de modo que la información sea veraz.

## **Liquidaciones como Actualizaciones**
___
En este Apartado confirmamos que la última actualización sea de Liquidación, de modo que se pueda categorizar Correctamente. Para realizar los pasos son simples:
1. **Preparamos una Copia de las Liquidaciones para agregar a actsDF**
2. **Agregar a las Actualizaciones las Liqs Finales**
3. **Agregar el Líder de la Actualización**

In [ ]:
# Paso 1: Preparar copia de concatencación
liqsActs = liqsDF.copy()
# Renombramos Fecha_Liq a Fecha_Act y PaB_Liq a PaB_Act para que no hayan conflictos al concatenar con actsDF
liqsActs = liqsActs.rename(columns={'Fecha_Liq':'Fecha_Act',
                                    'PaB_Liq':'PaB_Act',
}) # type: ignore

# Ahora a Fecha_Act la Dejamos como 23:59:59 (Para que queden al Final)
liqsActs['Fecha_Act'] = liqsActs['Fecha_Act'].apply(lambda dt: dt.replace(hour=23, minute=59, second=59))

# Creamos la Columna Etiqueta_Act como 'Liquidado' cuando Portafolio_Liq == 'NO' else 'Liquidado en Portafolio'
liqsActs['Etiqueta_Act'] = 'Liquidado'

# Creamos la Columna Comentario_Act como 'Actualización por Liquidación'
liqsActs['Comentario_Act'] = np.where((liqsActs['Portafolio_Liq'] == 'NO'),'Una Sola Deuda','En Portafolio')

# Ahora creamos la Columna Origen_Act con función robusta de búsqueda de Originador de liquidación
def getOrigenLiq(row):
  # Primero se Busca Negociador_Liq en el HeadCount para obtener el Origen
  negociador = row['Negociador_Liq']
  dataHC = headCountDF[headCountDF['Negociador_HC'] == negociador]
  if not dataHC.empty:
    return dataHC['Email'].values[0]

  # Si no se va a Realizar una acción manual
  nombresNego = negociador.split(' ')
  if len(nombresNego) >= 4:
    return f'{nombresNego[0]}.{nombresNego[2]}'.lower()
  elif len(nombresNego) > 2:
    return f'{nombresNego[0]}.{nombresNego[1]}'.lower()
  else:
    return negociador.replace(' ','.').lower()
# Aplicamos la función para obtener el Origen de la Liquidación
liqsActs['Origen_Act'] = liqsActs.apply(getOrigenLiq, axis=1)

# Agregamos Etapa_Act como 4
liqsActs['Etapa_Act'] = 4

# Ahora dejamos solo las Columnas necesarias para concatenar con actsDF
liqsActs = liqsActs[['Id_Deuda','Fecha_Act','PaB_Act','Etiqueta_Act','Origen_Act','Comentario_Act','Fecha_Cartera','Etapa_Act']]

# Aplicamos la Concatenación de liqsActs con actsDF
actsDF = pd.concat([actsDF, liqsActs], ignore_index=True)

# Ahora realizamos una union left entre actsDF Origen_Act y headCountDF Email para obtener el Lider
actsDF = pd.merge(
    actsDF,
    headCountDF[['Email','Lider']],
    left_on='Origen_Act',
    right_on='Email',
    how='left'
)
# Imputamos Lider con "Lider N/A"
imputeNans(actsDF, 'Lider', 'Lider N/A')
# Quitamos la Columna Email
actsDF = actsDF.drop(columns=['Email'])
# Renombramos Lider a Lider_Origen
actsDF = actsDF.rename(columns={'Lider':'Lider_Origen'}) # type: ignore

# Imprimimos para logs
print('🫡Número de Actualizaciones de Liquidaciones: {}'.format(
  len(liqsActs)
))

🫡Número de Actualizaciones de Liquidaciones: 8234


## **Limpiar DFs de BM**
___
Se limpian los datos cargados de:
- **Cartera**
- **Liquidaciones**
- **Actualizaciones**

Creando nuevos DFs que terminan en **BMDF** representando que son del mejor mes. Para crearlos hay que simplemente:
- **BMDF**: Fecha_Cartera == FECHA_MEJOR_MES
- **DF**: Fecha_Cartera == fecha_cartera_ultima

In [ ]:
# Primero para clientesDF
clientesBMDF = clientesDF[clientesDF['Fecha_Cartera'] == FECHA_MEJOR_MES].copy()
clientesDF = clientesDF[clientesDF['Fecha_Cartera'] == fecha_cartera_ultima].copy()
# Ahora para liqsDF
liqsBMDF = liqsDF[liqsDF['Fecha_Cartera'] == FECHA_MEJOR_MES].copy()
liqsDF = liqsDF[liqsDF['Fecha_Cartera'] == fecha_cartera_ultima].copy()
# Por último para actsDF
actsBMDF = actsDF[actsDF['Fecha_Cartera'] == FECHA_MEJOR_MES].copy()
actsDF = actsDF[actsDF['Fecha_Cartera'] == fecha_cartera_ultima].copy()

# Ahora Quitamos de todos los DFs la columna Fecha_Cartera
clientesBMDF = clientesBMDF.drop(columns=['Fecha_Cartera'])
clientesDF = clientesDF.drop(columns=['Fecha_Cartera'])
liqsBMDF = liqsBMDF.drop(columns=['Fecha_Cartera'])
liqsDF = liqsDF.drop(columns=['Fecha_Cartera'])
actsBMDF = actsBMDF.drop(columns=['Fecha_Cartera'])
actsDF = actsDF.drop(columns=['Fecha_Cartera'])

# Imprimimos para logs
print('😁✅DFs transformados y repartidos con Éxito')

😁✅DFs transformados y repartidos con Éxito


## **Adición de Masivas a Origen de Alianzas**
___
Se agrega a **originallyDF y originallyBMDF** sus respectivas actualizaciones de Alianzas, extraídas desde **actsDF y actsBMDF**.

In [ ]:
# Para reconocer una actualización de alianzas hay 4 filtros que realizar:
# Etiqueta_Act == "ACTUALIZADO_ALIANZAS"
# Origen_Act == "massive"
# Comentario_Act not contains "Bot de ofertas"
# Comentario Act not contains 'Validación Recibida desde Alianzas'
# Comentario_Act not contains "Acuerdo en curso desde Alianzas"

# Creamos la Función extractora de resultados
def extractMassiveAlianzas(acDF: pd.DataFrame) -> pd.DataFrame:
  # Primero creamos todas las máscaras
  maskEtiqueta = acDF['Etiqueta_Act'] == 'ACTUALIZADO_ALIANZAS'
  maskOrigen = acDF['Origen_Act'] == 'massive'
  maskComentario1 = ~acDF['Comentario_Act'].str.contains('Bot de ofertas',na=True,regex=False)
  maskComentario2 = ~acDF['Comentario_Act'].str.contains('Validación Recibida desde Alianzas',na=True,regex=False)
  maskComentario3 = ~acDF['Comentario_Act'].str.contains("Acuerdo en curso desde Alianzas",na=True,regex=False)

  # Ahora Creamos una copia del DF con todas las máscaras unidas
  maskTotal = maskEtiqueta & maskOrigen & maskComentario1 & maskComentario2 & maskComentario3
  acDF = acDF[maskTotal].copy()

  # Ahora vamos a estilizar el DF
  # Primero Renombramos Fecha_Act a Fecha_Alianzas
  acDF = acDF.rename(columns={'Fecha_Act':'Fecha_Alianzas'})
  # Ahora creamos la Columna Origen_Alianzas como "Masivas"
  acDF['Origen_Alianzas'] = 'Masivas'

  # Dejamos solo Columnas: Id_Deuda, Fecha_Alianzas y Origen_Alianzas
  acDF = acDF[['Id_Deuda','Fecha_Alianzas','Origen_Alianzas']]

  # Devolvemos el DF
  return acDF

# Obtenemos los datos para BMDF
massiveBMDF = extractMassiveAlianzas(actsBMDF)
# Obtenemos los datos para DF
massiveDF = extractMassiveAlianzas(actsDF)

# Ahora unimos estos DFs con originally
originallyBMDF = pd.concat([originallyBMDF, massiveBMDF], ignore_index=True)
originallyDF = pd.concat([originallyDF, massiveDF], ignore_index=True)

# Quitamos Columna Estado_Sol_Alianzas
originallyBMDF = originallyBMDF.drop(columns=['Estado_Sol_Alianzas'])
originallyDF = originallyDF.drop(columns=['Estado_Sol_Alianzas'])

# Imprimimos para logs
print('✅Datos de Origen de Alianzas Obtenidos con Éxito')
print('ℹ️Filas Actuales: {}'.format(
    len(originallyDF)
))
print('🆔Filas Mejor Mes: {}'.format(
  len(originallyBMDF)
))

✅Datos de Origen de Alianzas Obtenidos con Éxito
ℹ️Filas Actuales: 6586
🆔Filas Mejor Mes: 7042


## **Verificación de Duplicados**
___
Dado que pueden existir Duplicados que dañen el seguimiento, se aplica una verificación inicial para todas las bases.

In [ ]:
def detectDuplicates(df: pd.DataFrame, idCol: str,baseName: str):
  maskDup = df.duplicated(subset=[idCol], keep='first')
  if maskDup.any():
    print('✖️Error en Duplicados en Base de <{}>'.format(baseName))
    print('🚯Ids Duplicados ({} duplicados):'.format(
        maskDup.sum()
    ))
    print_grid(df[maskDup][idCol].tolist(), n = 7, sep = ' | ')
    raise ValueError(f'Error Crítico: 🚯Duplicados en {baseName}')
  else:
    print('✅Base de <{}> sin Duplicados'.format(baseName))

# Aplicamos la Verificación para todas las Bases
print('ℹ️Verificación para MEC\n')
detectDuplicates(depositosDF, 'Referencia', 'Depositos')
detectDuplicates(liqsDF, 'Id_Deuda', 'Liquidaciones')
detectDuplicates(potCredDF, 'Referencia', 'Potenciales de Crédito')
detectDuplicates(pabIdealDF, 'Id_Deuda', 'PaB Ideal de Crédito')
detectDuplicates(clientesDF, 'Id_Deuda', 'Datos de Asignación de Cartera')
detectDuplicates(prospDF, 'Id_Deuda', 'Prospectos de Tradicional')

# Siguiente: Si FECHA_MEJOR_MES existe verificamos los duplicados de los DFs de esa fecha
if not (FECHA_MEJOR_MES is None):
  print('\n⚛️ Verificación para Mejor Mes\n')
  detectDuplicates(depositosBMDF, 'Referencia', 'Depositos BM')
  detectDuplicates(liqsBMDF, 'Id_Deuda', 'Liquidaciones')
  detectDuplicates(potCredBMDF, 'Referencia', 'Potenciales de Crédito BM')
  detectDuplicates(pabIdealBMDF, 'Id_Deuda', 'PaB Ideal de Crédito BM')
  detectDuplicates(clientesBMDF, 'Id_Deuda', 'Datos de Asignación de Cartera')
  detectDuplicates(prospBMDF, 'Id_Deuda', 'Prospectos de Tradicional BM')

ℹ️Verificación para MEC

✅Base de <Depositos> sin Duplicados
✅Base de <Liquidaciones> sin Duplicados
✅Base de <Potenciales de Crédito> sin Duplicados
✅Base de <PaB Ideal de Crédito> sin Duplicados
✅Base de <Datos de Asignación de Cartera> sin Duplicados
✅Base de <Prospectos de Tradicional> sin Duplicados

⚛️ Verificación para Mejor Mes

✅Base de <Depositos BM> sin Duplicados
✅Base de <Liquidaciones> sin Duplicados
✅Base de <Potenciales de Crédito BM> sin Duplicados
✅Base de <PaB Ideal de Crédito BM> sin Duplicados
✅Base de <Datos de Asignación de Cartera> sin Duplicados
✅Base de <Prospectos de Tradicional BM> sin Duplicados


# **Funciones de Generación de Bucket**
___
En este apartado, se generan todos los Cálculos que permiten que el Bucket Cartera tenga un seguimiento Genial. Los Cálculos que se aplican son:
- **Lógica de Última Actualización**
- **Unión de Datos**
- **Cálculos Generales**
- **Cálculos para Tradicional**
- **Reajuste del Bucket**

## **Lógica de Última Actualización**
___
Dado que la construcción del Timeline es diferente a Bucket Cartera, se requiere realizar una diferenciación según cual se este creando según LA ÚLTIMA ACTUALIZACIÓN, teniendo en cuenta que se manejan 2 parámetros:
- **mode**: BM si se obtienen del mejor mes, de lo contrario MEC para representar el Mes Actual
- **limitDate**: Fecha Limité a considerar los datos.

In [ ]:
def getActsInfo(mode: Literal['BM','MEC'], limitDate: pd.Timestamp = pd.Timestamp(1999,12,12)) -> pd.DataFrame:
  if limitDate.year == 1999: # Si no hay limite dejamos el mayor limite posible
    limitDate = pd.Timestamp.now() + pd.Timedelta(days=2)

  # Definimos newActs según el modo
  if mode == 'BM':
    newActs = actsBMDF.copy()
  else:
    newActs = actsDF.copy()

  # Ordenamos por Id_Deuda y Fecha_Act
  newActs = newActs.sort_values(by=['Id_Deuda', 'Fecha_Act']).copy()

  # Filtramos para que Fecha_Act sea < limitDate
  newActs = newActs[newActs['Fecha_Act'] < limitDate]

  # Nos quedamos con el último registro por Id_Deuda
  newActs = newActs.drop_duplicates(subset=['Id_Deuda'], keep='last').reset_index(drop=True)

  return newActs

## **Unión de Datos**
___
Una vez ya se encuentren los datos limpios, se procede a realizar la Unión con el Fin de consolidar toda la información en una sola tabla/DataFrame

In [ ]:
def uniteInfo(mode: Literal['BM','MEC'], actsInfo: pd.DataFrame, limitDate: pd.Timestamp = endDate, verbose: bool=True) -> pd.DataFrame:
  # De primeras, definimos DFs locales según el modo
  if mode == 'BM':
    clientesLCL = clientesBMDF.copy()
    depositosLCL = depositosBMDF.copy()
    liqsLCL = liqsBMDF.copy()
    pabIdealLCL = pabIdealBMDF.copy()
    potCredLCL = potCredBMDF.copy()
    prospLCL = prospBMDF.copy()
    pipecredLCL = pipecredBMDF.copy()
    statusalyLCL = statusalyBMDF.copy()
    originallyLCL = originallyBMDF.copy()
    actsLCL = actsBMDF.copy()
  else:
    clientesLCL = clientesDF.copy()
    depositosLCL = depositosDF.copy()
    liqsLCL = liqsDF.copy()
    pabIdealLCL = pabIdealDF.copy()
    potCredLCL = potCredDF.copy()
    prospLCL = prospDF.copy()
    pipecredLCL = pipecredDF.copy()
    statusalyLCL = statusalyDF.copy()
    originallyLCL = originallyDF.copy()
    actsLCL = actsDF.copy()

  # Renombramos para actsLCL Fecha_Act a Fecha_Act_Anterior
  actsLCL = actsLCL.rename(columns={'Fecha_Act':'Fecha_Act_Anterior'}) # type: ignore

  # Primera Union: clientesLCL con depositosLCL para agregar la información de Meses_Sin_Deposito
  baseDF = pd.merge(clientesLCL, depositosLCL, on='Referencia', how='left')
  # Imputamos los NaNs de Meses_Sin_Deposito con 12 (Cuando no hay información de depósito se asume que hace un año que no deposita)
  imputeNans(baseDF, 'Meses_Sin_Deposito', 12)

  # Segunda Union: baseDF con actsInfo para agregar la información de Actualizaciones
  baseDF = pd.merge(baseDF, actsInfo, on='Id_Deuda', how='left') # No es Necesario realizar Imputaciones

  # Tercera Union: baseDF con pabIdealLCL para agregar la información de PaB_Ideal_Credito
  baseDF = pd.merge(baseDF, pabIdealLCL, on='Id_Deuda', how='left') # No es Necesario realizar Imputaciones

  # Cuarta Union: baseDF con potCredLCL para agregar la información de Potencial_Credito y Batch_Credito
  baseDF = pd.merge(baseDF, potCredLCL, on='Referencia', how='left')
  # Imputamos Batch_Credito con 'Tradicional'
  imputeNans(baseDF, 'Batch_Credito', 'Tradicional')

  # Quinta Union: baseDF con prospLCL por Id_Deuda
  baseDF = pd.merge(baseDF, prospLCL, on='Id_Deuda', how='left') # No es Necesario realizar Imputaciones
  # Imputamos Prosp_Liq como "Sin Priorizar"
  imputeNans(baseDF, 'Prosp_Liq', 'Sin Priorizar')

  # Creamos unas Liquidaciones Totales filtradas segun el LimitDate
  liqsTotales = liqsLCL[liqsLCL['Fecha_Liq'] < limitDate].copy()

  # Quinta Union: baseDF con liqsLCL para agregar la información de las Liquidaciones
  baseDF = pd.merge(baseDF, liqsTotales, on='Id_Deuda', how='left') # No es Necesario realizar Imputaciones

  # TEMPORALMENTE: Imputamos Fecha_Act con la Fecha máxima de Fecha_Act en el DF
  maskNotAct = baseDF['Fecha_Act'].isna()
  baseDF['Estaba_Sin_Act'] = maskNotAct
  baseDF.loc[maskNotAct, 'Fecha_Act'] = baseDF['Fecha_Act'].max()

  # Sexta Union: baseDF con Bajas usando merge_asof entre Fecha_Act y Fecha_Baja con dirección backwards  permitiendo match exacto
  baseDF = pd.merge_asof(
      baseDF.sort_values('Fecha_Act'),
      bajasDF.sort_values('Fecha_Baja'),
      left_on='Fecha_Act',
      right_on='Fecha_Baja',
      by='Referencia',
      direction='backward'
  )

  # Septíma Unión: baseDF con pipecredLCL usando merge_asof entre Fecha_Act y Fecha_Pipeline con dirección back permitiendo match exacto
  baseDF = pd.merge_asof(
      baseDF.sort_values('Fecha_Act'),
      pipecredLCL.sort_values('Fecha_Pipeline'),
      left_on='Fecha_Act',
      right_on='Fecha_Pipeline',
      by='Referencia',
      direction='backward'
  )
  # Quitamos la Columna Fecha_Pipeline
  baseDF = baseDF.drop(columns=['Fecha_Pipeline'])

  # Octava Unión: baseDF con statusalyLCLusando merge_asof entre Fecha_Act y Fecha_Sol con dirección back permitiendo match exacto por Id_Deuda
  baseDF = pd.merge_asof(
      baseDF.sort_values('Fecha_Act'),
      statusalyLCL.sort_values('Fecha_Sol'),
      left_on='Fecha_Act',
      right_on='Fecha_Sol',
      by='Id_Deuda',
      direction='backward'
  )
  # Quitamos la Columna Fecha_Sol
  baseDF = baseDF.drop(columns=['Fecha_Sol'])

  # Novena Unión: baseDF con originallyLCL usando merge_asof entre Fecha_Act y Fecha_Alianzas con dirección back permitiendo match exacto por Id_Deuda
  baseDF = pd.merge_asof(
      baseDF.sort_values('Fecha_Act'),
      originallyLCL.sort_values('Fecha_Alianzas'),
      left_on='Fecha_Act',
      right_on='Fecha_Alianzas',
      by='Id_Deuda',
      direction='backward',
  )
  # Quitamos la Columna Fecha_Alianzas
  baseDF = baseDF.drop(columns=['Fecha_Alianzas'])
  # Imputamos Origen_Alianzas con 'N/A'
  imputeNans(baseDF, 'Origen_Alianzas', 'N/A')

  # Última Unión: baseDF con actsLCL usando merge_asof con Fecha_Act con dirección back sin permitir match exacto por Id_Deuda
  baseDF = pd.merge_asof(
      baseDF.sort_values('Fecha_Act'),
      actsLCL.sort_values('Fecha_Act_Anterior'),
      left_on="Fecha_Act",
      right_on="Fecha_Act_Anterior",
      by='Id_Deuda',
      direction='backward',
      suffixes=('', '_Anterior'),
      allow_exact_matches=False,
  )

  # REAJUSTE: Volvemos de nuevo Fecha_Act a NaN para la máscara imputada
  baseDF.loc[baseDF['Estaba_Sin_Act'], 'Fecha_Act'] = pd.NaT
  # Quitamos la Columna Estaba_Sin_Act
  baseDF = baseDF.drop(columns=['Estaba_Sin_Act'])

  # Imprimimos para Logs
  if verbose:
    print('✅Base Final sin Cálculos creada con Éxito')
    print('🚼Filas totales: {} filas, 🆔Columnas Totales: {}, 🫴Las Columnas son:'.format(len(baseDF), len(baseDF.columns)))
    print_grid(baseDF.columns.tolist(), n = 7, sep = ' | ')

  # Devolvemos el DF
  return baseDF

## **Cálculos Generales**
___
Se van a realizar los cálculos para crear las siguientes Columnas:
- **Es_Baja**: 1 para sí, 0 para no
- **Limpieza de Potenciales de Crédito**: Los clientes con Mora en Crédito se convierten en No Viables. Además, los bancos demasiado dificiles se les agrega un **\***
- **Comision_Exito**: Columna Auxiliar para Cálculos de Potencial
- **Tiene_Act**: 1 o 0 dependiendo si tiene actualización
- **Descuento_Act**: El Descuento brindado por el PaB de la actualización
- **Efectividad**: El Tipo de Efectividad de la Actualización entre: NO EFECTIVA, EFECTIVA, EFECTIVA ALIANZAS, EFECTIVA DESCUENTO
- **Plazos_PaB**: Los Plazos a Considerar de PaB para la Estructuración
- **Fase**: La Fase en la cúal se encuentra la Deuda
  - **Fase 1**: No hay Actualización MEC
  - **Fase 2**: Hay Actualización MEC
  - **Fase 3**: Hay Actualización con Etiqueta en Pro de Liquidación
  - **Fase 4**: Deuda Liquidada

In [ ]:
def makeGeneralCalcs(baseDF: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
  # Creamos una Copia del DF
  baseDF = baseDF.copy()
  # Creamos la Columna Es_Baja como 1 o 0 dependiendo de si Fecha_Baja es NaN o no
  baseDF['Es_Baja'] = np.where(baseDF['Fecha_Baja'].notna(), 1, 0)

  # Limpiamos Potenciales de Crédito para los Datos con Mora_Credito > 0 porque no son Potenciales Reales
  maskMoraCredito = baseDF['Mora_Credito'] > 0
  baseDF.loc[maskMoraCredito, 'Potencial_Credito'] = np.nan

  # Creamos la Columna de Descuento_Act como (1 - (PaB_Act / Dum_Bravo)) para las filas donde PaB_Act exista
  baseDF['Descuento_Act'] = np.nan
  maskHayAct = (baseDF['PaB_Act'].notna() & (baseDF['PaB_Act']>0))
  baseDF.loc[maskHayAct, 'Descuento_Act'] = 1 - (baseDF.loc[maskHayAct, 'PaB_Act'] / baseDF.loc[maskHayAct, 'Dum_Bravo'])
  # Aplicamos la Misma Lógica pero con Datos con sufijo _Anterior
  baseDF['Descuento_Act_Anterior'] = np.nan
  maskHayActAnterior = (baseDF['PaB_Act_Anterior'].notna() & (baseDF['PaB_Act_Anterior']>0))
  baseDF.loc[maskHayActAnterior, 'Descuento_Act_Anterior'] = 1 - (baseDF.loc[maskHayActAnterior, 'PaB_Act_Anterior'] / baseDF.loc[maskHayActAnterior, 'Dum_Bravo'])

  # Calculamos Descuento_Mejor_PL como Descuento_Act >= Descuento_PL
  baseDF['Descuento_Mejor_PL'] = np.where(baseDF['Descuento_Act'] >= baseDF['Descuento_PL'], 1, 0)
  # Calculamos Descuento_Mejor_PL_Anterior
  baseDF['Descuento_Mejor_PL_Anterior'] = np.where(baseDF['Descuento_Act_Anterior'] >= baseDF['Descuento_PL'], 1, 0)

  # Calculamos Dias_Anterior_Act como la Diferencia en Días Hábiles entre Fecha_Act y Fecha_Act_Anterior
  # Para esto usamos la función getBDDaysDiffFloat_vectorized
  baseDF['Dias_Anterior_Act'] = getBDDaysDiffFloat_vectorized(baseDF['Fecha_Act_Anterior'], baseDF['Fecha_Act'])

  # Creamos una Columna de PaB_Real como Dum_Bravo cuando no existe PaB_Act y como PaB_Act cuando esta existe
  baseDF['PaB_Real'] = np.where(baseDF['PaB_Act'].notna() & (baseDF['PaB_Act']>0), baseDF['PaB_Act'], baseDF['Dum_Bravo'])
  # Creamos también una Columna de Descuento_Real que es igual a Descuento_Act cuando existe y a 0 cuando no existe Descuento_Act
  baseDF['Descuento_Real'] = np.where(baseDF['Descuento_Act'].notna(), baseDF['Descuento_Act'], 0)

  # Creamos Columna Tiene_Act que es 1 cuando existe una Actualización y 0 cuando no existe una actualización
  baseDF['Tiene_Act'] = np.where(baseDF['Fecha_Act'].notna(), 1, 0)
  # Creamos Columna Tiene_Act_Anterior
  baseDF['Tiene_Act_Anterior'] = np.where(baseDF['Fecha_Act_Anterior'].notna(), 1, 0)

  # Creamos la Columna de Comision_Exito como: PaB_Real * Descuento_Real * Pricing * 1.19 (añadiendo IVA)
  baseDF['Comision_Exito'] = baseDF['PaB_Real'] * baseDF['Descuento_Real'] * baseDF['Pricing'] * 1.19

  # --- Creación de Efectividad ---
  # Usamos la mascára de hay actualizaciones para realizar la siguiente comparación:
  # Definicion de Rango de Descuento: Descuento_Act >= DESC_CONSIDERAR_EFECTIVA y Descuento_Act <= 0.98 (Se pone 0.98 porque no se consideran descuentos del 100% como efectivos)
  # Caso A: Etiqueta_Act == 'ACTUALIZADO_ALIANZAS' y cumple rango de descuento: EFECTIVA DESCUENTO ALIANZAS
  # Caso B: Etiqueta_Act == 'ACTUALIZADO_ALIANZAS' y Descuento_Act no esta en el rango: EFECTIVA ALIANZAS
  # Caso C: Etiqueta_Act in {"ACUERDO", "DESCUENTO", "EN_CREDITO", "Liquidado", "SALDOS", "DEPOSITO"} y cumple rango de descuento: EFECTIVA
  # Caso D: El resto se considera no efectiva
  # 1. Define your conditions (order matters: top to bottom)
  conditions = [
      # Condition 1: ALIANZAS with discount in range
      (baseDF['Tiene_Act'] == 1) &
      (baseDF['Etiqueta_Act'] == 'ACTUALIZADO_ALIANZAS') &
      (baseDF['Descuento_Act'] >= DESC_CONSIDERAR_EFECTIVA) &
      (baseDF['Descuento_Act'] <= 0.98),

      # Condition 2: ALIANZAS without that specific discount range
      (baseDF['Tiene_Act'] == 1) &
      (baseDF['Etiqueta_Act'] == 'ACTUALIZADO_ALIANZAS'),

      # Condition 3: Other specific categories with discount in range
      (baseDF['Tiene_Act'] == 1) &
      (baseDF['Etiqueta_Act'].isin(["ACUERDO", "DESCUENTO","CAMBIO DE PAB","REVISAR","EN_CREDITO","VERIFICACION DE SALDOS",
                                    "Liquidado","Liquidado en Portafolio", "SALDOS", "DEPOSITO"])) &
      (baseDF['Descuento_Act'] >= DESC_CONSIDERAR_EFECTIVA) &
      (baseDF['Descuento_Act'] <= 0.98),

      # Condition 4: Has activity but didn't meet any of the "EFECTIVA" criteria
      (baseDF['Tiene_Act'] == 1)
  ]

  # Ahora Creamos las Mismas condiciones pero según la actividad anterior
  condition_anterior = [
      # Condition 1: ALIANZAS with discount in range
      (baseDF['Tiene_Act_Anterior'] == 1) &
      (baseDF['Etiqueta_Act_Anterior'] == 'ACTUALIZADO_ALIANZAS') &
      (baseDF['Descuento_Act_Anterior'] >= DESC_CONSIDERAR_EFECTIVA) &
      (baseDF['Descuento_Act_Anterior'] <= 0.98),

      # Condition 2: ALIANZAS without that specific discount range
      (baseDF['Tiene_Act_Anterior'] == 1) &
      (baseDF['Etiqueta_Act_Anterior'] == 'ACTUALIZADO_ALIANZAS'),

      # Condition 3: Other specific categories with discount in range
      (baseDF['Tiene_Act_Anterior'] == 1) &
      (baseDF['Etiqueta_Act_Anterior'].isin(["ACUERDO", "DESCUENTO","CAMBIO DE PAB","REVISAR","EN_CREDITO","VERIFICACION DE SALDOS",
                                    "Liquidado","Liquidado en Portafolio", "SALDOS", "DEPOSITO"])) &
      (baseDF['Descuento_Act_Anterior'] >= DESC_CONSIDERAR_EFECTIVA) &
      (baseDF['Descuento_Act_Anterior'] <= 0.98),

      # Condition 4: Has activity but didn't meet any of the "EFECTIVA" criteria
      (baseDF['Tiene_Act_Anterior'] == 1)
  ]

  # 2. Define the corresponding outputs for each condition
  choices = [
      'EFECTIVA DESCUENTO ALIANZAS',
      'EFECTIVA ALIANZAS',
      'EFECTIVA',
      'NO EFECTIVA'
  ]

  # 3. Apply np.select. The 'default' argument handles when 'Tiene_Act' != 1
  baseDF['Efectividad'] = np.select(conditions, choices, default='')
  # Creamos Efectividad_Anterior
  baseDF['Efectividad_Anterior'] = np.select(condition_anterior, choices, default='')

  # --- Creación de Fase ---
  # Definición de Fases:
  # Fase 1: No Tiene Actualizacion
  # Fase 2: Tiene alguna actualizaciónç
  # Fase 3: Tiene Actualizacion y Etiqueta_Act in CATEGORIAS_FASE_3
  # Fase 4: Etiqueta_Act == 'Liquidado'

  # 1. Definir Condiciones en sentido inverso (de la más específica a la más general)
  conditions = [
      # Fase 4: Liquidado
      (baseDF['Tipo_Liq'].notna()),

      # Fase 3: Tiene Actualización y Etiqueta_Act in CATEGORIAS_FASE_3
      (baseDF['Tiene_Act'] == 1) &
      (baseDF['Etiqueta_Act'].isin(CATEGORIAS_FASE_3)) & (baseDF['Fecha_Act'] >= startDate),

      # Fase 2: Tiene alguna actualización
      (baseDF['Tiene_Act'] == 1) & (baseDF['Fecha_Act'] >= startDate)
  ]

  # 2. Definir las salidas correspondientes para cada condición
  choices = [
      'Fase 4 — Liquidado',
      'Fase 3 — Avance',
      'Fase 2 — Actualizado'
  ]

  # 3. Aplicar np.select para crear la columna 'Fase'. El valor por defecto será 'Fase 1 — Sin Actualización'
  baseDF['Fase'] = np.select(conditions, choices, default='Fase 1 — Sin Actualización MEC')

  # Creamos la Columna Plazos_PaB usando el diccionario BANCOS_PLAZOS para asignar la Información, cuando no exista será 1
  BANCO_PLAZOS = defaultdict(lambda: 1, BANCOS_PLAZOS)
  baseDF['Plazos_PaB'] = baseDF['Banco'].map(BANCO_PLAZOS)

  # Imprimimos para Logs
  if verbose:
    print('✅Cálculos Necesarios para Análisis calculados con Éxito')
    print('🚼Filas totales: {} filas, 🆔Columnas Totales: {}'.format(*baseDF.shape))

  # Devolvemos el DF
  return baseDF

## **Cálculos para Tradicional**
___
Los cálculos que se van a aplicar para el seguimiento de Tradicional son:
- **Establecer Deudas no Viables a Calcular**: Se escoge que deudas no es necesario aplicar ninguno de estos cálculos
- **Preparar Datos de Cálculo**: Se preparan los diferentes datos para realizar todos los cálculos necesarios
- **Desgaste del Ahorro**: Se desgasta el ahorro del cliente en las diferentes deudas para eliminar duplicados de ahorro.
- **Potencial_Tradicional**: Los DE Necesarios para Liquidar una deuda con el Ahorro del Cliente
- **Potencial_Estructurado**: Los DE Necesarios para sacar un Estructurado dadas las condiciones previstas.
- **PaB_Ideal_Tradicional**: El PaB al cuál llegar para que pueda existir una reestructura dadas las condiciones previstas.

### **Establecer Deudas No Viables - Tradicional**
___
En este caso, no se van a tener en cuenta las deudas que cumplan alguna de las condiciones:
- **Deuda con Sub-Estado Especial**: Los subestados son:
  - **drop_requested**: Solicitud de quitar la deuda del programa
  - **cancelled**: Deuda sacada del Programa
- **Clientes con Batch_Credito 'Cr Parcial'**
- **Clientes con Mora_Credito > 0**
- **Clientes con Es_Baja > 0**

De aquí se crea la Columna **Motivo_NV** y la Columna **Es_Viable**, para poner luego en el Potencial.

In [ ]:
def establishNV(baseDF: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
  # Creamos una Copia del DF
  baseDF = baseDF.copy()

  # Máscara 1: Deudas con Sub-Estado específico
  subEstadosNV = ['drop_requested','cancelled']
  maskSubEstado = baseDF['Sub_Estado'].isin(subEstadosNV)

  # Máscara 2: Clientes con Batch_Credito == 'Cr Parcial'
  maskBatchCredito = (baseDF['Batch_Credito'] == 'Cr Parcial')

  # Máscara 3: Clientes con Mora_Credito > 0
  maskMoraCredito = (baseDF['Mora_Credito'] > 0)
  refsMoraCredito = baseDF.loc[maskMoraCredito, 'Referencia'].unique()
  maskMoraCredito = baseDF['Referencia'].isin(refsMoraCredito)

  # Máscara 4: Clientes con Es_Baja > 0
  maskEsBaja = (baseDF['Es_Baja'] > 0) | (baseDF['Estado_Cliente'] == 'drop')
  refsEsBaja = baseDF.loc[maskEsBaja, 'Referencia'].unique()
  maskEsBaja = baseDF['Referencia'].isin(refsEsBaja)

  # --- Creación de Columna Motivo_NV ---
  # Definimos las Condiciones
  conditions = [
      maskSubEstado,
      maskBatchCredito,
      maskMoraCredito,
      maskEsBaja
  ]

  # Definimos las Opciones para cada Condición
  choices = [
      'Sub-Estado No Viable',
      'Tiene Crédito',
      'Mora en Crédito',
      'Baja'
  ]

  # Aplicamos la Creación de la Columna Motivo_NV usando np.select con default = 'Viable'
  baseDF['Motivo_NV'] = np.select(conditions, choices, default='Viable')

  # Creamos la Columna Es_Viable dependiendo de si es viable o No
  baseDF['Es_Viable'] = np.where(baseDF['Motivo_NV'] == 'Viable', 1, 0)

  # Para las Deudas No Viables Cambiamos Estructurable a 0
  baseDF.loc[baseDF['Motivo_NV'] != 'Viable', 'Estructurable'] = 0

  # Imprimimos para Logs
  if verbose:
    print('✅Columna Motivo_NV creada con Éxito')
    print('ℹ️Deudas no Viables: {}'.format((baseDF['Motivo_NV'] != 'Viable').sum()))
    print('🆔Distribución de Motivos:')
    print(baseDF['Motivo_NV'].value_counts())

  # Devolvemos el DF
  return baseDF

### **Preparar Datos de Cálculos**
___
En este apartado se aplican los siguientes cálculos:
- **Columnas Actualizadas vs Esperadas**: Se realizan los cálculos bajo la definición de: Si existe actualización se usa, de lo contrario se asume el descuento esperado
- **Columna Eficiencia_CE**: Mide la Eficiencia de la Comisión de Éxito basado en el Pago Total, es decir = $\frac{CE}{PaB + CE}$
- **Columna Ahorro_Necesario_Est**: Es el Ahorro Necesario Actual para que el Estructurado se pueda realizar. Tiene las Formulas Iguales al Potencial Estructurado pero despejando el Ahorro.
- **Portafolios**: Se van a crear 2 columnas
  - **Es_Portafolio**: Con 0s y 1s se representa si la deuda hace parte de los Bancos Portafolio
  - **Lider_Portafolio**: Con 0s y 1s se representa si esa deuda va a ser la representante de dicho Portafolio

**A tener en cuenta**: Cuando se aplican los portafolios se calculan solo para los datos viables.

In [ ]:
def prepareTrad(baseDF: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
  # Creamos una copia del DF
  baseDF = baseDF.copy()

  # Creamos Columna Descuento_Auxiliar como Descuento_Act
  baseDF['Descuento_Auxiliar'] = baseDF['Descuento_Act']

  # Calculamos PaB_Auxiliar como Dum_Bravo * (1 - Descuento_Auxiliar)
  baseDF['PaB_Auxiliar'] = baseDF['Dum_Bravo'] * (1 - baseDF['Descuento_Auxiliar'])

  # Calculamos PaB_Or_Aux como Dum_Bravo
  baseDF['PaB_Or_Aux'] = baseDF['Dum_Bravo']

  # Calculamos Ingreso_Posible como Dum_Bravo * Descuento_Auxiliar * Pricing * 1.19 (con IVA, luego se quita)
  baseDF['Ingreso_Posible'] = baseDF['Dum_Bravo'] * baseDF['Descuento_Auxiliar'] * baseDF['Pricing'] * 1.19

  # Limitamos como Minimo la Comision de Exito a 0
  baseDF['Ingreso_Posible'] = np.where(baseDF['Ingreso_Posible'] < 0, 0, baseDF['Ingreso_Posible'])

  # Inicializamos Es_Portafolio con 0
  baseDF['Es_Portafolio'] = 0
  # Creamos la Columna Es_Portafolio bajo la siguiente lógica de agrupación:
  # Banco esta en BANCOS_PORTAFOLIO y Es_Viable es el Mismo
  maskPort = baseDF['Banco'].isin(BANCOS_PORTAFOLIO)
  # Calculate the size of each group and check if it's greater than 1
  group_sizes = baseDF.loc[maskPort].groupby(['Referencia','Banco','Es_Viable','Tiene_Act'])['Referencia'].transform('size')

  # Assign 1 if size > 1, else 0, mapping directly to the masked rows
  baseDF.loc[maskPort, 'Es_Portafolio'] = (group_sizes > 1).astype(int)

  # Ahora Actualizamos PaB_Auxiliar, PaB_Or_Aux y Ingreso_Posible bajo la misma lógica de agrupàción sumando los resultados
  baseDF.loc[maskPort,'PaB_Auxiliar'] = baseDF.loc[maskPort].groupby(['Referencia','Banco','Es_Viable','Tiene_Act'])['PaB_Auxiliar'].transform('sum')
  baseDF.loc[maskPort,'PaB_Or_Aux'] = baseDF.loc[maskPort].groupby(['Referencia','Banco','Es_Viable','Tiene_Act'])['PaB_Or_Aux'].transform('sum')
  baseDF.loc[maskPort,'Ingreso_Posible'] = baseDF.loc[maskPort].groupby(['Referencia','Banco','Es_Viable','Tiene_Act'])['Ingreso_Posible'].transform('sum')

  # Definimos las columnas de agrupación
  group_cols = ['Referencia', 'Banco', 'Es_Viable', 'Tiene_Act']

  # 1. Creamos una copia/vista con las columnas necesarias y calculamos el producto ponderado
  sub_df = baseDF.loc[maskPort, group_cols + ['Descuento_Auxiliar', 'PaB_Real']].copy()
  sub_df['Producto_Ponderado'] = sub_df['Descuento_Auxiliar'] * sub_df['PaB_Real']

  # 2. Sumamos por grupo de manera vectorial (sin usar apply)
  grouped = sub_df.groupby(group_cols)[['Producto_Ponderado', 'PaB_Real']].sum()

  # 3. Calculamos el promedio ponderado dividiendo la suma del producto entre la suma de los pesos
  weighted_avgs = grouped['Producto_Ponderado'] / grouped['PaB_Real']

  # 4. Asignamos de vuelta los valores al DataFrame original usando .map() con la serie indexada
  baseDF.loc[maskPort, 'Descuento_Auxiliar'] = (
      baseDF.loc[maskPort, group_cols]
      .set_index(group_cols)
      .index.map(weighted_avgs)
  )

  # Creamos la Columna Mayor_PL que será la diferencia entre el Descuento_Auxiliar y el Descuento_PL
  baseDF['Mayor_PL'] = (baseDF['Descuento_Auxiliar'] >= baseDF['Descuento_PL']).astype(int)

  # Creamos la Columna Eficiencia_CE como: Ingreso_Posible / (Ingreso_Posible + PaB_Auxiliar)
  baseDF['Eficiencia_CE'] = baseDF['Ingreso_Posible'] / (baseDF['Ingreso_Posible'] + baseDF['PaB_Auxiliar'])

  # Último: Creamos la Deuda Representante del Portafolio que será bajo la lógica:
  # Ordenamos por Tiene_Act, Es_Viable, Es_Portafolio; luego aplicamos el groupby y dejamos la primera deuda como representante del portafolio
  baseDF = baseDF.sort_values(by=['Es_Viable','Tiene_Act','Es_Portafolio','Mayor_PL','Eficiencia_CE'],
                              ascending=[False,False,False,False,False])
  baseDF['Deuda_Representante'] = baseDF.groupby(['Referencia','Banco','Es_Viable','Tiene_Act'])['Id_Deuda'].transform('first')
  baseDF['Lider_Portafolio'] = np.where(baseDF['Id_Deuda'] == baseDF['Deuda_Representante'], 1, 0)

  # Volvemos Lider_Portafolio a 1 para los casos donde el banco no esta en el Portafolio
  baseDF['Lider_Portafolio'] = np.where(~baseDF['Banco'].isin(BANCOS_PORTAFOLIO), 1, baseDF['Lider_Portafolio'])
  # Cambiamos Deuda_Representatne a el Id_Deuda para los casos donde el abnco no esta en el Portafolio
  baseDF['Deuda_Representante'] = np.where(~baseDF['Banco'].isin(BANCOS_PORTAFOLIO), baseDF['Id_Deuda'], baseDF['Deuda_Representante'])

  # Creamos la Columna Deudas_Contar con el Conteo de Deuda Representante
  baseDF['Deudas_Contar'] = baseDF.groupby('Deuda_Representante')['Id_Deuda'].transform('count')

  # --- Creacion de Columna Ahorro_Necesario_Est ---
  # Condicion 1: Primera Comisión
  cond1Ahorro = (baseDF['PaB_Auxiliar'] / baseDF['Plazos_PaB']) + baseDF['Ingreso_Posible'] * COMISION_INICIAL
  # Condición 2: Todo el PaB
  cond2Ahorro = (baseDF['PaB_Auxiliar'] + baseDF['Ingreso_Posible'] *(
      COMISION_INICIAL + (((1 - COMISION_INICIAL) / (MESES_ESTRUCTURACION - 1)) * (baseDF['Plazos_PaB'] - 1)
  )
  ) - (baseDF['Apartado_Neto'] * (baseDF['Plazos_PaB']-1)))
  # Condición 3: Todo el Estructurado
  cond3Ahorro = (baseDF['PaB_Auxiliar'] + baseDF['Ingreso_Posible'] - (baseDF['Apartado_Neto'] * (MESES_ESTRUCTURACION-1)))

  # Definimos el Ahorro Máximo entre los 3
  ahorroParaEstrMax = np.maximum(cond1Ahorro, np.maximum(cond2Ahorro, cond3Ahorro))

  # Actualizamos la Columna Ahorro_Necesario_Est con este
  baseDF['Ahorro_Necesario_Est'] = ahorroParaEstrMax

  # Creamos la Columna Ahorro_Necesario_Liq como: PaB_Auxiliar + Ingreso_Posible
  baseDF['Ahorro_Necesario_Liq'] = baseDF['PaB_Auxiliar'] + baseDF['Ingreso_Posible']

  # Imprimimos para Logs
  if verbose:
    print('✅Cálculo de Portafolio y Columnas Auxiliares realizados con éxito')

  return baseDF

### **Desgaste de Ahorro (Clogging Fundamental)**
___
Dado que no se puede asumir que un cliente puede liquidar muchas deudas al mismo tiempo manteniendo el mismo tipo de ahorro (esto sería duplicar el ahorro). Es Necesario aplicar un desgaste del ahorro de modo que se escoja la mejor deuda, cúal se aplica bajo estas escogencias:
- **Viabilidad**: Las Deudas que si son viables
- **Tiene Actualización**: La deuda ya cuenta con una actualización
- **Distancia del Descuento al Descuento PL**: Se priorizan primero las deudas que ya cumplen el plan de liquidación


La Escogencia de la mejor deuda para desgastar el ahorro esta dada por la **Eficiencia de Ingreso**, es decir: Que tanto ingreso se obtiene en comparación con el pago total ($\frac{CE}{PaB+CE}$)

In [ ]:
def makeDesgaste(baseDF: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
  baseDF = baseDF.copy()
  # Desgaste del Ahorro V2: Vectorizado
  inicio = time()

  # Paso 1: Columnas Auxiliares de Ponderacion
  baseDF['Pond_Eficiencia_CE'] = baseDF['Dum_Bravo'] * baseDF['Eficiencia_CE']
  baseDF['Pond_Descuento_PL'] = baseDF['Dum_Bravo'] * baseDF['Descuento_PL']
  baseDF['Pond_Tiene_Act'] = baseDF['Dum_Bravo'] * baseDF['Tiene_Act']
  baseDF['Pond_Descuento_Act'] = baseDF['Dum_Bravo'] * baseDF['Descuento_Auxiliar']

  # Paso 2: Agrupación por Deuda Representante
  ahorroV2DF = baseDF.groupby('Deuda_Representante').agg({
      'Es_Viable': 'mean',
      'Referencia': 'first',
      'Apartado_Neto': 'first',
      'Ahorro_Neto': 'first',
      'Por_Cobrar': 'first',
      'Ahorro_Necesario_Est': 'first',
      'Ahorro_Necesario_Liq': 'first',
      'Comision_Mensual': 'first',
      'Dum_Bravo': 'sum',
      'Pond_Eficiencia_CE': 'sum',
      'Pond_Descuento_PL': 'sum',
      'Pond_Tiene_Act': 'sum',
      'Pond_Descuento_Act': 'sum',
  }).reset_index()

  # Creamos Ahorro_Auxiliar y Ahorro_Neto como Ahorro_Neto
  ahorroV2DF['Ahorro_Auxiliar'] = ahorroV2DF['Ahorro_Neto'].values
  ahorroV2DF['Ahorro_Estructurado'] = ahorroV2DF['Ahorro_Neto'].values

  # Al Ahorro_Auxiliar y al Ahorro_Estructurado le restamos la Comision_Mensual y el Por_Cobrar
  ahorroV2DF['Ahorro_Auxiliar'] -= (ahorroV2DF['Comision_Mensual'] + ahorroV2DF['Por_Cobrar'])
  ahorroV2DF['Ahorro_Estructurado'] -= (ahorroV2DF['Comision_Mensual'] + ahorroV2DF['Por_Cobrar'])

  # Paso 3: Ajustar las Ponderaciones dividiendo Dum_Bravo
  ahorroV2DF['Pond_Eficiencia_CE'] /= ahorroV2DF['Dum_Bravo']
  ahorroV2DF['Pond_Descuento_PL'] /= ahorroV2DF['Dum_Bravo']
  ahorroV2DF['Pond_Tiene_Act'] /= ahorroV2DF['Dum_Bravo']
  ahorroV2DF['Pond_Descuento_Act'] /= ahorroV2DF['Dum_Bravo']

  # Generamos de Nuevo Mayor_PL como Pond_Descuento_Act>= Pond_Descuento_PL
  ahorroV2DF['Mayor_PL'] = (ahorroV2DF['Pond_Descuento_Act'] >= ahorroV2DF['Pond_Descuento_PL']).astype(int)

  # Paso 4: Ordenamiento por Prioridad
  # Aplicamos el Ordenamiento por Es_Viable, Tiene_Act y Mayor_PL de forma descendente
  ahorroV2DF = ahorroV2DF.sort_values(by=['Es_Viable', 'Pond_Tiene_Act', 'Mayor_PL','Pond_Eficiencia_CE'],
                                      ascending=[False,False,False,False])

  # Paso 5: Columnas Auxiliares
  # Creamos la suma Acumulada de los pagos por Cliente (Referencia)
  ahorroV2DF['Suma_Pagos_Normal'] = ahorroV2DF.groupby('Referencia')['Ahorro_Necesario_Liq'].cumsum()
  ahorroV2DF['Suma_Pagos_Estr'] = ahorroV2DF.groupby('Referencia')['Ahorro_Necesario_Est'].cumsum()

  # Creamos el Ahorro_Previo_Gastado como el shift de los pagos del cliente
  ahorroV2DF['Ahorro_Previo_Gastado_Normal'] = ahorroV2DF['Suma_Pagos_Normal'].groupby(ahorroV2DF['Referencia']).shift(1).fillna(0)
  ahorroV2DF['Ahorro_Previo_Gastado_Estr'] = ahorroV2DF['Suma_Pagos_Estr'].groupby(ahorroV2DF['Referencia']).shift(1).fillna(0)

  # Definimos el Ahorro_Disponible como el Ahorro_Actual - el Ahorro_Previo_Gastado
  # Condición especial: Si el Ahorro es Negativo se deja igual
  ahorroV2DF['Ahorro_Auxiliar'] = np.where(ahorroV2DF['Ahorro_Auxiliar'] < 0, ahorroV2DF['Ahorro_Auxiliar'],
                                        np.maximum(ahorroV2DF['Ahorro_Auxiliar'] - ahorroV2DF['Ahorro_Previo_Gastado_Normal'], 0))
  ahorroV2DF['Ahorro_Estructurado'] = np.where(ahorroV2DF['Ahorro_Estructurado'] < 0, ahorroV2DF['Ahorro_Estructurado'],
                                        np.maximum(ahorroV2DF['Ahorro_Estructurado'] - ahorroV2DF['Ahorro_Previo_Gastado_Estr'], 0))

  # Por último, el Ahorro que cubrió la deuda será el minimo entre el valor que se necesitaba pagar y el que se dejo
  ahorroV2DF['Ahorro_Auxiliar'] = np.minimum(ahorroV2DF['Ahorro_Necesario_Liq'], ahorroV2DF['Ahorro_Auxiliar'])
  ahorroV2DF['Ahorro_Estructurado'] = np.minimum(ahorroV2DF['Ahorro_Necesario_Est'], ahorroV2DF['Ahorro_Estructurado'])

  # Ahora que tenemos el Ahorro_Auxiliar y el Ahorro_Estructurado, procedemos a unirlo con baseDF
  baseDF = pd.merge(
      baseDF,
      ahorroV2DF[['Deuda_Representante', 'Ahorro_Auxiliar', 'Ahorro_Estructurado']],
      on='Deuda_Representante',
      how='inner'
  )
  fin = time()
  if verbose:
    print('\n---✅Desgaste de Ahorro realizado con éxito---')
    print('---⏱️Tiempo de Desgaste de Ahorro: {} segundos---\n'.format(round(fin - inicio, 2)))

  return baseDF

### **Cálculo de Potencial Tradicional**
___
El Cálculo del Potencial Tradicional se define de la siguiente forma:
$$Potencial\_Tradicional = \frac{PaB\_Actual + CE\_Actual - (Ahorro\_Neto)}{Apartado\_Neto}$$

In [ ]:
def calcPotTrad(baseDF: pd.DataFrame, verbose = True) -> pd.DataFrame:
  # Creamos una copia del DF
  baseDF = baseDF.copy()
  # Inicializamos Potencial_Tradicional con NaN
  baseDF['Potencial_Tradicional'] = np.nan

  # Definimos la Máscara de Posibles a Aplicar (El Apartado_Neto es > 0)
  aplicar = (baseDF['Apartado_Neto'] > 0)
  appDF = baseDF[aplicar].copy()

  # Cálculo 1: Primera Comision
  ahorro = np.where(appDF['Ahorro_Auxiliar']>=0,appDF['Ahorro_Auxiliar'],appDF['Ahorro_Neto'])
  potTrad = ((appDF['PaB_Auxiliar'] + appDF['Ingreso_Posible']) - ahorro) / appDF['Apartado_Neto']

  # Actualizamos el Potencial Tradicional en el DataFrame Original
  baseDF.loc[aplicar, 'Potencial_Tradicional'] = np.ceil(potTrad)

  # Imprimimos para Logs
  if verbose:
    print('✅Cálculo de Potencial Tradicional realizado con éxito, 🚯Datos sin Aplicar: {}'.format(
        np.sum(baseDF['Potencial_Tradicional'].isna())
    ))

  return baseDF

### **Ajustes de No Viabilidades**
___
Por Último, se ajustan las No-Viabilidades dentro de los Potenciales y el PaB_Ideal_Tradicional de modo que se pueda definir que puede salir y que no. Además, los Portafolios que no sean liderés se les añade la etiqueta **'- En Portafolio'**

In [ ]:
def cleanCalcs(baseDF: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
  # Creamos una copia del DF
  baseDF = baseDF.copy()
  # A. --- Limpieza de Potenciales y PaB Ideal para Análisis ---

  # Guardamos Copias de los Originales e Imputamos NaNs con 100
  baseDF['Pot_Trad_Original'] = baseDF['Potencial_Tradicional'].values
  imputeNans(baseDF, 'Pot_Trad_Original', 100)

  # Volvemos Ingreso_Esperado e Ingreso_Posible a np.ceil
  baseDF['Ingreso_Esperado'] = np.ceil(baseDF['Ingreso_Esperado'])
  baseDF['Ingreso_Posible'] = np.ceil(baseDF['Ingreso_Posible'] / 1.19)  # Quitamos el IVA

  # Guardamos la Copia del Ingreso
  baseDF['Ingreso_Original'] = baseDF['Ingreso_Posible'].copy(deep=True)
  # Imputamos el Ingreso_Original con 0
  imputeNans(baseDF, 'Ingreso_Original', 0)

  # Primero Creamos la Máscara de las Inviabilidades
  maskInviable = (baseDF['Es_Viable'] == 0)

  # Los Portafolios Viables que no sean Lider se coloca:
  # {Valor} - En Portafolio
  maskPortafolio = (baseDF['Es_Portafolio'] == 1) & (baseDF['Es_Viable'] == 1) & (baseDF['Lider_Portafolio'] == 0)

  # 1. Creamos las Máscaras de NaNs para los Potenciales, Depósitos Extra y PaB Ideal y el PaB Ideal
  colsAdjust = ['Potencial_Tradicional','Ingreso_Posible']
  for col in colsAdjust:
    # Creamos la Máscara de NaNs
    maskNaN = baseDF[col].isna()
    # Volvemos la Columna a String
    baseDF[col] = baseDF[col].astype(str)
    # Cambiamos los NaNs por 'No Viable - AM = CM'
    baseDF.loc[maskNaN, col] = 'No Viable - AM = CM'
    # Ajustamos las Inviabilidades agregando ' - No Viable'
    baseDF.loc[maskInviable & ~maskNaN, col] = baseDF.loc[maskInviable & ~maskNaN, col] + ' - No Viable'

  if verbose:
    print('✅Limpieza de Potenciales, Depósitos Extra y PaB Ideal realizada con éxito')
    print('ℹ️Inviables Actualizados: {}'.format(maskInviable.sum()))
    print('ℹ️Portafolios Actualizados: {}'.format(maskPortafolio.sum()))

  return baseDF

## **Re-Ajuste del Bucket**
___
Dado que las condiciones del cliente pueden cambiar en el mes, se va reajustando el Bucket para dejar los clientes relevantes como relevantes.

### **Pasos de Cálculo del Bucket**
___
El Bucket inicial se basa en la Contactabilidad distribuida de forma equitativa

Los Ajustes que se aplican al Bucket son:
- **Penalización por Liquidación**: Aumenta 1
- **Mora Estructurado**: Hay Mora -> Mínimo Bucket 4, No hay -> Mínimo Bucket 3
- **Distribución de Potencial Tradicional**: Se calcula la Distribución en grupos y se crea un nuevo bucket
- **Distribución de Potencial Estructurado**: Potencial <= 0 -> Baja 1, de lo contrario sube 1
- **Distrbución de Potencial de Crédito**: Potencial <= 1 -> Baja 1, Potencial >= 5 -> Sube 1
- **Categorías de Potencial de Crédito**: Potencial = Tradicional -> Máximo 1, Potencial = No Viable -> Mínimo 4
- **Distribución de Mora de Deuda**
  - **Mora Baja**: Mora <= 4 -> Se Calcula Distribución  donde: Moras Bajas Bajas Min 5, Moras Bajas Altas Min 4
  - **Mora Media y >3 Meses en el Programa**: Baja 1, Es 6 <= Mora <= 15
  - **Mora Alta**: Mora > 15 -> Se Calcula Distribución  donde: Moras Altas Bajas Min 3, Moras Bajas Altas Min 4
- **Mora en Crédito**: Minímo 5
- **Categorías Inviables**: Mínimo 5, las Categorías son:
  - **NO_VIABLE**
  - **DEMANDA**
  - **ILOCALIZADO**
  - **PAGO_POR_FUERA**
- **Calificación de Bancos**: Se Calcula una Score de Bancos y se distribuye
- **Proporción Ahorro Deuda**: Se calcula la Distribución en grupos y se crea un nuevo bucket
- **Dum_Bravo muy Alto**: Si el Dum_Bravo excede el 190% de la deuda promedio se fuerza un Bucket bajo sin quitar los otros limites (Exluyendo Deudas de Davivienda)
- **Negociador por Referencia**: Se escoge el Negociador por Referencia, cuando hay más de una asignación se escoge el más frecuente

### **Diferenciación de Buckets**
___
Los Buckets que se calculan dependiendo de diferentes condiciones son:
- **Bucket de Contactabilidad (Inicial)**
- **Bucket de Potencial**
- **Bucket de Banco**
- **Bucket de Proporcion Ahorro/Deuda**

Luego de esto se crea una **Score** para las Referencias de modo que luego sean distribuidas de peor a mejor en los diferentes Buckets.

### **Distribución de Buckets**
___
Una vez todos los cálculos estan realizados, se aplica esta lógica:
1. Se definen los **clientes por Bucket** según las Proporciones Dadas
2. Se ajusta el **tamaño de los Bucket** dependiendo de los clientes dificiles (Bucket 5 Mínimo)
3. El resto de clientes se asigna de forma equitativa **según el Score**
4. Los clientes que no tienen Bucket se calculan manualmente según la **Distribución del Score**

### **Pasos de Cálculo**
___
Estos son los pasos a seguir para poder calcular el bucket de forma efectiva:
1. **Calcular Bucket de Contactabilidad y de Potencial**, dejando el Máximo para definir el Bucket Base
2. **Calcular los Impulsos y Mínimos** para todas las Referencias
3. **Calcular los Otros Buckets Faltantes** según sus condiciones individuales
4. **Aplicar la Distribución de los Buckets**

### **Paso 1: Definir el Bucket Base**
___
Se define el Bucket inicial teniendo en cuenta:
- **Contactabilidad**
- **Liquidaciones**
- **Mora_Estructurado**
- **Potencial**
- **Estructurabilidad**
- **Potencial de Crédito**

In [ ]:
def calcInitBucket(baseDF: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
  # Creamos una copia del DF
  baseDF = baseDF.copy()
  # --- Generación de Bucket Inicial
  # Imputamos Meses_Sin_Contacto con 12
  imputeNans(baseDF, 'Meses_Sin_Contacto', 12)
  # Creamos el Bucket_Base dividiendo Meses_Sin_Contacto de forma Equitativa en 6 buckets (de 0 a 5)
  baseDF['Bucket_Base'] = pd.qcut(baseDF['Meses_Sin_Contacto'].rank(method='first'), q=6, labels=list(range(6)))

  # Creamos maskLiqMes
  maskLiqMes = baseDF['Tipo_Liq'].notna()
  refsLiqMes = baseDF.loc[maskLiqMes, 'Referencia'].unique()
  maskLiqMes = baseDF['Referencia'].isin(refsLiqMes)

  # Añadimos Penalización de +1 para Referencias con Liquidaciones este mes (Dejando rango 0-5)
  baseDF['Bucket_Base'] = np.where(maskLiqMes, baseDF['Bucket_Base'].astype(int) + 1, baseDF['Bucket_Base']).clip(max=5)
  # Añadimos Penalización de +1 para Mora_Estructurado > 1 (Dejando rango 0-5)
  baseDF['Bucket_Base'] = np.where((baseDF['Mora_Estructurado'] > 1), baseDF['Bucket_Base'].astype(int) + 1, baseDF['Bucket_Base']).clip(max=5)
  # Añadimos Impulso para Mora_Estructrado == 0 (Dejando rango 0-5)
  baseDF['Bucket_Base'] = np.where((baseDF['Mora_Estructurado'] == 0), baseDF['Bucket_Base'].astype(int) - 1, baseDF['Bucket_Base']).clip(min=0)

  # Primer Ajuste de Bucket: Se crea el Bucket_Potencial dividiendo el Pot_Trad_Original de forma Equitativa en 6 buckets (de 0 a 5)
  baseDF['Bucket_Potencial'] = pd.qcut(baseDF['Pot_Trad_Original'].rank(method='first', na_option='bottom'), q=6, labels=list(range(6)))
  # Ahora Fusionamos el Bucket_Potencial con el Bucket_Base dejando el Máximo entre los 2 como Bucket_Base
  baseDF['Bucket_Base'] = np.maximum(baseDF['Bucket_Base'].astype(int), baseDF['Bucket_Potencial'].astype(int))

  # Añadimos Penalización de +1 para Potencial_Credito >= 5
  potCredReal = pd.to_numeric(baseDF['Potencial_Credito'], errors='coerce')
  baseDF['Bucket_Base'] = np.where((potCredReal >= 5), baseDF['Bucket_Base'].astype(int) + 1, baseDF['Bucket_Base']).clip(max=5)
  # Añadimos Impulso para Potencial_Credito <= 1
  baseDF['Bucket_Base'] = np.where((potCredReal <= 1), baseDF['Bucket_Base'].astype(int) - 1, baseDF['Bucket_Base']).clip(min=0)
  # Definimos Límite cuando Potencial_Credito como string contiene 'tradicional', Límite Bucket 1
  baseDF['Bucket_Base'] = np.where((baseDF['Potencial_Credito'].astype(str).str.contains('tradicional', case=False, na=False)), baseDF['Bucket_Base'].clip(lower=1), baseDF['Bucket_Base'])
  # Definimos Limite cuando Potencial_Credito como string contiene 'no', Límite Bucket 4
  baseDF['Bucket_Base'] = np.where((baseDF['Potencial_Credito'].astype(str).str.contains('no', case=False, na=False)), baseDF['Bucket_Base'].clip(lower=4), baseDF['Bucket_Base'])

  # -- Añadimos Re-Ajuste de Bucket para Mora_Deuda < 5, dejando Bucket 4 o 5 dependiendo de que tan reciente es
  maskMoraBaja = (baseDF['Mora_Deuda'] < 5) & (baseDF['Mora_Deuda'] >= 0)
  # Distribuimos de forma equitativa en 2 buckets dependiendo de la Mora
  distrMora = pd.qcut(baseDF.loc[maskMoraBaja, 'Mora_Deuda'], q=2, labels=[4,5])
  # Aplicamos el Ajuste para la Máscara de Mora Baja dejando como mínimo distrMora
  baseDF['Bucket_Base'] = np.where(maskMoraBaja, np.maximum(baseDF['Bucket_Base'].astype(int), distrMora.astype(int)), baseDF['Bucket_Base'])

  ## -- Añadimos Re-Ajuste de Bucket para Mora > 15, dejando Bucket 3 o 4 dependiendo de que tan reciente es
  maskMoraAlta = (baseDF['Mora_Deuda'] > 15)
  # Distribuimos de forma equitativa en 2 buckets dependiendo de la Mora
  distrMoraAlta = pd.qcut(baseDF.loc[maskMoraAlta, 'Mora_Deuda'], q=2, labels=[3,4])
  # Aplicamos el Ajuste para la Máscara de Mora Alta dejando como mínimo distrMoraAlta
  baseDF['Bucket_Base'] = np.where(maskMoraAlta, np.maximum(baseDF['Bucket_Base'].astype(int), distrMoraAlta.astype(int)), baseDF['Bucket_Base'])

  ## Añadimos Impulso de Bucket para (Mora entre 5 y 15) y (Meses_en_Programa > 3)
  maskMoraMedia = (baseDF['Mora_Deuda'] >= 5) & (baseDF['Mora_Deuda'] <= 15) & (baseDF['Meses_en_Programa'] > 3)
  baseDF['Bucket_Base'] = np.where(maskMoraMedia, baseDF['Bucket_Base'].astype(int) - 1, baseDF['Bucket_Base']).clip(min=0)

  # Por Último, como el Bucket es por Cliente, se deja el máximo Bucket_Base por Referencia
  baseDF['Bucket_Base'] = baseDF.groupby('Referencia')['Bucket_Base'].transform('max')

  # Mostramos la Distribución Final del Bucket_Base
  if verbose:
    print('✅Bucket Base generado con éxito, 🚼Distribución de Buckets:')
    print(baseDF['Bucket_Base'].value_counts().sort_index())

  return baseDF

### **Paso 2: Definir Clientes con Bucket 5 Obligatorio**
___
Los Clientes con Bucket 5 Obligatorio son:
- **Mora en Crédito**
- **Dentro de las Etiquetas no Viables**
- **Clientes Baja**

In [ ]:
def calcBucket5(baseDF: pd.DataFrame, verbose: True) -> pd.DataFrame:
  # Creamos una copia del DF
  baseDF = baseDF.copy()
  # Máscara de Bucket 5 Obligatorio es:
  # Mora_Credito > 0
  # Etiqueta_Act.isin(BUCKET_CATEGORIAS_NV)
  # Es_Baja > 0

  maskMoraCred = (baseDF['Mora_Credito'] > 0)
  stringRegexCat = '|'.join([cat.replace('_',' ') for cat in BUCKET_CATEGORIAS_NV])
  maskCatNV = baseDF['Etiqueta_Act'].str.contains(stringRegexCat, case=False, na=False)
  maskEsBaja = (baseDF['Es_Baja'] > 0)
  maskLiqMes = baseDF['Tipo_Liq'].notna()

  maskBucket5 = maskMoraCred | maskCatNV | maskEsBaja | maskLiqMes  # El Ajuste se realizará después

  # Obtenemos las Referencias que cumplen con la Máscara de Bucket 5 Obligatorio
  refsBucket5 = baseDF.loc[maskBucket5, 'Referencia'].unique()

  if verbose:
    print('🤔Número de Deudas en Bucket 5 Obligatorio:',len(refsBucket5))
    print('- 🫴Con Mora en Crédito: {}'.format(np.sum(maskMoraCred)))
    print('- 🫴Con Actualización Inviable: {}'.format(np.sum(maskCatNV)))
    print('- 🫴Son Baja: {}'.format(np.sum(maskEsBaja)))
    print('- 🫴Con Liquidación este Mes: {}'.format(np.sum(maskLiqMes)))

  return refsBucket5

### **Paso 3: Definir el Bucket por Banco**
___
Para esto, se tiene la lista de los Bancos Organizados de más relevante a menos relevante, de modo que su Score sea netamente su posición en la lista.

In [ ]:
def calcBucketBank(baseDF: pd.DataFrame, verbose: bool =True) -> pd.DataFrame:
  # Creamos una copia del DF
  baseDF = baseDF.copy()
  # 1. Definimos el Orden de los Bancos
  bancosOrder = ["Cobrando","Rappipay","Alkomprar","Rappicard","Pichincha","Bancolombia",
      "Scotiabank Colpatria","Davi Bank","Rapicredit","Bancoomeva","Codensa",
      "Covinoc","Zinobe","Banco Popular","SisteCredito","Banco AV Villas","Colsubsidio",
      "Refinancia","Tuya","Credijamar","Agaval","Alkosto","Banco Falabella","Banco de Bogotá",
      "Sistemcobro","Bancamia","Banco Finandina","Itaú","FGA",
      "BBVA Colombia","Serlefin","Flamingo","Mundo Mujer","Banco Davivienda","Banco Caja Social","Comultrasan",
      "Juriscoop","Credivalores","Banco de Occidente","Compensar","JOHN","Fincomercio","GNB Sudameris","Serfinanza","Nu","LuloBank",
      "Aslegal","Baninca","Coltefinanciera","Confiar","Contacto solución","Garantias Comunitarias","GRUPO JURIDICO DEUDU",
      "Inversionistas Estratégicos","Juancho te Presta","QNT SAS",
  ]

  # Creamos el Diccionario de Mapeo de Bancos en su Orden
  bancoMapping = {banco: idx for idx, banco in enumerate(bancosOrder)}
  # Lo Convertimos en un defaultdict con Valor por Defecto del Promedio: len(bancosOrder)//2
  bancoMapping = defaultdict(lambda: len(bancosOrder)//2, bancoMapping)

  # Creamos la Columna Bucket_Banco usando el mapeo de bancos
  baseDF['Bucket_Banco'] = baseDF['Banco'].map(bancoMapping)
  # Ahora Creamos el Score_Banco que será Agrupando por Referencia:
  # max(Bucket_Banco) * BUCKET_DISTRIBUCION_BANCO + min(Bucket_Banco) * (1 - BUCKET_DISTRIBUCION_BANCO)
  baseDF['Score_Banco'] = baseDF.groupby('Referencia')['Bucket_Banco'].transform(
      lambda x: x.max() * BUCKET_DISTRIBUCION_BANCO + x.min() * (1 - BUCKET_DISTRIBUCION_BANCO)
  )

  # Ahora esta score la distribuimos como Bucket 0-5
  baseDF['Bucket_Banco'] = pd.qcut(baseDF['Score_Banco'], q=6, labels=list(range(6)))

  if verbose:
    print('✅Bucket Banco generado con éxito, 🚼Distribución de Buckets:')
    print(baseDF['Bucket_Banco'].value_counts().sort_index())

  return baseDF

### **Paso 4: Definir el Bucket por Ahorro/Deuda**
___
En este caso, se verifica que la distribución que se existe del ratio entre **Dum_Bravo** y **Ahorro_Neto**, de modo que se tiene una aproximación de ahorro por cliente de forma normalizada.

In [ ]:
def calcBucketRatio(baseDF: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
  # Creamos una copia del DF
  baseDF = baseDF.copy()
  # Creamos la Columna Bucket_Ratio como Ahorro_Neto / Dum_Bravo
  baseDF['Bucket_Ratio'] = baseDF['Ahorro_Neto'] / baseDF['Dum_Bravo']

  # Como el Bucket es por Cliente, dejamos el Máximo Ratio por Referencia
  baseDF['Bucket_Ratio'] = baseDF.groupby('Referencia')['Bucket_Ratio'].transform('max')

  # Ahora Creamos el Bucket_Ratio dividiendo en 6 buckets de forma equitativa
  baseDF['Bucket_Ratio'] = pd.qcut(baseDF['Bucket_Ratio'].rank(method='first'), q=6, labels=list(range(6)))

  # Por Último, como a mayor el Ratio mejor, le damos la vuelta al Bucket_Ratio para que 5 sea el peor y 0 el mejor
  baseDF['Bucket_Ratio'] = 5 - baseDF['Bucket_Ratio'].astype(int)

  return baseDF

### **Paso 5: Crear Score de Cliente**
___
El Score del Cliente combina los 3 pasos anteriores en uno solo de modo que se defina el **Score del Cliente** que luego será distribuido por Bucket según la distribución de los Clientes del Negociador. El score del cliente es:
___
$$Score\_Cliente =
(BUCKET\_PESO\_BASE * Bucket\_Base) + \\
(BUCKET\_PESO\_BANCO * Bucket\_Banco) + \\
(BUCKET\_PESO\_RATIO * Bucket\_Ratio)
$$
___

In [ ]:
def calcScoreCliente(baseDF: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
  # Creamos una copia
  baseDF = baseDF.copy()
  # Creamos el Score_Cliente
  baseDF['Score_Cliente'] = (baseDF['Bucket_Base'].astype(int) * BUCKET_PESO_BASE +
                              baseDF['Bucket_Banco'].astype(int) * BUCKET_PESO_BANCO +
                              baseDF['Bucket_Ratio'].astype(int) * BUCKET_PESO_RATIO)
  return baseDF

### **Paso 6: Definir atención de Clientes por Negociadores**
___
Dado que un Cliente por x o y razón puede tener múltiples Negociadores, se define quién atiende el Cliente de verdad bajo la lógica de **la mayor cantidad de Deudas asignadas** por cada Negociador.

In [ ]:
def stablishRefNegs(baseDF: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
  # Creamos una copia del DF
  baseDF = baseDF.copy()
  # Se desea: Dejar el Negociador con más deudas del cliente por Cliente
  # 1. Obtener el Conteo de Negociador por Referencia
  infoNegosRefs = baseDF.groupby(['Referencia','Negociador_Asignado'], as_index=False).agg(Conteo=('Id_Deuda','count'))
  # 2. Obtener el Negociador_Asignado con más deudas por Referencia
  infoNegosRefs = infoNegosRefs.sort_values(by=['Referencia','Conteo'], ascending=False).groupby('Referencia').first().reset_index()
  # 3. Crear la Columna Negociador_Principal en el DataFrame Original asignando el Negociador_Asignado con más deudas por Referencia (se deja left por si acaso)
  baseDF = baseDF.merge(infoNegosRefs[['Referencia','Negociador_Asignado']].rename(columns={'Negociador_Asignado': 'Negociador_Principal'}), on='Referencia', how='left')

  if verbose:
    print('✅Negociador Principal asignado con éxito, 🚯Negociadores sin asignar:', baseDF['Negociador_Principal'].isna().sum())

  return baseDF

### **Paso 7: Aplicar la Distribución del Bucket**
___
Este ejercicio consta de 3 Pasos:
1. **Crear Tabla Resumen por Cliente**, para tener a la Mano de forma muy fácil los Diferentes Scores Necesarios
2. **Definir Tamaños de Bucket**: Se Consolidan los tamaños por Negociador según la cantidad de Clientes que tengan como *Inatendibles*
2. **Distribuir Clientes en Bucket**: Se utiliza una distribución ya predefinida acerca de la proporción de clientes por cada Bucket, de modo que no existan quejas acerca de la cantidad de clientes
3. **Perfilar Clientes Raros**: Los Clientes que no hayan sido distribuidos en el Bucket por X o Y razón son perfilados al Bucket Designado según la comparación entre su Distribución con el Global

In [ ]:
def stablishFinalBucket(baseDF: pd.DataFrame, refsBucket5, verbose: bool = True):
  # Creamos una copia del DF
  baseDF = baseDF.copy()
  # 1. Creamos una tabla resumen por Referencia y Negociador_Principal contando el número de deudas, junto con el Score del Cliente
  resumenNegociadores = baseDF.groupby(['Referencia','Negociador_Principal'], as_index=False, group_keys=False).agg(
      Conteo_Deudas=('Id_Deuda','count'),
      Score_Cliente=('Score_Cliente','first'),
  )

  # Creamos Función Auxiliar para aplicar la lógica de asignación de Prioridad (Bucket_MEC) a las Referencias por Negociador
  def asignarPrioridad(group: pd.DataFrame) -> pd.DataFrame:
    # Creamos una copia del grupo para no modificar el original
    copyGroup = group.copy()
    # 2. --- Definir Tamaños de Bucket ---
    # Definimos la distribución en Número de los Clientes para el Negociador usando DISTRIBUCION_BUCKETS como Guía
    estimatedProps = [np.floor(len(copyGroup) * prop) for prop in DISTRIBUCION_BUCKETS]
    # Definimos la Cantidad de Clientes Faltantes para Asignar (debido a los redondeos)
    faltantes = len(copyGroup) - sum(estimatedProps)
    # Repartimos los Faltantes de forma Equitativa
    estimatedProps = [prop + (1 if i < faltantes else 0) for i, prop in enumerate(estimatedProps)]

    # 3. --- Asignar Bucket_MEC ---
    # Creamos Bucket_MEC Inicializando con NaN
    copyGroup['Bucket_MEC'] = np.nan
    # Ahora: Referencias en refsBucket5 se asignan directamente a Bucket_MEC 5
    copyGroup.loc[copyGroup['Referencia'].isin(refsBucket5), 'Bucket_MEC'] = 5

    # 3.0 Ajustar las Distribuciones según los Buckets Obligatorios
    # Ahora Verificamos si se cumple o no la Distribución prevista del Bucket 5
    nBucket5 = (copyGroup['Bucket_MEC'] == 5).sum()
    if nBucket5 > estimatedProps[5]:
      # Vamos a Reajustar las Proporciones
      # 1. La Proporción de Bucket 5 se actualiza a la Proporción Real
      extraBucket5 = nBucket5 - estimatedProps[5]
      estimatedProps[5] = nBucket5
      # 2. Las Proporciones de los Buckets 0-4 se recalculan recortando los extras desde bucket 4 hacia abajo
      for i in range(4, -1, -1):
        if extraBucket5 <= 0:
          break
        maxDelete = min(extraBucket5, estimatedProps[i])
        estimatedProps[i] -= maxDelete
        extraBucket5 -= maxDelete
    else:
      # Si el Bucket 5 no se supera, se ajusta en función de la distribución prevista y el extra se deja en Bucket 4
      extraBucket5 = estimatedProps[5] - nBucket5
      estimatedProps[5] = nBucket5
      estimatedProps[4] += extraBucket5

    # 3.1 Distribuir el Resto con la Distribución Establecida donde Bucket Bajo es Mejor Score
    # Ordenamos los Datos por Score_Cliente de Mejor a Peor
    copyGroup = copyGroup.sort_values(by=['Bucket_MEC','Score_Cliente'],na_position='first').reset_index(drop=True)
    # Asignamos los Buckets según la Distribución Establecida
    asignados = 0
    for i in range(6):
      asignarBloque = estimatedProps[i]
      if asignarBloque <= 0:
        continue
      finalBloque = asignados + asignarBloque
      copyGroup.loc[asignados:finalBloque-1, 'Bucket_MEC'] = i
      asignados = finalBloque

    # 4. --- Imputación de Buckets que no se asignaron ---
    # En Dado caso donde Bucket_MEC siga siendo NaN, se asigna aplicando distribución equitativa de la Score
    distEquitativa = pd.qcut(copyGroup['Score_Cliente'].rank(method='first',na_option='bottom'), q=6, duplicates='drop',labels=False).fillna(5)
    copyGroup['Bucket_MEC'] = np.where(copyGroup['Bucket_MEC'].isna(), distEquitativa.astype(int), copyGroup['Bucket_MEC'])

    # Devolvemos el Grupo con la Nueva Columna Bucket_MEC
    return copyGroup

  # Aplicamos la Función de Asignación de Prioridad a cada Negociador
  inicio = time()
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=False).apply(asignarPrioridad).reset_index(drop=True)
  fin = time()

  # Por Último, unimos Base_Bucket y baseDF por Referencia
  baseDF = baseDF.merge(baseBucket[['Referencia','Bucket_MEC']], on='Referencia', how='left')

  if verbose:
    print('⏱️Tiempo de Asignación de Prioridad por Negociador:', round(fin - inicio, 2), 'segundos')
    print('✅Asignación de Prioridad por Negociador realizada con éxito, 🚯Clientes sin Bucket: {0}'.format(baseDF['Bucket_MEC'].isna().sum()))
    print('ℹ️Distribución de Bucket_MEC:')
    print(baseBucket['Bucket_MEC'].value_counts().sort_index())

  return baseDF

# **Orquestador de Información**
___
Se define la Función principal que es la que va a permitir generar toda la ejecución de los cálculos de Bucket Cartera. La Función es **orquestaDora** y recibe dos Inputs:
- **Modo**: El Modo de Ejecución de los Datos (**BM** para mejor mes o **MEC** para actualizar el mes en curso)
- **Fecha_Lim**: La Fecha Limite para tomar las Actualizaciones (Solo aplica para Funnel)
Esta Función aplica todas las Funciones anteriores de modo que sea utilizable muchas veces.

In [ ]:
def orquestaDora(mode: Literal['BM','MEC'], Fecha_Lim: pd.Timestamp = pd.Timestamp(1999,1,1)):
  # Paso 1: Obtener las Actualizaciones
  actsLocal = getActsInfo(mode, Fecha_Lim)
  # Paso 2: Union de Datos
  localDF = uniteInfo(mode,actsLocal, Fecha_Lim, verbose = False)
  # Paso 3: Cálculos Generales
  localDF = makeGeneralCalcs(localDF, verbose = False)

  # Paso 4: --- Calculos para Tradicional ---
  # Paso 4.1 Inviabilidades
  localDF = establishNV(localDF, verbose = False)
  # Paso 4.2: Preparacion de Datos
  localDF = prepareTrad(localDF, verbose = False)
  # Paso 4.3: Desgaste de Ahorro
  localDF = makeDesgaste(localDF, verbose = False)
  # Paso 4.4: Potencial Tradicional
  localDF = calcPotTrad(localDF, verbose = False)
  # Paso 4.5: Limpieza de Cálculos
  localDF = cleanCalcs(localDF, verbose = False)

  # Paso 5: --- Reajuste del Bucket ---
  # Paso 5.1: Definir el Bucket Base
  localDF = calcInitBucket(localDF, verbose = False)
  # Paso 5.2: Calcular Bucket 5 Ob
  refsBucket5 = calcBucket5(localDF, verbose = False)
  # Paso 5.3: Calcular Bucket Banco
  localDF = calcBucketBank(localDF, verbose = False)
  # Paso 5.4: Calcular Bucket Ratio
  localDF = calcBucketRatio(localDF, verbose = False)
  # Paso 5.5: Calcular Score Cliente
  localDF = calcScoreCliente(localDF, verbose = False)
  # Paso 5.6: Definir Negociador Principal
  localDF = stablishRefNegs(localDF, verbose = False)
  # Paso 5.7: Calcular el Bucket Final
  localDF = stablishFinalBucket(localDF, refsBucket5, verbose = False)

  # Devolvemos el DF
  return localDF

# **Generación de Actualización de Mes Actual**
___
Utilizando el Orquestador se aplica la Lógica de:


___
**Antigua: Desde la Última Actualización en el Timeline** se buscan las nuevas actualizaciones y se van haciendo los reajustes de los buckets
___
**Nueva: Se busca por Id y Fecha las Actualizaciones Faltantes**, y estas se agregan al Timeline

## **Obtención de Actualizaciones Faltantes**
___
Se obtienen los Datos como un DataFrame para obtener la Última Fecha de Actualización si existe, de lo contrario se crea desde **la Primera Actualización del Año**

In [ ]:
# Funcion Auxiliar para obtener worksheet con objeto sheet
def getWorksheet(sh, wsName, colLen):
  try:
    return _retry(lambda: (True, sh.worksheet(wsName)))
  except:
    return _retry(lambda: (False, sh.add_worksheet(wsName, 1000, colLen)))

# Abrimos la Hoja del Bucket
bucketSH = _retry(lambda: client.open_by_key(PIPACT_SHEET_ID))
# Abrimos la Hoja de 'Mes_Actual'
existed, tmWS = getWorksheet(bucketSH, 'Mes_Actual', 30)

# Definimos lastUpdate como None
tmWasEmpty = False

# --> Copia de Actualizaciones: Dejar la Ultima antes de startDate y el resto todo
# Paso 1: Crear Copia de Actualizaciones
copactDF = actsDF.copy()
# Paso 2: Dejar solo Deudas de la Cartera Asignada
copactDF = copactDF[copactDF['Id_Deuda'].isin(clientesDF['Id_Deuda'])]

# Ahora Definimos un Valor por Defecto de idsNecessaryUpds como lista de tuplas de (Id_Deuda, Fecha_Act)
defaultValUpd = set(zip(copactDF['Id_Deuda'], copactDF['Fecha_Act']))

if (not existed) or REESCRITURA_TOTAL: # SI no existia o esta vacia entonces se define como la Fecha minima en actsDF - 1 dia
  idsNecessaryUpds = defaultValUpd
  idsDF = copactDF.copy()
  updDF = pd.DataFrame(columns=['Id_Deuda','Fecha_Act'])
elif data:= isWSEmpty(tmWS):
  tmWasEmpty = data
  idsNecessaryUpds = defaultValUpd
  updDF = pd.DataFrame(columns=['Id_Deuda','Fecha_Act'])
  idsDF = copactDF.copy()
else:
  # Vamos a Obtener los Headers actuales
  headers = _retry(lambda: tmWS.row_values(1))

  # Definimos el Nombre de Columna a Buscar
  updCols = ['Fecha_Act', 'inserted_at', 'inserted_at_ultima', 'Fecha_Act_ultima']
  updateIdx = None
  idDebtCols = ['Id_Deuda','Id deuda','Id Deuda','ID Deuda','ID deuda']
  idDebtIdx = None
  # Buscamos el Índice de la Columna
  for i, col in enumerate(headers,start=1):
    # Buscamos los Índices para ambas Columnas
    if col.strip() in updCols:
      updateIdx = i
    if col.strip() in idDebtCols:
      idDebtIdx = i

    if updateIdx is not None and idDebtIdx is not None:
      break

  # Verificamos que tengamos ambos Ids
  if (updateIdx is None) or (idDebtIdx is None):
    # Vamos a Definir los idsNecessaryUpds como el valor por defecto
    idsNecessaryUpds = defaultValUpd
    updDF = pd.DataFrame(columns=['Id_Deuda','Fecha_Act'])
    idsDF = copactDF.copy()
  else:
    print('😣Obteniendo los Rangos Necesarios')
    # Obtenemos los Datos para Dichas Columnas
    idsCols = _retry(lambda: tmWS.col_values(idDebtIdx))[1:]
    updCols = _retry(lambda: tmWS.col_values(updateIdx))[1:]

    # Ajustamos los updCols añadiendo None al final si es necesario
    updCols += [None] * (len(idsCols)-len(updCols))

    # Creamos un DataFrame con los Datos
    updDF = pd.DataFrame({
        'Id_Deuda': idsCols,
        'Fecha_Act': updCols
    })

    # Volvemos la Columna Fecha_Act a Datetime con dayfirst = False
    updDF['Fecha_Act'] = pd.to_datetime(updDF['Fecha_Act'], dayfirst=False)

    # Volvemos la Columna Id_Deuda a String
    updDF['Id_Deuda'] = updDF['Id_Deuda'].apply(lambda s: str(s).replace('.0','').strip()).astype(str)

    # Ahora creamos de la misma forma el set de antes
    auxNSUpds = set(zip(updDF['Id_Deuda'],updDF['Fecha_Act']))

    # Ahora Definimos idsNecessaryUpds como la Diferencia entre todas las de Actualizaciones y las recien traidas
    idsNecessaryUpds = defaultValUpd - auxNSUpds

    # Volvemos idsNecessaryUpds a DF
    idsDF = pd.DataFrame(idsNecessaryUpds, columns=['Id_Deuda','Fecha_Act'])

print('🥸Hay <{}> Actualizaciones Faltantes por Subir'.format(
    len(idsNecessaryUpds)
))

[RETRY 1/10]  -> APIError: [429]: Quota exceeded for quota metric 'Read requests' and limit 'Read requests per minute per user' of servic... sleep 1.5s
[RETRY 2/10]  -> APIError: [429]: Quota exceeded for quota metric 'Read requests' and limit 'Read requests per minute per user' of servic... sleep 3.0s
[RETRY 3/10]  -> APIError: [429]: Quota exceeded for quota metric 'Read requests' and limit 'Read requests per minute per user' of servic... sleep 6.5s
[RETRY 4/10]  -> APIError: [429]: Quota exceeded for quota metric 'Read requests' and limit 'Read requests per minute per user' of servic... sleep 12.4s
[RETRY 5/10]  -> APIError: [429]: Quota exceeded for quota metric 'Read requests' and limit 'Read requests per minute per user' of servic... sleep 24.4s
😣Obteniendo los Rangos Necesarios
🥸Hay <1070> Actualizaciones Faltantes por Subir


## **Función Auxiliar de Batch Único**
___
Dado que para crear nuestro timeline tenemos que contar todas las actualizaciones pero a la vez no permitir duplicados tenemos que crear **batches individuales**, donde maximicemos la cantidad de cálculos por batch y a la vez evitando duplicados.

In [ ]:
def get_non_duplicate_dates(df, id_col, date_col, search_start):
    """
    Finds limit dates to create batches where no duplicate IDs exist.
    Uses a Date -> Set(IDs) map approach to resolve intra-timestamp
    sorting issues and consecutive duplicate blindspots.
    """

    if pd.isna(search_start):
        return [search_start]
    # 1. Standardize types and filter from search start
    df_copy = df.copy()
    df_copy[date_col] = pd.to_datetime(df_copy[date_col])
    search_start = pd.to_datetime(search_start)

    df_filtered = df_copy[df_copy[date_col] >= search_start]
    if df_filtered.empty:
        return [search_start]

    # 2. Create the Map: Date -> Set of IDs (Sorted chronologically)
    # This automatically deduplicates identical IDs sharing the exact same timestamp
    date_map = df_filtered.groupby(date_col)[id_col].apply(set).sort_index()

    batch_dates = [search_start]
    seen_ids = set()

    # 3. Step through time atomically
    for current_date, ids_at_timestamp in date_map.items():

        # Check if any ID at this timestamp has already been seen in the current batch window
        if ids_at_timestamp.intersection(seen_ids):

            # Duplicate found across the timeline! Create our exclusive upper limit (<)
            if current_date > batch_dates[-1]:
                batch_dates.append(current_date)

            # Reset the batch tracking: The new batch starts AT this timestamp,
            # so its initial memory consists of all IDs active at this timestamp.
            seen_ids = ids_at_timestamp.copy()

        else:
            # No duplicates found. Absorb these IDs into the current batch's memory
            seen_ids.update(ids_at_timestamp)

    # 4. Handle the final max date to close the last batch window
    max_date = df_filtered[date_col].max()
    if max_date > batch_dates[-1]:
        batch_dates.append(max_date)
    else:
        batch_dates.append(max_date + pd.Timedelta(seconds=1))

    return batch_dates

In [ ]:
fechasBatch = get_non_duplicate_dates(idsDF, 'Id_Deuda','Fecha_Act',idsDF['Fecha_Act'].min())
# Iterate through the boundaries to generate your timeline batches
failedBatch = 0
for i in range(len(fechasBatch) - 1):
    start_date = fechasBatch[i]
    end_date = fechasBatch[i+1]

    # Apply exclusive upper limit (<), except for the absolute last batch which includes the max date
    if i == len(fechasBatch) - 2:
        batch_df = idsDF[(idsDF['Fecha_Act'] >= start_date) & (idsDF['Fecha_Act'] <= end_date)]
    else:
        batch_df = idsDF[(idsDF['Fecha_Act'] >= start_date) & (idsDF['Fecha_Act'] < end_date)]

    # --- Run your calculations safely on batch_df here ---
    #print(f"\nBatch {i+1}: From {start_date} to {end_date} (Rows: {len(batch_df)})")

    # Verificamos que no haya duplicados
    if batch_df['Id_Deuda'].duplicated().any():
      print('🚯Duplicates for Batch {}. Dates between: {}-{}'.format(
          i+1,
          start_date.strftime('%Y-%m-%d %H:%M:%S'),
          end_date.strftime('%Y-%m-%d %H:%M:%S')
      ))
      print(batch_df[batch_df['Id_Deuda'].duplicated(keep=False)][['Fecha_Act','Id_Deuda']])
      failedBatch += 1
    else:
      pass
      # print('✅Correct Batch {}'.format(
      #     i+1
      # ))
if failedBatch > 0:
  raise ValueError('🚯Hay un Error con las Actualizaciones')
print('😁Los Batch para actualizar estan funcionando Correctamente')
print('✅Batch Totales: {}'.format(len(fechasBatch)))
print('ℹ️Tiempo Estimado: {:.2f} minutos'.format(
    len(fechasBatch) / 12
))

😁Los Batch para actualizar estan funcionando Correctamente
✅Batch Totales: 14
ℹ️Tiempo Estimado: 1.17 minutos


# **Actualizar Bloques MEC**

___
Una vez se obtiene la Fecha desde la cúal reconstruir, se procede a ejecutar la reconstrucción basado en la siguiente lógica:

Para definir los cambios de la deuda tras la actualización es necesario re-ejecutar el orquestador por cada actualización. Sin embargo, esto acaba siendo ineficiente ya que existen muchas actualizaciones que se pueden consolidar en una sola ejecución. Entonces la lógica sería:

**Ejecutar actualizaciones por Batch**, el batch se define por la siguiente fecha en la que existe más una actualización para un Id_Deuda.

In [ ]:
# Definimos la Lista para guardar los Timelines
tmDFList = []

print('🥷Ejecutando Actualización para <{}> Batches\n'.format(
    len(fechasBatch)
))

for i in range(len(fechasBatch) - 1):
  startW = fechasBatch[i]
  endW = fechasBatch[i+1]

  # Si es la Última se agrega un día
  if i == len(fechasBatch) - 2:
    endW += pd.Timedelta(days=1)

  # Definimos las Deudas que se van a actualizar
  deudasUpd = idsDF[(idsDF['Fecha_Act'] >= startW) & (idsDF['Fecha_Act'] < endW)]['Id_Deuda'].unique()

  # Obtenemos los Datos del Funnel para endW en modo MEC
  tmDF = orquestaDora(mode='MEC', Fecha_Lim=endW)

  # Ahora Filtramos solo las Deudas que necesitabamos
  tmDF = tmDF[tmDF['Id_Deuda'].isin(deudasUpd)]

  # Lo Agregamos a la Lista
  tmDFList.append(tmDF)

  if i%3 == 0:
    print('- ✅Batch hasta {}: {} Procesado, 🫡Deudas Actualizadas: {} | ℹ️Progreso: {:.2%}'.format(
        endW.strftime('%Y-%m-%d %H:%M:%S'),
        str(i+1).ljust(3,' '),
        str(len(deudasUpd)).ljust(5,' '),
        i/len(fechasBatch)
    ).center(50,'-'))

# Ahora Definimos el DF final
if tmDFList:
  newTMDF = pd.concat(tmDFList, ignore_index=True)
else:
  newTMDF = pd.DataFrame(columns=['Referencia','Id_Deuda','Fecha_Act'])

# Ordenamos los Datos por Referencia, Id_Deuda y Fecha_Act
newTMDF = newTMDF.sort_values(by=['Referencia','Id_Deuda','Fecha_Act'], ignore_index=True)

print('\n\n\n✅Timeline a Actualizar Generado con Éxito')
print('ℹ️Datos a Actualizar: {}'.format(len(newTMDF)))
print('🆔Ids que Recibieron actualización: {}'.format(
    newTMDF['Id_Deuda'].nunique()
))

🥷Ejecutando Actualización para <14> Batches



/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=False).apply(asignarPrioridad).reset_index(drop=True)


- ✅Batch hasta 2026-11-17 10:19:42: 1   Procesado, 🫡Deudas Actualizadas: 29    | ℹ️Progreso: 0.00%


/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=False).apply(asignarPrioridad).reset_index(drop=True)
/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=Fals

- ✅Batch hasta 2026-11-17 10:32:07: 4   Procesado, 🫡Deudas Actualizadas: 674   | ℹ️Progreso: 21.43%


/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=False).apply(asignarPrioridad).reset_index(drop=True)
/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=Fals

... (2 outputs left out)

- ✅Batch hasta 2026-11-17 11:18:37: 10  Procesado, 🫡Deudas Actualizadas: 21    | ℹ️Progreso: 64.29%


/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=False).apply(asignarPrioridad).reset_index(drop=True)
/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=Fals

- ✅Batch hasta 2026-11-18 11:53:32: 13  Procesado, 🫡Deudas Actualizadas: 27    | ℹ️Progreso: 85.71%



✅Timeline a Actualizar Generado con Éxito
ℹ️Datos a Actualizar: 1070
🆔Ids que Recibieron actualización: 982


/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=False).apply(asignarPrioridad).reset_index(drop=True)


# **Actualizar Bloques Mejor Mes**
___
En caso de ser necesario, se procede a crear la construcción del mejor mes de la misma forma que el mes Actual.

In [ ]:
# 1: Verificar que se pueda generar
if not(FECHA_MEJOR_MES is None):
  # Definimos las Actualizaciones a Obtener
  # Paso 1: Crear Copia de Actualizaciones
  idsBMDF = actsBMDF.copy()
  # Paso 2: Dejar solo Deudas de la Cartera Asignada
  idsBMDF = idsBMDF[idsBMDF['Id_Deuda'].isin(clientesBMDF['Id_Deuda'])]
  # Paso 3: Dejar solo columnas Fecha_Act e Id_Deuda
  idsBMDF = idsBMDF[['Fecha_Act','Id_Deuda']]

  # --- Definimos los Batches a usar ---
  fechasBMBatch = get_non_duplicate_dates(idsBMDF, 'Id_Deuda','Fecha_Act',idsBMDF['Fecha_Act'].min())
  # Iterate through the boundaries to generate your timeline batches
  failedBatch = 0
  for i in range(len(fechasBMBatch) - 1):
      start_date = fechasBMBatch[i]
      end_date = fechasBMBatch[i+1]

      # Apply exclusive upper limit (<), except for the absolute last batch which includes the max date
      if i == len(fechasBMBatch) - 2:
          batch_df = idsBMDF[(idsBMDF['Fecha_Act'] >= start_date) & (idsBMDF['Fecha_Act'] <= end_date)]
      else:
          batch_df = idsBMDF[(idsBMDF['Fecha_Act'] >= start_date) & (idsBMDF['Fecha_Act'] < end_date)]

      # --- Run your calculations safely on batch_df here ---
      #print(f"\nBatch {i+1}: From {start_date} to {end_date} (Rows: {len(batch_df)})")

      # Verificamos que no haya duplicados
      if batch_df['Id_Deuda'].duplicated().any():
        print('🚯Duplicates for Batch {}. Dates between: {}-{}'.format(
            i+1,
            start_date.strftime('%Y-%m-%d %H:%M:%S'),
            end_date.strftime('%Y-%m-%d %H:%M:%S')
        ))
        print(batch_df[batch_df['Id_Deuda'].duplicated(keep=False)][['Fecha_Act','Id_Deuda']])
        failedBatch += 1
      else:
        pass
        # print('✅Correct Batch {}'.format(
        #     i+1
        # ))
  if failedBatch > 0:
    raise ValueError('🚯Hay un Error con las Actualizaciones')
  print('😁Los Batch para actualizar estan funcionando Correctamente')
  print('✅Batch Totales: {}'.format(len(fechasBMBatch)))
  print('ℹ️Tiempo Estimado: {:.2f} minutos\n'.format(
      len(fechasBMBatch) / 12
  ))

  # --- Ejecutamos la Búsqueda para Estos Meses
  # Definimos la Lista para guardar los Timelines
  bmDFList = []

  print('🥷Ejecutando Actualización para <{}> Batches\n'.format(
      len(fechasBMBatch)
  ))

  for i in range(len(fechasBMBatch) - 1):
    startW = fechasBMBatch[i]
    endW = fechasBMBatch[i+1]

    # Si es la Última se agrega un día
    if i == len(fechasBMBatch) - 2:
      endW += pd.Timedelta(days=1)

    # Definimos las Deudas que se van a actualizar
    deudasUpd = idsBMDF[(idsBMDF['Fecha_Act'] >= startW) & (idsBMDF['Fecha_Act'] < endW)]['Id_Deuda'].unique()

    # Obtenemos los Datos del Funnel para endW en modo BM
    bmDF = orquestaDora(mode='BM', Fecha_Lim=endW)

    # Ahora Filtramos solo las Deudas que necesitabamos
    bmDF = bmDF[bmDF['Id_Deuda'].isin(deudasUpd)]

    # Lo Agregamos a la Lista
    bmDFList.append(bmDF)

    if i%3 == 0:
      print('- ✅Batch hasta {}: {} Procesado, 🫡Deudas Actualizadas: {} | ℹ️Progreso: {:.2%}'.format(
          endW.strftime('%Y-%m-%d %H:%M:%S'),
          str(i+1).ljust(3,' '),
          str(len(deudasUpd)).ljust(5,' '),
          i/len(fechasBMBatch)
      ).center(50,'-'))

  # Ahora Definimos el DF final
  if bmDFList:
    bmDF = pd.concat(bmDFList, ignore_index=True)
  else:
    bmDF = pd.DataFrame()

  print('\n\n\n✅Timeline a Actualizar Generado con Éxito')
  print('ℹ️Datos a Actualizar: {}'.format(len(bmDF)))
  print('🆔Ids que Recibieron actualización: {}'.format(
      bmDF['Id_Deuda'].nunique()
  ))
else:
  bmDF = pd.DataFrame()
  print('😁No es Necesario Generar el Mejor Mes')

😁Los Batch para actualizar estan funcionando Correctamente
✅Batch Totales: 642
ℹ️Tiempo Estimado: 53.50 minutos

🥷Ejecutando Actualización para <642> Batches



/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=False).apply(asignarPrioridad).reset_index(drop=True)


- ✅Batch hasta 2026-06-02 09:13:31: 1   Procesado, 🫡Deudas Actualizadas: 104   | ℹ️Progreso: 0.00%


/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=False).apply(asignarPrioridad).reset_index(drop=True)
/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=Fals

- ✅Batch hasta 2026-06-02 10:02:26: 4   Procesado, 🫡Deudas Actualizadas: 51    | ℹ️Progreso: 0.47%


/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=False).apply(asignarPrioridad).reset_index(drop=True)
/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=Fals

... (421 outputs left out)

/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=False).apply(asignarPrioridad).reset_index(drop=True)
/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=Fals

- ✅Batch hasta 2026-07-02 19:24:57: 640 Procesado, 🫡Deudas Actualizadas: 26    | ℹ️Progreso: 99.53%


/tmp/ipykernel_2475/3196805499.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  baseBucket = resumenNegociadores.groupby('Negociador_Principal', as_index=False, group_keys=False).apply(asignarPrioridad).reset_index(drop=True)





✅Timeline a Actualizar Generado con Éxito
ℹ️Datos a Actualizar: 27747
🆔Ids que Recibieron actualización: 12353


# **Subida de Datos**
___
Se procede a subir la Información en **Bucket Cartera** para que quede a disposición de quien lo necesite

## **Funciones Auxiliares de Subida de Datos**
___
Son Funciones Robustas que permiten la actualización de los Datos y la subida de estos de una forma segura y sin errores

In [ ]:
# Función Auxiliar para subir los datos a una hoja
def uploadToSheets(ws, df, retry_label="Upload Data"):
  """
  Uploads a DataFrame to a Google Sheet, ensuring column alignment
  and cleaning up unnecessary columns.
  """
  _retry(lambda: ws.clear(), label="clear sheet")
  _retry(lambda: set_with_dataframe(ws, df), label=f"{retry_label} ({len(df)} rows)")

def batch_update_column_a_from_dict(ws, data_dict: dict[str, int], cell_threshold: int = 10000):
  """
  Transforms a {Value: Row} dictionary to update Column A using existing batch functions.
  Consolidates consecutive rows into single blocks for maximum efficiency.
  """
  if not data_dict:
      return

  # 1. Invert the dict to {Row: Value} and sort by Row Number
  # Note: If a value was duplicated in your original dict,
  # this ensures the row mapping is handled correctly.
  nodes = sorted((row, val) for val, row in data_dict.items())

  row_blocks = []

  # Initialize the first block
  start_row, first_val = nodes[0]
  current_end_row = start_row
  current_matrix = [[first_val]]

  # 2. Group consecutive rows together into blocks
  for row, val in nodes[1:]:
    if row == current_end_row + 1:
      # It's consecutive! Append to the current block matrix
      current_matrix.append([val])
      current_end_row = row
    else:
      # Gap detected. Save the completed block and start a new one
      row_blocks.append((start_row, current_end_row, current_matrix))
      start_row = row
      current_end_row = row
      current_matrix = [[val]]

  # Don't forget to append the final block
  row_blocks.append((start_row, current_end_row, current_matrix))

  # 3. Feed it straight into your existing function
  _batch_update_rows(
      ws=ws,
      start_col_letter="A",
      end_col_letter="A",
      row_blocks=row_blocks,
      cell_threshold=cell_threshold
  )

# Funcion Auxiliar para comparar DF y SheetRows cambiando columnas tipo DateTime
def _prepare_df_for_sheets(df: pd.DataFrame) -> pd.DataFrame:
    df_out = df.copy()
    df_out = df_out.replace([np.inf, -np.inf], np.nan)

    # Si alguna columna es DateTime la volvemos string
    for c in df_out.columns:
        if pd.api.types.is_datetime64_any_dtype(df_out[c]):
            # datetime64[ns] y datetime64[ns, tz]
            df_out[c] = pd.to_datetime(df_out[c], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")

    # Cambiamos NaNs por valores vacios
    df_out = df_out.where(pd.notna(df_out), "")
    return df_out

# Función Auxiliar para compara DF y SheetRows convirtiendo las columnas en strings
def _df_to_str_matrix(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Normaliza para comparar contra Sheets (que casi siempre devuelve strings).
    """
    out = df[cols].copy()
    for c in cols:
        # todo como string comparable; "" para vacíos
        out[c] = out[c].astype(str)
        out.loc[out[c].isin(["nan", "NaT", "<NA>", "None"]), c] = ""
    return out

def _batch_update_rows(ws, start_col_letter: str, end_col_letter: str, row_blocks: list[tuple[int,int,list[list[str]]]], cell_threshold: int = 10000):
    """
    Updates Google Sheets using batch_update to minimize API calls.
    Groups row_blocks into 'mega-batches' based on a cell_threshold.
    """

    current_batch_data = []
    current_cell_count = 0

    for (r1, r2, mat) in row_blocks:
        # Calculate cells in this specific block
        block_cells = len(mat) * len(mat[0]) if mat else 0
        rng = f"{start_col_letter}{r1}:{end_col_letter}{r2}"

        # Prepare the update object for this block
        update_item = {
            'range': rng,
            'values': mat
        }

        # Check if adding this block exceeds our threshold
        if current_cell_count + block_cells > cell_threshold and current_batch_data:
            # Execute the accumulated batch before starting a new one
            _execute_batch_retry(ws, current_batch_data)
            current_batch_data = []
            current_cell_count = 0
            sleep(0.5) # Slight breather between mega-batches

        current_batch_data.append(update_item)
        current_cell_count += block_cells

    # Final execution for any remaining data
    if current_batch_data:
        _execute_batch_retry(ws, current_batch_data)

def _execute_batch_retry(ws, data_list):
    """
    Helper to wrap the batch_update in your retry logic.

    Importante:
    gspread modifica cada diccionario recibido en batch_update()
    agregando el nombre de la hoja al campo 'range'. Como _retry()
    puede ejecutar varias veces la misma función, debemos enviar una
    copia nueva de los diccionarios en cada intento para evitar que
    el nombre de la hoja se concatene repetidamente.
    """

    def _send_batch():
        # Copia superficial suficiente: gspread modifica únicamente
        # el valor del campo 'range' del diccionario.
        payload = [dict(item) for item in data_list]

        return ws.batch_update(
            payload,
            value_input_option="USER_ENTERED"
        )

    _retry(
        _send_batch,
        label=f"batch_update for {len(data_list)} ranges"
    )

# Funcion auxiliar para unir filas actualizadas en una sola y poder realizar cambios enteros por chunks
def _make_consecutive_blocks(rownums_sorted: list[int], values_by_rownum: dict[int, list[str]]):
    """
    Agrupa filas consecutivas para reducir llamadas a la API.
    Retorna [(start_row, end_row, matrix_values)]
    """
    blocks = []
    if not rownums_sorted:
        return blocks

    start = prev = rownums_sorted[0]
    mat = [values_by_rownum[start]]

    for r in rownums_sorted[1:]:
        # Si la fila es adyacente a la anterior se uno como un bloque
        if r == prev + 1:
            mat.append(values_by_rownum[r])
            prev = r
        else:
          # Si no, entonces se guarda el bloque y se crea uno nuevo
            blocks.append((start, prev, mat))
            start = prev = r
            mat = [values_by_rownum[r]]
    # Se guarda el último bloque en memoria
    blocks.append((start, prev, mat))
    return blocks

def applyChanges(ws, df: pd.DataFrame, identifierCol: str, numericCols = [], dateCols = [], semiStrCols = []) -> tuple[bool, str]:
  # Transformamos el DataFrame previo a la comparación
  dfOut = _prepare_df_for_sheets(df)

  # Construir DF de Sheets
  df_sheet = _retry(lambda: get_as_dataframe(ws, evaluate_formulas=True), label="fetch sheet data for comparison")
  sheet_header = df_sheet.columns.tolist()
  cols = dfOut.columns.tolist()

  # Ponemos los valores "" como NaNs
  df_sheet = df_sheet.replace("", np.nan)
  dfOut = dfOut.replace("", np.nan)

  # Si por cualquier razón Sheets no trae todas las columnas, nos quedamos con intersección
  missing_in_sheet = [c for c in cols if c not in df_sheet.columns]
  if missing_in_sheet:
      print("⚠️ En Sheets faltan estas columnas vs DF:", missing_in_sheet)
      print(sheet_header,cols)

      # Si son más de 10 Columnas, entonces se va a reescribir toda la información del Sheets
      if len(missing_in_sheet) > 10:
          print("⚠️ Más de 10 columnas faltantes, se va a reescribir toda la información del Sheets para evitar inconsistencias.")
          print('🆔Comenzando a Subir información a Sheets...')
          uploadToSheets(ws, dfOut, retry_label="Full Upload due to many missing columns")
          return True, f"✅ Completado: Se subió toda la información debido a la falta de {len(missing_in_sheet)} columnas en Sheets. Columnas faltantes: {missing_in_sheet}"

      # Igual seguimos, pero esas columnas las tratamos como "" al comparar
      for c in missing_in_sheet:
          df_sheet[c] = ""

  print('- ℹ️Aplicando Cambio de Columnas...')

  ##### Solución a Problema de comparar floats a int
  # La solucion consiste en convertir todas las variables numericas a float
  dfOut[identifierCol] = dfOut[identifierCol].apply(lambda s: str(s).replace('.0',''))
  df_sheet[identifierCol] = df_sheet[identifierCol].apply(lambda s: str(s).replace('.0',''))
  for c in numericCols:
    dfOut[c] = dfOut[c].astype(float).round(2)
    df_sheet[c] = df_sheet[c].astype(float).round(2)
    # Reemplazamos Infinitos con NaNs
    dfOut[c] = dfOut[c].replace([np.inf, -np.inf], np.nan)
    df_sheet[c] = df_sheet[c].replace([np.inf, -np.inf], np.nan)

  # Ahora se cambian tambien todas las variables tipo Datetime
  for c in dateCols:
    temp_py = pd.to_datetime(dfOut[c], errors='coerce')
    temp_sheet = pd.to_datetime(df_sheet[c], errors='coerce')
    # Se ponen las columnas como formato largo
    dfOut[c] = temp_py.dt.strftime("%Y-%m-%d %H:%M:%S").fillna("")
    df_sheet[c] = temp_sheet.dt.strftime("%Y-%m-%d %H:%M:%S").fillna("")

  # Ahora para las Columnas semiString, se vuelve todo a string y se quitan los .0 sobrantes
  for c in semiStrCols:
    dfOut[c] = dfOut[c].astype(str).apply(lambda s: str(s).replace('.0','').replace('nan','').replace('NaT','').replace('<NA>','').replace('None',''))
    df_sheet[c] = df_sheet[c].astype(str).apply(lambda s: str(s).replace('.0','').replace('nan','').replace('NaT','').replace('<NA>','').replace('None',''))

  # Volvemos a hacer fill de nans con ""
  dfOut = dfOut.fillna("")
  df_sheet = df_sheet.fillna("")

  print('- 🫡Cambio Aplicado, ℹ️Buscando Diferencias')

  # Map id -> row_number en Sheets (row 2 = primer dato)
  id_to_rownum = {}
  for i, v in enumerate(df_sheet[identifierCol].tolist(), start=2):
      if v != "":
          # si hay duplicados, nos quedamos con el primero (puedes cambiarlo si prefieres el último)
          if v not in id_to_rownum:
              id_to_rownum[v] = i

  # Convertimos el df local para comparar a solo strings
  df_py_cmp = _df_to_str_matrix(dfOut, cols)

  # Creamos sets para comparar la interseccion
  ids_sheet = set(id_to_rownum.keys())
  ids_py = set(df_py_cmp[identifierCol].tolist())

  # Diferenciamos Ids comunes de los ID que son nuevos en la SpreadSheet
  ids_common = ids_py.intersection(ids_sheet)
  ids_new = sorted(list(ids_py - ids_sheet))

  # Indexar para comparación rápida
  df_sheet_cmp = _df_to_str_matrix(df_sheet, cols)

  sheet_by_id = df_sheet_cmp.set_index(identifierCol, drop=False)
  py_by_id = df_py_cmp.set_index(identifierCol, drop=False)

  # Encontrar ids cambiados (comparando todas las columnas excepto Referencia)
  compare_cols = [c for c in cols if c != identifierCol]
  changed_ids = []
  for _id in ids_common:
      # si por algo no aparece, saltamos
      if _id not in sheet_by_id.index or _id not in py_by_id.index:
          continue
      a = sheet_by_id.loc[_id, compare_cols]
      b = py_by_id.loc[_id, compare_cols]
      # a y b son Series
      ##### Problema Fundamental: Cuando un valor cambia de int a float, se agrega como n.0 y se produce automaticamente un cambio

      if not a.equals(b):
          changed_ids.append(_id)
  # Preparar updates por row_number
  values_by_rownum = {}
  for _id in changed_ids:
    rownum = id_to_rownum.get(_id)
    if not rownum:
        continue
    row_values = py_by_id.loc[_id, cols].tolist() # type: ignore

    # Último Intento de Corregir Error JSON out of range
    # Convertimos NaNs inf o -inf en ''
    # 2. Convertimos y limpiamos de forma segura para JSON:
    sanitized_row = []
    for r in row_values:
      # Si es un valor nulo/nan/infinito de numpy o pandas, lo volvemos string vacío
      if pd.isna(r) or r in [np.inf, -np.inf]:
          sanitized_row.append('')
      # Si es un tipo entero/flotante de numpy, lo casteamos a primitivo de Python
      elif hasattr(r, 'item'):
          sanitized_row.append(r.item())
      else:
          sanitized_row.append(r)

      values_by_rownum[rownum] = sanitized_row

  rownums_sorted = sorted(values_by_rownum.keys())

  start_col_letter = "A"
  end_col_letter = gspread.utils.rowcol_to_a1(1, len(df.columns)).replace("1", "")
  blocks = _make_consecutive_blocks(rownums_sorted, values_by_rownum)

  print('- 🫡Diferencias Encontradas,ℹ️Aplicando Cambios para {} bloques ({} Datos)'.format(len(blocks), len(rownums_sorted)))

  # Aplicar updates (en bloques)
  if blocks:
      _batch_update_rows(ws, start_col_letter, end_col_letter, blocks)

  print('- 🫡Cambios Realizados, ℹ️Añadiendo Ids Nuevos para {} bloques ({} Datos)'.format(np.ceil(len(ids_new)/1000),len(ids_new)))

  # Agregar los nuevos
  if ids_new:
    # Asegurar que todo se convierta a tipos nativos de Python (o string) antes de enviar
    rows_new = []
    for _id in ids_new:
        row_values = py_by_id.loc[_id, cols].tolist()
        # Convertir cada elemento a string si persiste el error, o usar .item()
        sanitized_row = ["" if pd.isna(x) or x in [np.inf, -np.inf] else (x.item() if hasattr(x, 'item') else x) for x in row_values]
        rows_new.append(sanitized_row)

    for i in range(0, len(rows_new), 1000):
        chunk = rows_new[i:i+1000]
        _retry(lambda ch=chunk: ws.append_rows(ch, value_input_option="USER_ENTERED"), label=f"append_rows new {i}-{i+len(chunk)-1}")
        sleep(1)

  return True, (
      f"✅ Completado: "
      f"👍{len(changed_ids):,} filas actualizadas, ℹ️{len(ids_new):,} nuevas {identifierCol}s. "
      f"Total DF={len(dfOut):,} / Total sheet aprox={len(df_sheet):,}"
  )

def deleteRows(worksheet: gspread.Worksheet, rows_to_delete: list[int]):
    """
    Deletes multiple rows from a gspread worksheet efficiently by grouping
    adjacent rows and deleting from bottom to top.
    """
    if not rows_to_delete:
        return

    # 1. Remove duplicates and sort in descending order
    unique_rows = sorted(list(set(rows_to_delete)), reverse=True)

    # 2. Group adjacent numbers.
    # To use groupby on consecutive numbers, we use the trick: row_number - index.
    # Consecutive numbers will yield the same result.
    for _, g in groupby(enumerate(unique_rows), lambda ix: ix[0] + ix[1]):
        group = list(map(itemgetter(1), g))

        # Because unique_rows is descending, the group is also descending.
        # e.g., [10, 9, 8] -> start_row is 8, and we want to delete 3 rows.
        end_row = group[0]
        start_row = group[-1]
        length = end_row - start_row + 1

        # 3. Batch delete this block
        _retry(lambda: worksheet.delete_rows(start_index=start_row, end_index=end_row))

def appendDataFrameToEnd(ws: gspread.Worksheet, df: pd.DataFrame, retry_label="Append Data"):
    """
    Appends a DataFrame to the end of a Google Sheet, ensuring column alignment
    and cleaning up unnecessary columns.
    """
    if df.empty:
      print('⚠️No hay datos para añadir.')
      return False
    # 1. Obtener los Headers
    # We use index 1 to get the first row (headers)
    sheet_headers = _retry(lambda: ws.row_values(1))
    if not sheet_headers:
        raise ValueError("The target sheet is empty. Please add headers first.")

    # 2. Eliminar Columnas Unnamed
    df_clean = df.loc[:, ~df.columns.str.contains('^Unnamed')].copy()

    # 3. Organizar las Columnas
    # - Solo se dejan columnas existentes en el sheets
    # - Si una columna falta, se agrega y se deja vacia
    # - Se ordenan las columnas del mismo modo en el que estan en sheets
    df_final = pd.DataFrame(columns=sheet_headers)
    for col in sheet_headers:
        if col in df_clean.columns:
            df_final[col] = df_clean[col]
        else:
            df_final[col] = "" # Fill missing columns with empty strings

    # 4. Data Preparation: Convert to list of lists and handle NaNs/Dates
    # We reuse the logic from your previous preparation steps
    df_final = df_final.replace([np.inf, -np.inf], None)

    # Format dates to string to avoid JSON serializing errors
    for c in df_final.columns:
        if pd.api.types.is_datetime64_any_dtype(df_final[c]):
            df_final[c] = df_final[c].dt.strftime("%Y-%m-%d %H:%M:%S")
        elif "float" in str(df_final[c].dtype) or "int" in str(df_final[c].dtype):
            df_final[c] = df_final[c].replace([np.inf, -np.inf], '')


    # Fill NaNs with empty string
    values_to_append = df_final.fillna("").values.tolist()

    # 5. Execute Append with Retry Logic
    if values_to_append:
        # We use value_input_option="USER_ENTERED" so dates/numbers are parsed by Sheets
        _retry(
            lambda: ws.append_rows(values_to_append, value_input_option="USER_ENTERED"),
            label=f"{retry_label} ({len(values_to_append)} rows)"
        )
        print(f"✅Se añadieron correctamente {len(values_to_append)} filas a {ws.title}")
    else:
        print("❌No hay datos a añadir.")

    return True

# Funcion Auxiliar para obtener worksheet con objeto sheet
def getWorksheet(sh, wsName, colLen):
  try:
    return _retry(lambda: (True, sh.worksheet(wsName)))
  except:
    return _retry(lambda: (False, sh.add_worksheet(wsName, 1000, colLen)))

def robust_resize(ws, target_rows: int, target_cols: int):
    """
    Safely adjusts a gspread worksheet's dimensions to match target rows and columns
    without crashing or unexpectedly collapsing data.

    :param ws: The gspread Worksheet object.
    :param target_rows: The exact number of rows the sheet should have.
    :param target_cols: The exact number of columns the sheet should have.
    :param _retry: Your custom retry wrapper function, e.g., lambda f: _retry(f)
    """
    # 1. Fetch current dimensions safely
    current_rows = _retry(lambda: ws.row_count)
    current_cols = _retry(lambda: ws.col_count)

    # 2. Adjust Columns first (prevents grid distortion)
    if target_cols > current_cols:
        cols_to_add = target_cols - current_cols
        _retry(lambda: ws.add_cols(cols_to_add))
    elif target_cols < current_cols:
        cols_to_delete = current_cols - target_cols
        # Delete from the end of the sheet moving backwards
        _retry(lambda: ws.delete_columns(start_index=target_cols + 1, end_index=current_cols))

    # 3. Adjust Rows
    if target_rows > current_rows:
        rows_to_add = target_rows - current_rows
        _retry(lambda: ws.add_rows(rows_to_add))
    elif target_rows < current_rows:
        # Delete rows from the end moving backwards
        _retry(lambda: ws.delete_rows(start_index=target_rows + 1, end_index=current_rows))

def batch_upload_dataframe(
    ws: gspread.Worksheet,
    df: pd.DataFrame,
    batch_size: int = 10_000,
    pause_between_batches: float = 1.0
):
    """
    Clears a worksheet and uploads a large DataFrame in batches.

    Parameters:
    - ws: gspread Worksheet object
    - df: pandas DataFrame to upload
    - batch_size: number of rows per update API call (default: 5000)
    - pause_between_batches: pause in seconds to prevent API quota limit issues
    """
    # 1. Clean the contents of the worksheet
    print(f"ℹ️Limpiando Contenidos de Hoja '{ws.title}'...")
    _retry(lambda: ws.clear())

    # Pre-process DataFrame to ensure JSON compliance for Google Sheets API
    # Replace NaN/NaT/Inf with empty strings or clean values
    clean_df = df.replace([np.inf, -np.inf], np.nan).fillna("")

    # Format dates to string to avoid JSON serializing errors
    for c in clean_df.columns:
        if pd.api.types.is_datetime64_any_dtype(clean_df[c]):
            clean_df[c] = clean_df[c].dt.strftime("%Y-%m-%d %H:%M:%S")
        elif "float" in str(clean_df[c].dtype) or "int" in str(clean_df[c].dtype):
            clean_df[c] = clean_df[c].replace([np.inf, -np.inf], '')

    # Aplicamos el Resizing
    robust_resize(ws, len(clean_df) + 1, len(clean_df.columns))

    # Prepare header and data as lists of lists
    headers = [clean_df.columns.tolist()]
    total_rows = len(clean_df)

    # Upload headers first (Row 1)
    _retry(lambda: ws.update(range_name='A1', values=headers))
    print("Headers uploaded successfully.")

    if total_rows == 0:
        print("😁DataFrame Vació, proceso finalizado")
        return

    # 2. Start making the batch updates
    print(f"🗒️ Comenzando Subida Batch de {total_rows:,} filas (📞Tamaño Batch: {batch_size:,})...")

    # Start writing data right below headers (Row 2)
    start_row = 2

    for i in range(0, total_rows, batch_size):
        # Extract the chunk
        chunk = clean_df.iloc[i : i + batch_size]
        chunk_values = chunk.values.tolist()

        # Determine cell range string (e.g., "A2")
        target_cell = f"A{start_row}"

        # Perform batch update
        _retry(lambda: ws.update(range_name=target_cell, values=chunk_values,value_input_option='USER_ENTERED'))

        current_uploaded = min(i + batch_size, total_rows)
        print(f"  -> 🔏Filas Añadidas {i + 1:,} a {current_uploaded:,} , Total: {total_rows:,} ({current_uploaded/total_rows:.1%} Avance)")

        # Advance row pointer
        start_row += len(chunk)

        # Optional brief sleep to adhere to Google Sheets write API limits (60 requests/min)
        if pause_between_batches > 0:
            sleep(pause_between_batches)

    # 3. Show a Message of Success
    print("\n" + "=" * 50)
    print(f" ✅😎: Añadidas Correctamente {total_rows:,} filas a '{ws.title}'!")
    print("=" * 50)

# Funcion Auxiliar para Escribir Datos en una celda específica
def writeToCell(ws, row, col, value):
  _retry(lambda: ws.update_cell(row, col, value), label=f"Update Cell ({row},{col})")

## **Organización de Columnas y Datos**
___
Se aplica la Organización de Columnas dada esta Estructura:
- **Características del Cliente**
- **Características de la Deuda**
- **Proyecciones**
- **Oportunidades por Tradicional**
- **Oportunidades por Crédito**
- **Actualizaciones**
- **Liquidaciones**
- **Funnel de Datos**

In [ ]:
# Definimos las Columnas de
columnaId = ColumnaBucket('Id_Deuda','string', 'Id deuda')
colsCliente = [
    ColumnaBucket('Referencia','semistring'),
    columnaId,
    ColumnaBucket('Negociador_Asignado','string', 'Negociador'),
    ColumnaBucket('Meses_en_Programa','numeric','Meses en el Programa'),
    ColumnaBucket('Batch_Credito','string','Tipo de Liquidacion'),
    ColumnaBucket('Ahorro_Neto','numeric'),
    ColumnaBucket('Meses_Sin_Contacto','numeric','ultimo contacto'),
    ColumnaBucket('Meses_Sin_Deposito','numeric'),
    ColumnaBucket('Pricing','numeric','CE'),
]
colsDeuda = [
    ColumnaBucket('Banco','string','BANCOS_ESTANDAR'),
    ColumnaBucket('Dum_Bravo','numeric','D_BRAVO'),
    ColumnaBucket('Mora_Deuda','numeric','MORA'),
]
colsProyecciones = [
    ColumnaBucket('Ingreso_Esperado','semistring'),
    ColumnaBucket('Ingreso_Posible','semistring'),
]
colsOppTrad = [
    ColumnaBucket('Potencial_Esperado','semistring'),
    ColumnaBucket('Potencial_Tradicional','semistring','Potencial'),
    ColumnaBucket('Motivo_NV','string'),
    ColumnaBucket("Tipo_Sol_Alianzas","string"),
    ColumnaBucket("Estado_Sol_Alianzas","string"),
    ColumnaBucket("Origen_Alianzas","string"),
]
colsOppCredito = [
    ColumnaBucket('PaB_Ideal_Credito','numeric','PB Ideal Credito'),
    ColumnaBucket('Potencial_Credito','semistring','Potencial Credito'),
    ColumnaBucket('Bucket_Credito','string'),
    ColumnaBucket("Status_Credito","string"),
]
colsActs = [
    ColumnaBucket('Fecha_Act','date','inserted_at_ultima'),
    ColumnaBucket('Origen_Act','string','end_ultima'),
    ColumnaBucket('Etiqueta_Act','string','CATEGORIA_PRED_ultima'),
    ColumnaBucket('Comentario_Act','string','observations_ultima'),
    ColumnaBucket('PaB_Act','numeric','payment_to_bank_ultima'),
    ColumnaBucket('Descuento_Act','numeric','Descuento_Actualizacion'),
    ColumnaBucket('Descuento_Mejor_PL',"numeric"),
    ColumnaBucket('Etapa_Act',"numeric"),
    ColumnaBucket('Efectividad','string','Tipo de Actividad'),
    ColumnaBucket("Dias_Anterior_Act","numeric"),
    ColumnaBucket('Fecha_Act_Anterior','date','inserted_at_ultima'),
    ColumnaBucket('Origen_Act_Anterior','string','end_ultima'),
    ColumnaBucket('Etiqueta_Act_Anterior','string','CATEGORIA_PRED_ultima'),
    ColumnaBucket('Descuento_Act_Anterior','numeric','Descuento_Actualizacion'),
    ColumnaBucket('Descuento_Mejor_PL_Anterior',"numeric"),
    ColumnaBucket('Etapa_Act_Anterior',"numeric"),
    ColumnaBucket('Efectividad_Anterior','string','Tipo de Actividad'),
]
colsLiqs = [
    ColumnaBucket('Tipo_Liq','string','Por?'),
    ColumnaBucket('Ingreso_Liq','numeric'),
]
colsFunnel = [
    ColumnaBucket('Bucket_Inicial','numeric'),
    ColumnaBucket('Bucket_MEC','numeric','Bucket'),
    ColumnaBucket('Prosp_Liq','string'),
]

## **Definición de Columnas para Actualización**
___
Se van a crear las listas de las Diferentes Columnas dependiendo de su tipo para que la actualización de datos sea efectiva.

In [ ]:
colsTotal = colsCliente + colsDeuda + colsProyecciones + colsOppTrad + colsOppCredito + colsActs + colsLiqs + colsFunnel

# Aplicamos los Renombramientos dependiendo si se usan los nombres antiguos o
if USAR_NOMBRES_ANTIGUOS:
  for col in colsTotal:
    actualName = col.getActualName()
    oldName = col.getOldName()
    if actualName != oldName:
      newTMDF.rename(columns={actualName: oldName}, inplace=True) # type: ignore
      # Actualizamos las de bmDF si FECHA_MEJOR_MES existe
      if FECHA_MEJOR_MES is not None:
        bmDF.rename(columns={actualName: oldName}, inplace=True) # type: ignore
  print('✅ Se han aplicado los nombres antiguos a las columnas para mantener compatibilidad con versiones anteriores.')
else:
  print('✅ Se mantienen los nombres nuevos de las columnas para mayor claridad y consistencia con el análisis actual.')

# Definimos las Columnas Númericas, de Fecha y Semi-String para el manejo especial de comparación
colsNumericas = [col.getRequiredName(USAR_NOMBRES_ANTIGUOS) for col in colsTotal if col.isNumeric()]
colsFecha = [col.getRequiredName(USAR_NOMBRES_ANTIGUOS) for col in colsTotal if col.isDate()]
colsSemiString = [col.getRequiredName(USAR_NOMBRES_ANTIGUOS) for col in colsTotal if col.isSemiString()]

# Imprimimos para Logs:
print('✅En Total se van a manejar {} columnas.'.format(len(colsTotal)))

# Ahora Calculamos las Columnas por Fuera y Columnas Extra
colsFuera = (newTMDF.columns.difference([c.getRequiredName(USAR_NOMBRES_ANTIGUOS) for c in colsTotal])).to_list()
colsExtra = list(set([c.getRequiredName(USAR_NOMBRES_ANTIGUOS) for c in colsTotal]) - set(newTMDF.columns))

print('🫡Columnas Dejadas por Fuera: {}'.format(len(colsFuera)))
print_grid(colsFuera, n=3)
print('🫡Columnas Extra a Calcular: {}'.format(len(colsExtra)))
print_grid(colsExtra, n=3)

# Dejamos solo las Columnas Requeridas
newTMDF = newTMDF[[c.getRequiredName(USAR_NOMBRES_ANTIGUOS) for c in colsTotal]]
if FECHA_MEJOR_MES is not None:
  bmDF = bmDF[[c.getRequiredName(USAR_NOMBRES_ANTIGUOS) for c in colsTotal]]

# Ordenamos los Datos por Fecha_Act
newTMDF.sort_values(by='Fecha_Act', inplace=True)
if FECHA_MEJOR_MES is not None:
  bmDF.sort_values(by='Fecha_Act', inplace=True)

print('✅😁ℹ️Preparación de Datos Completada.')

✅ Se mantienen los nombres nuevos de las columnas para mayor claridad y consistencia con el análisis actual.
✅En Total se van a manejar 46 columnas.
🫡Columnas Dejadas por Fuera: 63
Ahorro_Auxiliar / Ahorro_Estructurado / Ahorro_Medio
Ahorro_Necesario_Est / Ahorro_Necesario_Liq / Ahorro_Total
Apartado_Neto / Bucket_Banco / Bucket_Base
Bucket_Potencial / Bucket_Ratio / Cedula
Comentario_Act_Anterior / Comision_Exito / Comision_Mensual
Descuento_Auxiliar / Descuento_Esperado / Descuento_PL
Descuento_Real / Deuda_Representante / Deudas_Contar
Eficiencia_CE / Es_Baja / Es_Portafolio
Es_Viable / Estado / Estado_Cliente
Estructurable / Fase / Fecha_Baja
Fecha_Liq / Ingreso_Original / Lider_Origen
Lider_Origen_Anterior / Lider_Portafolio / Mayor_PL
Mora_Credito / Mora_Estructurado / Negociador_Liq
Negociador_Principal / Nombre_Cliente / PaB_Act_Anterior
PaB_Auxiliar / PaB_Esperado / PaB_Liq
PaB_Or_Aux / PaB_PL / PaB_Real
Plazos_PaB / Pond_Descuento_Act / Pond_Descuento_PL
Pond_Eficiencia_CE / 

## **Actualización de Mes_Actual**
___
Se añaden los nuevos Datos al final del Timeline

In [ ]:
# Abrimos el Sheets llamado "Bucket Cartera"
actualSHId = PIPACT_SHEET_ID
actualSH = _retry(lambda: client.open_by_key(actualSHId))
# Abrimos la Hoja de 'Mes_Actual'
existed, actualWS = getWorksheet(actualSH, 'Mes_Actual', len(colsTotal))
print('😁Hoja Obtenida')

# Lógica de Escritura
if (tmWasEmpty) or REESCRITURA_TOTAL or (not existed) or tmWasEmpty:
  print('🆔Reescritura Total por Fecha de Corte o Hoja Vacia')

  # Reemplazamos Infinitos con NaNs
  newTMDF = newTMDF.replace([np.inf, -np.inf, np.nan], "")

  # Volvemos inserted_at_ultima a formato %Y-%m-%d %X
  for col in colsFecha:
    newTMDF[col] = newTMDF[col].apply(lambda x: x.strftime("%Y-%m-%d %X") if pd.notna(x) else "")

  # Ahora Subimos los Datos
  batch_upload_dataframe(actualWS, newTMDF)
  print('✅Información de Mes Actual Reescrita con Éxito')
  print('ℹ️Filas Subidas: {}'.format(
      len(newTMDF)
  ))
else:
  print('🆔Adición de Datos al Final de la Hoja')
  appendDataFrameToEnd(actualWS, newTMDF)
  print('✅Información de Mes Actual Actualizada con Éxito')
  print('ℹ️Filas Añadidas: {}'.format(
      len(newTMDF)
  ))
#

😁Hoja Obtenida
🆔Adición de Datos al Final de la Hoja
✅Se añadieron correctamente 1070 filas a Mes_Actual
✅Información de Mes Actual Actualizada con Éxito
ℹ️Filas Añadidas: 1070


# **Escritura de Mejor Mes**
___
En caso de que haya cambiado el mejor mes, se procede a realizar la escritura de este cambio.

In [ ]:
if not(FECHA_MEJOR_MES is None):
  print('🆔Reescritura de Mejor Mes')
  # Abrimos la Hoja llamada 'Mejor_Mes'
  bmWS = _retry(lambda: actualSH.worksheet("Mejor_Mes"))
  # Debe de existir el df llamado bmDF, por lo que aplicamos la misma limpieza de subida que a bmDF
  # 1: Reemplazar Nans e infinitos por ""
  bmDF = bmDF.replace([np.inf, -np.inf, np.nan], "")

  # Volvemos inserted_at_ultima a formato %Y-%m-%d %X
  for col in colsFecha:
    bmDF[col] = bmDF[col].apply(lambda x: x.strftime("%Y-%m-%d %X") if pd.notna(x) else "")

  # Ahora Subimos los Datos
  batch_upload_dataframe(bmWS, bmDF)
  print('✅Información de Timeline Reescrita con Éxito')
  print('ℹ️Filas Subidas: {}'.format(
      len(bmDF)
  ))
else:
  print('👌No se necesita cambiar el Mejor Mes')

🆔Reescritura de Mejor Mes
ℹ️Limpiando Contenidos de Hoja 'Mejor_Mes'...
Headers uploaded successfully.
🗒️ Comenzando Subida Batch de 27,747 filas (📞Tamaño Batch: 10,000)...
  -> 🔏Filas Añadidas 1 a 10,000 , Total: 27,747 (36.0% Avance)
  -> 🔏Filas Añadidas 10,001 a 20,000 , Total: 27,747 (72.1% Avance)
  -> 🔏Filas Añadidas 20,001 a 27,747 , Total: 27,747 (100.0% Avance)

 ✅😎: Añadidas Correctamente 27,747 filas a 'Mejor_Mes'!
✅Información de Timeline Reescrita con Éxito
ℹ️Filas Subidas: 27747


# **Actualización de Referencias en Mes Actual**
___
Se realiza un ajuste de posibles cambios de referencia que hayan ocurrido

## **Obtención de Referencias en Mes Actual**
___
Se realizan estos 3 pasos:
1. **Obtener Referencias en Mes Actual**
2. **Definir Referencias Cambiadas**
3. **Crear Diccionario de Actualización de Referencias**

In [ ]:
# Definimos los Valores de Referencias
refValues = _retry(lambda: actualWS.col_values(1)[1:])

# Creamos el Diccionario
refsIndexes = [(ref,i) for i,ref in enumerate(refValues,start=2)]

# Filtramos los que tengan cambio de Referencia
refsIndexes = [(ref,idx) for (ref,idx) in refsIndexes if refChangesDict.get(ref,ref) != ref]

# Creamos el Diciconario myRefChange con el nuevo cambio de Referencias
myRefChange = {refChangesDict.get(ref,ref):idx for (ref,idx) in refsIndexes}

print('✅Diccionario de Cambios de Referencia Creado con Éxito')

✅Diccionario de Cambios de Referencia Creado con Éxito


## **Ejecución de Actualización de Referencias**
___

Se procede a ejecutar la Actualización de Referencias en el Mes_Actual usando **myRefChange**

In [ ]:
print('😁Actualizando las Referencias del Funnel')
batch_update_column_a_from_dict(actualWS, myRefChange)
print('✅Referencias Actualizadas con Éxito')
print('ℹ️Referencias Totales Actualizadas: {}'.format(
    len(myRefChange)
))

😁Actualizando las Referencias del Funnel
✅Referencias Actualizadas con Éxito
ℹ️Referencias Totales Actualizadas: 0


# **Ajuste de Liquidaciones**
___
Dado que las Liquidaciones pueden tener Ajustes antes o después, es necesario realizar los siguientes cambios:
- **Eliminar Liquidaciones Inexistentes**
- **Eliminar Duplicados de Liquidaciones**

## **Obtención de Datos**
___
Se obtienen los Datos usando Gspread para minimizar la Carga del API dada la cantidad de Datos (**localTMDF**)
- Id_Deuda
- Fase
- Sheets_Row

In [ ]:
# Primero vamos a Obtener 2 Rangos:
# Rango de Id_Deuda
# Rango de Fase

# 1. Obtener los Headers del Timeline
tmHeaders = _retry(lambda: actualWS.row_values(1))
# 2. Buscar los Indices de las Columnas
idIdx = None
etiquetaIdx = None
for i,h in enumerate(tmHeaders,start=1):
  if h in ['Id_Deuda','Id deuda','Id Deuda','Id_deuda']:
    idIdx = i
  elif h in ['Etiqueta_Act','Status_Act']:
    etiquetaIdx = i

# Verificamos que hayan sido encontrados
if (idIdx is None) or (etiquetaIdx is None):
  raise ValueError('🚯Error: no se encontraron los Headers de Id_Deuda y Etiqueta en el Timeline')

# Obtenemos los Datos usando .col_values
idValues = _retry(lambda: actualWS.col_values(idIdx)[1:])
etiquetaValues = _retry(lambda: actualWS.col_values(etiquetaIdx)[1:])

# Volvemos los Datos a un DF
localTMDF = pd.DataFrame({'Id_Deuda':idValues,'Etiqueta':etiquetaValues})

# Creamos la Columna Sheets_Row
localTMDF['Sheets_Row'] = localTMDF.index + 2

print('✅Datos del Timeline Obtenidos con Éxito')
print('ℹ️Datos Totales: {}'.format(
    len(localTMDF)
))

✅Datos del Timeline Obtenidos con Éxito
ℹ️Datos Totales: 43962


## **Eliminar Liquidaciones Inexistentes**
___
Dados los Ajustes del Pipeline, es Necesario quitar las Liquidaciones que fueron ajustadas y no acabaron siendo realizadas

In [ ]:
# La Detección es: Tiene Fase 4 pero no esta en liqsDF el Id_Deuda
maskQuit = localTMDF['Etiqueta'].str.contains('Liquidado',case=False,na=False) & ~(localTMDF['Id_Deuda'].isin(liqsDF['Id_Deuda']))

# Obtenemos las Filas a Eliminar
rowsToDelete = localTMDF.loc[maskQuit,'Sheets_Row'].tolist()

print('🫡Eliminando {} Falsas Liquidaciones'.format(
    len(rowsToDelete)
))
deleteRows(actualWS, rowsToDelete)
print('✅Eliminaciones Realizadas con Éxito')

🫡Eliminando 0 Falsas Liquidaciones
✅Eliminaciones Realizadas con Éxito


## **Eliminar Duplicados de Liquidaciones**
___
Se verifica que por Id_Deuda exista tan solo un registro por liquidación, de lo contrario se eliminan dejando el último registro

In [ ]:
# Paso 1: Agrupar por Id_Deuda y dejar:
# Cantidad de Registros
# Lista de Sheets_Row

# Primero vamos a Obtener 2 Rangos:
# Rango de Id_Deuda
# Rango de Fase

# 1. Obtener los Headers del Timeline
tmHeaders = _retry(lambda: actualWS.row_values(1))
# 2. Buscar los Indices de las Columnas
idIdx = None
etiquetaIdx = None
for i,h in enumerate(tmHeaders,start=1):
  if h in ['Id_Deuda','Id deuda','Id Deuda','Id_deuda']:
    idIdx = i
  elif h in ['Etiqueta_Act','Status_Act']:
    etiquetaIdx = i

# Verificamos que hayan sido encontrados
if (idIdx is None) or (etiquetaIdx is None):
  raise ValueError('🚯Error: no se encontraron los Headers de Id_Deuda y Fase en el Timeline')

# Obtenemos los Datos usando .col_values
idValues = _retry(lambda: actualWS.col_values(idIdx)[1:])
etiquetaValues = _retry(lambda: actualWS.col_values(etiquetaIdx)[1:])

# Volvemos los Datos a un DF
localTMDF = pd.DataFrame({'Id_Deuda':idValues,'Etiqueta':etiquetaValues})

# Creamos la Columna Sheets_Row
localTMDF['Sheets_Row'] = localTMDF.index + 2

# Primero Dejamos solo las Liquidaciones
localLiqs = localTMDF[localTMDF['Etiqueta'].str.contains('liquidado',case=False,na=False)].copy()

# Ahora Aplicamos la Agrupación
grpDF = localLiqs.groupby('Id_Deuda').agg(
    Registros=('Sheets_Row','count'),
    Sheets_Row=('Sheets_Row',lambda x: x.tolist())
).reset_index()

# Ahora Vamos a Filtrar los que tengan más de 1 Registro
grpDelete = grpDF[grpDF['Registros'] > 1].copy()

# Unimos todas las Listas quitando el último Registro para eliminar Dichas Filas
rowsToDelete = [
    row for rows in grpDelete['Sheets_Row'].tolist() for row in rows[:-1]
]

# Eliminamos Dichas Filas
print('🚯Eliminando {} filas por Liquidaciones Duplicadas'.format(
    len(rowsToDelete)
))
deleteRows(actualWS, rowsToDelete)
print('✅Filas Eliminadas con Éxito')
print('ℹ️Filas Totales Eliminadas: {}'.format(
    len(rowsToDelete)
))

🚯Eliminando 0 filas por Liquidaciones Duplicadas
✅Filas Eliminadas con Éxito
ℹ️Filas Totales Eliminadas: 0


# **Escritura de Inputs**
___
Una vez queda completada toda la subida se procede a escribir los inputs de:
- **Mejor-Mes** si aplica
- **Mes-Mec**
- **Ultima-Ejecución**

In [ ]:
# Abrimos la Hoja de los Controles llamada 'Control'
controlSHId = PIPACT_SHEET_ID
controlSH = _retry(lambda: client.open_by_key(controlSHId))
_,controlWS = getWorksheet(controlSH, 'Control', len(colsTotal))

# Escritura de Mejor Mes
if (FECHA_MEJOR_MES is not None):
  writeToCell(controlWS, 1,2, FECHA_MEJOR_MES.strftime("%Y-%m"))

# Escritura de Mes-Mec
writeToCell(controlWS,2,2, fecha_cartera_ultima.strftime("%Y-%m"))

# Escritura de Última-Ejecución
writeToCell(controlWS,3,2, today.strftime("%Y-%m-%d %X"))

print('✅ℹ️Los Inputs fueron actualizados con Éxito')

✅ℹ️Los Inputs fueron actualizados con Éxito
